# MolGenerate — воспроизводимый генератор UV-поглощающих MOST-молекул

Этот notebook — единая линейная реализация и исполнимый аудит всего контура:
реальные открытые endpoint-данные → независимые reviewer ensembles →
family-specific transfer learning → настоящий REINVENT4 LibInvent curriculum →
нейросетевой sampling → matched evaluation → safety veto → независимая
переоценка → GFN2-xTB/sTDA-xTB unit-oracle → отчёт и машинная проверка.

Главная граница утверждений: это **исследовательский screening proxy**. Молекулы
не сертифицированы по ISO 24444/24443 и не признаны безопасными косметическими
ингредиентами. Нулевая итоговая выборка допустима, если данные/AD/неопределённость
не позволяют пройти все жёсткие ворота.


## 1. Среда, версии и воспроизводимость

Основной анализ работает в Python 3.10. REINVENT4 v4.8.24 закреплён commit
`80a8d21aefd9c0d3ec806377522effb30cfca12a` и запускается в отдельном Python
3.12 окружении. LibInvent prior проверяется по SHA-256. Физический oracle
использует libxTB через Python/ASE и официальные `xtb4stda`/`stda` binaries.


In [1]:
from collections import Counter
from pathlib import Path
import importlib.metadata
import json
import platform
import sys

from IPython.display import Markdown, display
from rdkit import RDLogger

for channel in ("rdApp.debug", "rdApp.info", "rdApp.warning", "rdApp.error"):
    RDLogger.DisableLog(channel)

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config/default.json").is_file():
    raise RuntimeError("Запускайте MolGenerate.ipynb из директории TEST")

packages = {}
for package in ("rdkit", "numpy", "pandas", "scikit-learn", "openpyxl", "ase", "xtb"):
    packages[package] = importlib.metadata.version(package)
display({"project": str(PROJECT_ROOT), "python": sys.version.split()[0],
         "platform": platform.platform(), "packages": packages})


{'project': '/home/sigmatau17/TEST',
 'python': '3.10.12',
 'platform': 'Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.35',
 'packages': {'rdkit': '2026.3.6',
  'numpy': '2.2.6',
  'pandas': '2.3.3',
  'scikit-learn': '1.7.2',
  'openpyxl': '3.1.5',
  'ase': '3.29.0',
  'xtb': '22.1'}}

## 2. Что используется и как устроен pipeline

1. `prepare-data`: неизменяемые raw-файлы нормализуются в длинную endpoint-
   таблицу. λmax, кинетика, Kp, фототоксичность и сенсибилизация не смешиваются.
2. `train-reviewers`: Random Forest reward ensembles и отдельные Extra Trees
   evaluators обучаются по exact-Murcko scaffold split; uncertainty — 90%
   split-conformal radius, не только разброс деревьев.
3. `train-generator`: для NBD/QC, Dewar-pyrimidinone и spiropyran создаются
   family-specific LibInvent TL agents из закреплённого reaction prior.
4. `generate`: REINVENT4 проходит curriculum chemistry → spectrum → MOST →
   safety. ExternalProcess возвращает плотный stage score. Затем checkpoint
   сэмплируется; разрешены только точные продукты фиксированных синтонов.
5. Два baseline: random reaction-library sampling и weighted retraining.
6. `metrics` и `review`: равные полные reviewer-бюджеты и итоговые evaluation
   sets, diversity/AD/gates,
   независимые evaluator models и карточки причин pass/fail.
7. Physical oracle: conformer search → GFN2-xTB оптимизация пары → ΔE/Wh·kg⁻¹;
   xtb4stda/sTDA transitions → broadened 290–400 nm proxy. Он не входит в RL.

ITI оставлен только для class-shift validation и не генерируется.


## 3. Полный исходный код внутри notebook

Каждый модуль ниже встроен отдельной последовательной ячейкой. Они загружаются
в изолированный namespace текущего kernel. Внешний REINVENT-процесс по своей
природе не видит память notebook, поэтому вызывает идентичный дисковый bridge;
его исходник и checksum также показаны здесь.


In [2]:
import types

for loaded_name in list(sys.modules):
    if loaded_name == "mostgen" or loaded_name.startswith("mostgen."):
        del sys.modules[loaded_name]

embedded_package = types.ModuleType("mostgen")
embedded_package.__package__ = "mostgen"
embedded_package.__path__ = [str(PROJECT_ROOT / "mostgen")]
embedded_package.__file__ = str(PROJECT_ROOT / "mostgen/__init__.py")
sys.modules["mostgen"] = embedded_package

def load_embedded(name, source, filename):
    module = types.ModuleType(name)
    module.__file__ = str(filename)
    module.__package__ = name.rpartition(".")[0]
    sys.modules[name] = module
    exec(compile(source, str(filename), "exec"), module.__dict__)
    return module


### 3.6 Версия пакета — `__init__.py`


In [3]:
_source = '"""MOSTGen: reproducible UV/MOST computational screening prototype."""\n\n__version__ = "0.2.0"\n'
exec(compile(_source, embedded_package.__file__, 'exec'), embedded_package.__dict__)
print('loaded mostgen', embedded_package.__version__)


loaded mostgen 0.2.0


### 3.8 Конфигурация и бюджеты — `config.py`


In [4]:
_source = 'from __future__ import annotations\n\nimport copy\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\n\nROOT = Path(__file__).resolve().parents[1]\nDEFAULT_CONFIG = ROOT / "config" / "default.json"\n\n\nclass ConfigError(ValueError):\n    """Raised when an experiment configuration violates the contract."""\n\n\ndef _deep_update(base: dict[str, Any], patch: dict[str, Any]) -> dict[str, Any]:\n    for key, value in patch.items():\n        if isinstance(value, dict) and isinstance(base.get(key), dict):\n            _deep_update(base[key], value)\n        else:\n            base[key] = value\n    return base\n\n\ndef load_config(path: str | Path | None = None, mode: str = "full") -> dict[str, Any]:\n    config_path = Path(path) if path else DEFAULT_CONFIG\n    with config_path.open(encoding="utf-8") as handle:\n        config = json.load(handle)\n    config["_config_path"] = str(config_path.resolve())\n    if mode == "smoke":\n        smoke = copy.deepcopy(config["smoke"])\n        config["execution"].update({k: v for k, v in smoke.items() if k != "training_rows_per_family"})\n        config["execution"]["mode"] = "smoke"\n        config["training_rows_per_family"] = smoke["training_rows_per_family"]\n    elif mode == "full":\n        config["execution"]["mode"] = "full"\n        config["training_rows_per_family"] = 180\n    elif mode == "production":\n        config["execution"]["mode"] = "production"\n        config["execution"]["backend"] = "reinvent4"\n        config["oracle"]["run_automatically"] = True\n        config["training_rows_per_family"] = 180\n    else:\n        raise ConfigError(f"Unknown mode: {mode}")\n    validate_config(config)\n    return config\n\n\ndef apply_override(config: dict[str, Any], override_path: str | Path | None) -> dict[str, Any]:\n    if not override_path:\n        return config\n    with Path(override_path).open(encoding="utf-8") as handle:\n        patch = json.load(handle)\n    merged = _deep_update(copy.deepcopy(config), patch)\n    validate_config(merged)\n    return merged\n\n\ndef validate_config(config: dict[str, Any]) -> None:\n    required_families = {"nbd_qc", "dewar_pyrimidinone", "spiropyran"}\n    missing = required_families - set(config.get("families", {}))\n    if missing:\n        raise ConfigError(f"Missing family configurations: {sorted(missing)}")\n    project = config.get("project", {})\n    if int(project.get("max_training_structures", 0)) > 30_000:\n        raise ConfigError("max_training_structures exceeds the 30,000 structure contract")\n    execution = config.get("execution", {})\n    if float(execution.get("gpu_hours", 0.0)) > float(project.get("max_gpu_hours", 8.0)):\n        raise ConfigError("Configured GPU budget exceeds project maximum")\n    if int(execution.get("reviewer_budget_per_run", 0)) < int(execution.get("n_per_run", 0)):\n        raise ConfigError("reviewer_budget_per_run must be at least n_per_run")\n    if len(config.get("synthons", [])) < 2:\n        raise ConfigError("At least two synthons are required")\n    ids = [item["id"] for item in config["synthons"]]\n    if len(ids) != len(set(ids)):\n        raise ConfigError("Synthon identifiers must be unique")\n    if not 0.0 < float(config["reward"]["floor"]) < 1.0:\n        raise ConfigError("Reward floor must be between zero and one")\n    reviewers = config.get("reviewers", {})\n    if float(reviewers.get("specific_energy_min_wh_kg", 0.0)) < 0.0:\n        raise ConfigError("specific_energy_min_wh_kg must be non-negative")\n    for key in ("uvb_transmittance_max", "uva_transmittance_max"):\n        if not 0.0 < float(reviewers.get(key, 0.0)) <= 1.0:\n            raise ConfigError(f"{key} must be in (0, 1]")\n    if float(reviewers.get("beer_lambert_loading_scale", 0.0)) <= 0.0:\n        raise ConfigError("beer_lambert_loading_scale must be positive")\n\n\ndef dump_resolved_config(config: dict[str, Any], path: str | Path) -> None:\n    clean = {k: v for k, v in config.items() if not k.startswith("_")}\n    Path(path).write_text(json.dumps(clean, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n'
load_embedded('mostgen.config', _source, PROJECT_ROOT / 'mostgen' / 'config.py')
print('loaded mostgen.config')


loaded mostgen.config


### 3.10 SHA-256 и manifests — `provenance.py`


In [5]:
_source = 'from __future__ import annotations\n\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport platform\nimport subprocess\nimport sys\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nfrom . import __version__\n\n\ndef sha256_file(path: str | Path) -> str:\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef stable_hash(text: str, length: int = 16) -> str:\n    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:length]\n\n\ndef package_versions(names: Iterable[str]) -> dict[str, str]:\n    result: dict[str, str] = {}\n    for name in names:\n        try:\n            result[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            result[name] = "not-installed"\n    return result\n\n\ndef git_state(root: Path) -> dict[str, Any]:\n    try:\n        commit = subprocess.run(\n            ["git", "rev-parse", "HEAD"], cwd=root, check=True,\n            text=True, capture_output=True, timeout=5,\n        ).stdout.strip()\n        dirty = bool(subprocess.run(\n            ["git", "status", "--porcelain"], cwd=root, check=True,\n            text=True, capture_output=True, timeout=5,\n        ).stdout.strip())\n        return {"commit": commit, "dirty": dirty}\n    except (OSError, subprocess.SubprocessError):\n        return {"commit": "not-a-git-worktree", "dirty": None}\n\n\ndef write_manifest(\n    path: str | Path,\n    config: dict[str, Any],\n    inputs: Iterable[str | Path] = (),\n    command: list[str] | None = None,\n) -> dict[str, Any]:\n    root = Path(__file__).resolve().parents[1]\n    input_records = []\n    for item in inputs:\n        source = Path(item)\n        if source.exists() and source.is_file():\n            input_records.append({\n                "path": str(source.resolve()),\n                "bytes": source.stat().st_size,\n                "sha256": sha256_file(source),\n            })\n    manifest = {\n        "schema_version": "1.0",\n        "created_at_utc": datetime.now(timezone.utc).isoformat(),\n        "mostgen_version": __version__,\n        "python": sys.version,\n        "platform": platform.platform(),\n        "executable": sys.executable,\n        "command": command or sys.argv,\n        "cwd": str(Path.cwd()),\n        "timezone": os.environ.get("TZ", "system-default"),\n        "random_seeds": list(config["execution"]["seeds"]),\n        "budget": {\n            "max_training_structures": config["project"]["max_training_structures"],\n            "max_gpu_hours": config["project"]["max_gpu_hours"],\n            "configured_gpu_hours": config["execution"]["gpu_hours"],\n            "reviewer_calls_per_run": config["execution"]["reviewer_budget_per_run"],\n        },\n        "backend": config["execution"]["backend"],\n        "packages": package_versions(["rdkit", "numpy", "pandas", "scikit-learn", "PyYAML"]),\n        "git": git_state(root),\n        "inputs": input_records,\n        "sources": [\n            {\n                **source,\n                "version": source.get("version", "REFERENCE_UNPINNED"),\n                "checksum": source.get("checksum", "NOT_FETCHED_NO_LOCAL_ARTIFACT"),\n            }\n            for source in config["sources"]\n        ],\n    }\n    destination = Path(path)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    destination.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    return manifest\n\n\ndef write_artifact_manifest(root: str | Path) -> dict[str, Any]:\n    experiment = Path(root).resolve()\n    destination = experiment / "artifact_manifest.json"\n    artifacts = []\n    for path in sorted(experiment.rglob("*")):\n        if not path.is_file() or path == destination:\n            continue\n        artifacts.append({\n            "path": str(path.relative_to(experiment)),\n            "bytes": path.stat().st_size,\n            "sha256": sha256_file(path),\n        })\n    manifest = {"schema_version": "1.0", "root": str(experiment), "artifacts": artifacts}\n    destination.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    return manifest\n\n\ndef append_event(path: str | Path, event: str, payload: dict[str, Any]) -> None:\n    record = {\n        "time_utc": datetime.now(timezone.utc).isoformat(),\n        "event": event,\n        **payload,\n    }\n    with Path(path).open("a", encoding="utf-8") as handle:\n        handle.write(json.dumps(record, sort_keys=True) + "\\n")\n'
load_embedded('mostgen.provenance', _source, PROJECT_ROOT / 'mostgen' / 'provenance.py')
print('loaded mostgen.provenance')


loaded mostgen.provenance


### 3.12 Пары изомеров и safety veto — `chemistry.py`


In [6]:
_source = 'from __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom functools import lru_cache\nfrom pathlib import Path\nfrom typing import Any, Iterable\nimport xml.etree.ElementTree as ET\n\nfrom rdkit import Chem, DataStructs\nfrom rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors\nfrom rdkit.Chem.MolStandardize import rdMolStandardize\nfrom rdkit.Chem.Scaffolds import MurckoScaffold\nfrom rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator\n\n\nclass ChemistryError(ValueError):\n    """A molecule cannot be standardized or does not satisfy the chemistry contract."""\n\n\n@dataclass(frozen=True)\nclass MoleculePair:\n    smiles: str\n    charged_smiles: str\n    family: str\n    synthon_a: str\n    synthon_b: str\n    formula: str\n\n\n@dataclass(frozen=True)\nclass SafetyVeto:\n    veto: bool\n    psoralen_alert: bool\n    known_phototoxic_match: bool\n    psoralen_similarity: float\n    psoralen_similarity_warning: bool\n    reactive_alerts: tuple[str, ...]\n    unsupported_elements: tuple[str, ...]\n    reasons: tuple[str, ...]\n\n    def to_dict(self) -> dict[str, Any]:\n        result = asdict(self)\n        result["reactive_alerts"] = ";".join(self.reactive_alerts)\n        result["unsupported_elements"] = ";".join(self.unsupported_elements)\n        result["reasons"] = ";".join(self.reasons)\n        return result\n\n\ndef parse_smiles(smiles: str) -> Chem.Mol:\n    if not isinstance(smiles, str) or not smiles.strip():\n        raise ChemistryError("empty_smiles")\n    mol = Chem.MolFromSmiles(smiles.strip(), sanitize=True)\n    if mol is None:\n        raise ChemistryError("invalid_smiles")\n    return mol\n\n\n@lru_cache(maxsize=100_000)\ndef standardize_smiles(smiles: str) -> str:\n    mol = parse_smiles(smiles)\n    try:\n        mol = rdMolStandardize.Cleanup(mol)\n        mol = rdMolStandardize.FragmentParent(mol)\n        uncharger = rdMolStandardize.Uncharger(canonicalOrder=True)\n        # Preserve genuine zwitterions: only accept uncharging when net formal\n        # charge is non-zero.  Merocyanines contain balanced explicit charges.\n        if Chem.GetFormalCharge(mol) != 0:\n            mol = uncharger.uncharge(mol)\n        Chem.SanitizeMol(mol)\n    except Exception as exc:  # RDKit exposes several sanitizer exception types.\n        raise ChemistryError(f"standardization_failed:{exc}") from exc\n    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)\n\n\ndef molecular_formula(smiles: str) -> str:\n    return rdMolDescriptors.CalcMolFormula(parse_smiles(smiles))\n\n\ndef build_pair(config: dict[str, Any], family: str, synthon_a: str, synthon_b: str) -> MoleculePair:\n    if family not in config["families"]:\n        raise ChemistryError(f"unsupported_family:{family}")\n    library = {entry["id"]: entry for entry in config["synthons"]}\n    try:\n        first, second = library[synthon_a], library[synthon_b]\n    except KeyError as exc:\n        raise ChemistryError(f"unknown_synthon:{exc.args[0]}") from exc\n    if not first.get("commercial") or not second.get("commercial"):\n        raise ChemistryError("noncommercial_synthon")\n    family_config = config["families"][family]\n    raw_ground = family_config["ground_template"].format(r1=first["smiles"], r2=second["smiles"])\n    raw_charged = family_config["charged_template"].format(r1=first["smiles"], r2=second["smiles"])\n    ground = standardize_smiles(raw_ground)\n    charged = standardize_smiles(raw_charged)\n    ground_formula = molecular_formula(ground)\n    charged_formula = molecular_formula(charged)\n    if ground_formula != charged_formula:\n        raise ChemistryError(f"isomer_formula_mismatch:{ground_formula}!={charged_formula}")\n    if ground == charged:\n        raise ChemistryError("isomer_pair_identical")\n    return MoleculePair(ground, charged, family, synthon_a, synthon_b, ground_formula)\n\n\ndef murcko_scaffold(smiles: str) -> str:\n    mol = parse_smiles(smiles)\n    scaffold = MurckoScaffold.GetScaffoldForMol(mol)\n    if scaffold.GetNumAtoms() == 0:\n        return Chem.MolToSmiles(mol, canonical=True)\n    generic = MurckoScaffold.MakeScaffoldGeneric(scaffold)\n    return Chem.MolToSmiles(generic, canonical=True)\n\n\n@lru_cache(maxsize=8)\ndef _morgan_generator(bits: int):\n    return GetMorganGenerator(radius=2, fpSize=bits, includeChirality=True)\n\n\n@lru_cache(maxsize=100_000)\ndef fingerprint(smiles: str, bits: int = 256):\n    return _morgan_generator(bits).GetFingerprint(parse_smiles(smiles))\n\n\ndef fingerprint_indices(smiles: str, bits: int = 256) -> list[int]:\n    return list(fingerprint(smiles, bits).GetOnBits())\n\n\ndef tanimoto(smiles_a: str, smiles_b: str, bits: int = 256) -> float:\n    return float(DataStructs.TanimotoSimilarity(fingerprint(smiles_a, bits), fingerprint(smiles_b, bits)))\n\n\ndef max_similarity(smiles: str, references: Iterable[str], bits: int = 256) -> float:\n    query = fingerprint(smiles, bits)\n    ref_fps = [fingerprint(reference, bits) for reference in references]\n    if not ref_fps:\n        return 0.0\n    return float(max(DataStructs.BulkTanimotoSimilarity(query, ref_fps)))\n\n\ndef descriptors(smiles: str) -> dict[str, float]:\n    mol = parse_smiles(smiles)\n    atoms = max(1, mol.GetNumHeavyAtoms())\n    rings = float(rdMolDescriptors.CalcNumRings(mol))\n    aromatic = float(rdMolDescriptors.CalcNumAromaticRings(mol))\n    rotors = float(Lipinski.NumRotatableBonds(mol))\n    hetero = float(rdMolDescriptors.CalcNumHeteroatoms(mol))\n    charge_separation = float(sum(abs(atom.GetFormalCharge()) for atom in mol.GetAtoms()))\n    return {\n        "mol_wt": float(Descriptors.MolWt(mol)),\n        "logp": float(Descriptors.MolLogP(mol)),\n        "tpsa": float(rdMolDescriptors.CalcTPSA(mol)),\n        "hbd": float(Lipinski.NumHDonors(mol)),\n        "hba": float(Lipinski.NumHAcceptors(mol)),\n        "rings": rings,\n        "aromatic_rings": aromatic,\n        "rotatable_bonds": rotors,\n        "hetero_fraction": hetero / atoms,\n        "fraction_csp3": float(rdMolDescriptors.CalcFractionCSP3(mol)),\n        "formal_charge": float(Chem.GetFormalCharge(mol)),\n        "charge_separation": charge_separation,\n        "heavy_atoms": float(atoms),\n    }\n\n\ndef synthetic_accessibility(smiles: str) -> float:\n    """Transparent Ertl-inspired proxy on a 1 (easy) to 10 (hard) scale.\n\n    This is deliberately named and reported as a proxy; production review may\n    replace it with the RDKit contrib SA implementation without changing the\n    CSV contract.\n    """\n    d = descriptors(smiles)\n    raw = (\n        1.0\n        + 0.20 * d["rings"]\n        + 0.13 * d["rotatable_bonds"]\n        + 0.035 * max(0.0, d["heavy_atoms"] - 12.0)\n        + 0.18 * abs(d["logp"] - 2.0)\n        + 0.12 * d["charge_separation"]\n    )\n    return min(10.0, max(1.0, raw))\n\n\n@lru_cache(maxsize=256)\ndef _query(pattern: str) -> Chem.Mol | None:\n    query = Chem.MolFromSmarts(pattern)\n    return query if query is not None else Chem.MolFromSmiles(pattern)\n\n\n@lru_cache(maxsize=64)\ndef _core_query(pattern: str) -> Chem.Mol | None:\n    """A sanitized aromatic topology query for fused-core matching."""\n    return Chem.MolFromSmiles(pattern)\n\n\n@lru_cache(maxsize=4)\ndef _u16_positive_registry(directory_name: str) -> tuple[str, ...]:\n    """Read exact experimental positives from the pinned U16 QsarDB archive."""\n    directory = Path(directory_name)\n    if not directory.is_absolute():\n        directory = Path(__file__).resolve().parents[1] / directory\n    compounds_xml = directory / "compounds" / "compounds.xml"\n    values_path = directory / "properties" / "3T3_NRU_Phototoxicity" / "values"\n    if not compounds_xml.exists() or not values_path.exists():\n        return ()\n    namespace = {"q": "http://www.qsardb.org/QDB"}\n    compounds: dict[int, str] = {}\n    for node in ET.parse(compounds_xml).getroot().findall("q:Compound", namespace):\n        cid = int(node.findtext("q:Id", namespaces=namespace))\n        inchi = node.findtext("q:InChI", default="", namespaces=namespace)\n        mol = Chem.MolFromInchi(inchi) if inchi else None\n        if mol is not None:\n            compounds[cid] = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)\n    positives = []\n    with values_path.open(encoding="utf-8") as handle:\n        next(handle, None)\n        for line in handle:\n            fields = line.rstrip().split("\\t")\n            if len(fields) == 2 and fields[1] == "P" and int(fields[0]) in compounds:\n                try:\n                    positives.append(standardize_smiles(compounds[int(fields[0])]))\n                except ChemistryError:\n                    pass\n    return tuple(sorted(set(positives)))\n\n\ndef safety_veto(smiles: str, config: dict[str, Any]) -> SafetyVeto:\n    reasons: list[str] = []\n    try:\n        mol = parse_smiles(smiles)\n    except ChemistryError as exc:\n        return SafetyVeto(True, False, False, 0.0, False, (), (), (str(exc),))\n\n    allowed = set(config["elements"])\n    unsupported = tuple(sorted({atom.GetSymbol() for atom in mol.GetAtoms()} - allowed))\n    if unsupported:\n        reasons.append("unsupported_elements")\n\n    safety = config["safety"]\n    psoralen = False\n    for pattern in safety["psoralen_core_smarts"]:\n        query = _core_query(pattern)\n        if query is not None and mol.HasSubstructMatch(query):\n            psoralen = True\n            reasons.append("psoralen_or_furocoumarin_core")\n            break\n\n    canonical = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)\n    known = False\n    known_canonical: list[str] = []\n    registry = _u16_positive_registry(str(safety.get("u16_qsardb_directory", ""))) if safety.get("u16_qsardb_directory") else ()\n    for known_smiles in [*safety["known_phototoxic_smiles"], *registry]:\n        try:\n            normalized_known = standardize_smiles(known_smiles)\n            known_canonical.append(normalized_known)\n            if canonical == normalized_known:\n                known = True\n                reasons.append("known_phototoxic_structure")\n        except ChemistryError:\n            continue\n    psoralen_similarity = max_similarity(canonical, known_canonical) if known_canonical else 0.0\n    psoralen_similarity_warning = (\n        not psoralen\n        and not known\n        and psoralen_similarity >= float(safety.get("psoralen_similarity_warning_threshold", 0.45))\n    )\n\n    reactive: list[str] = []\n    for pattern in safety["reactive_fragments"]:\n        query = _query(pattern)\n        if query is not None and mol.HasSubstructMatch(query):\n            reactive.append(pattern)\n    if reactive:\n        reasons.append("reactive_or_unstable_alert")\n\n    return SafetyVeto(\n        veto=bool(psoralen or known or reactive or unsupported),\n        psoralen_alert=psoralen,\n        known_phototoxic_match=known,\n        psoralen_similarity=psoralen_similarity,\n        psoralen_similarity_warning=psoralen_similarity_warning,\n        reactive_alerts=tuple(reactive),\n        unsupported_elements=unsupported,\n        reasons=tuple(reasons),\n    )\n\n\ndef validate_pair(pair: MoleculePair, config: dict[str, Any]) -> tuple[bool, list[str]]:\n    reasons: list[str] = []\n    try:\n        ground_formula = molecular_formula(pair.smiles)\n        charged_formula = molecular_formula(pair.charged_smiles)\n        if ground_formula != charged_formula:\n            reasons.append("isomer_formula_mismatch")\n        if pair.smiles == pair.charged_smiles:\n            reasons.append("isomer_pair_identical")\n    except ChemistryError as exc:\n        reasons.append(str(exc))\n    veto = safety_veto(pair.smiles, config)\n    reasons.extend(veto.reasons)\n    return not reasons, sorted(set(reasons))\n'
load_embedded('mostgen.chemistry', _source, PROJECT_ROOT / 'mostgen' / 'chemistry.py')
print('loaded mostgen.chemistry')


loaded mostgen.chemistry


### 3.14 AUC, λc и reward transforms — `numerics.py`


In [7]:
_source = 'from __future__ import annotations\n\nimport math\nfrom statistics import fmean\nfrom typing import Iterable, Sequence\n\n\ndef trapezoid_auc(wavelengths: Sequence[float], absorbance: Sequence[float], low: float, high: float) -> float:\n    if len(wavelengths) != len(absorbance) or len(wavelengths) < 2:\n        raise ValueError("wavelength and absorbance arrays must have equal length >= 2")\n    if any(b <= a for a, b in zip(wavelengths, wavelengths[1:])):\n        raise ValueError("wavelengths must be strictly increasing")\n    if low >= high:\n        raise ValueError("integration interval must have positive width")\n    points: list[tuple[float, float]] = []\n    for x, y in zip(wavelengths, absorbance):\n        if low <= x <= high:\n            points.append((float(x), float(y)))\n    for edge in (low, high):\n        if not any(abs(x - edge) < 1e-12 for x, _ in points):\n            for index in range(len(wavelengths) - 1):\n                left, right = wavelengths[index], wavelengths[index + 1]\n                if left <= edge <= right:\n                    ratio = (edge - left) / (right - left)\n                    y = absorbance[index] + ratio * (absorbance[index + 1] - absorbance[index])\n                    points.append((edge, float(y)))\n                    break\n    points.sort()\n    return sum((x2 - x1) * (y1 + y2) / 2.0 for (x1, y1), (x2, y2) in zip(points, points[1:]))\n\n\ndef critical_wavelength(wavelengths: Sequence[float], absorbance: Sequence[float], fraction: float = 0.90) -> float:\n    if not 0.0 < fraction < 1.0:\n        raise ValueError("fraction must be between zero and one")\n    total = trapezoid_auc(wavelengths, absorbance, wavelengths[0], wavelengths[-1])\n    if total <= 0.0:\n        return float(wavelengths[0])\n    target = total * fraction\n    cumulative = 0.0\n    for (x1, y1), (x2, y2) in zip(zip(wavelengths, absorbance), zip(wavelengths[1:], absorbance[1:])):\n        segment = (x2 - x1) * (y1 + y2) / 2.0\n        if cumulative + segment >= target:\n            if segment <= 0.0:\n                return float(x2)\n            return float(x1 + (x2 - x1) * (target - cumulative) / segment)\n        cumulative += segment\n    return float(wavelengths[-1])\n\n\ndef beer_lambert_transmittance(absorbance: Iterable[float], loading_scale: float = 1.0) -> float:\n    values = [10.0 ** (-max(0.0, float(value)) * loading_scale) for value in absorbance]\n    return fmean(values) if values else 1.0\n\n\ndef sigmoid(value: float, midpoint: float, scale: float) -> float:\n    if scale <= 0:\n        raise ValueError("scale must be positive")\n    exponent = max(-60.0, min(60.0, -(value - midpoint) / scale))\n    return 1.0 / (1.0 + math.exp(exponent))\n\n\ndef interval_score(value: float, low: float, high: float, softness: float) -> float:\n    if not low < high:\n        raise ValueError("low must be less than high")\n    return sigmoid(value, low, softness) * sigmoid(high - value, 0.0, softness)\n\n\ndef weighted_geometric_mean(\n    components: dict[str, float],\n    weights: dict[str, float],\n    floor: float = 1e-3,\n) -> float:\n    if not components:\n        return 0.0\n    numerator = 0.0\n    denominator = 0.0\n    for key, raw in components.items():\n        weight = float(weights.get(key, 1.0))\n        if weight <= 0.0:\n            continue\n        value = max(floor, min(1.0, float(raw)))\n        numerator += weight * math.log(value)\n        denominator += weight\n    return math.exp(numerator / denominator) if denominator else 0.0\n\n\ndef lower_confidence_bound(mean: float, std: float, z: float = 1.645) -> float:\n    return float(mean) - float(z) * max(0.0, float(std))\n\n\ndef upper_confidence_bound(mean: float, std: float, z: float = 1.645) -> float:\n    return float(mean) + float(z) * max(0.0, float(std))\n\n'
load_embedded('mostgen.numerics', _source, PROJECT_ROOT / 'mostgen' / 'numerics.py')
print('loaded mostgen.numerics')


loaded mostgen.numerics


### 3.16 Reaction library и dispatch данных — `data.py`


In [8]:
_source = 'from __future__ import annotations\n\nimport csv\nimport json\nimport math\nimport random\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nfrom .chemistry import build_pair, descriptors, murcko_scaffold, parse_smiles, safety_veto\nfrom .numerics import critical_wavelength, trapezoid_auc\nfrom .provenance import sha256_file, stable_hash\n\n\nWAVELENGTHS = tuple(range(290, 401, 5))\n\n\ndef write_csv(path: str | Path, rows: Iterable[dict[str, Any]], fieldnames: list[str] | None = None) -> int:\n    materialized = list(rows)\n    destination = Path(path)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if fieldnames is None:\n        fieldnames = list(materialized[0]) if materialized else []\n    with destination.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")\n        if fieldnames:\n            writer.writeheader()\n            writer.writerows(materialized)\n    return len(materialized)\n\n\ndef read_csv(path: str | Path) -> list[dict[str, str]]:\n    with Path(path).open(encoding="utf-8", newline="") as handle:\n        return list(csv.DictReader(handle))\n\n\ndef _noise(key: str, magnitude: float) -> float:\n    integer = int(stable_hash(key, 12), 16)\n    return magnitude * (2.0 * (integer / float(16**12 - 1)) - 1.0)\n\n\ndef _gaussian(x: float, centre: float, width: float, amplitude: float) -> float:\n    return amplitude * math.exp(-0.5 * ((x - centre) / width) ** 2)\n\n\ndef reference_labels(smiles: str, charged_smiles: str, family: str) -> dict[str, float]:\n    """Deterministic fixture labels used only by the smoke backend.\n\n    Values are smooth functions of RDKit descriptors plus deterministic noise,\n    which makes leakage/error tests meaningful without pretending that the\n    records are experimental observations.\n    """\n    d = descriptors(smiles)\n    q = descriptors(charged_smiles)\n    family_peak = {"nbd_qc": (314.0, 366.0), "dewar_pyrimidinone": (307.0, 354.0), "spiropyran": (325.0, 382.0)}[family]\n    family_energy = {"nbd_qc": 91.0, "dewar_pyrimidinone": 55.0, "spiropyran": 32.0}[family]\n    family_log_half = {"nbd_qc": 1.02, "dewar_pyrimidinone": 0.78, "spiropyran": 0.62}[family]\n    shift = 3.5 * d["aromatic_rings"] + 0.065 * d["tpsa"] + 1.7 * d["logp"]\n    donor_acceptor = min(2.0, d["hba"] / 3.0 + d["hbd"] / 2.0)\n    peak_a = family_peak[0] + 0.30 * shift + _noise(smiles + "p1", 4.0)\n    peak_b = family_peak[1] + 0.65 * shift + _noise(smiles + "p2", 6.0)\n    amp_a = 0.45 + 0.065 * d["aromatic_rings"] + 0.025 * donor_acceptor\n    amp_b = 0.22 + 0.095 * donor_acceptor + 0.035 * d["aromatic_rings"]\n    spectrum = [\n        max(0.0, _gaussian(w, peak_a, 17.0, amp_a) + _gaussian(w, peak_b, 29.0, amp_b) + _noise(f"{smiles}:{w}", 0.012))\n        for w in WAVELENGTHS\n    ]\n    uvb_auc = trapezoid_auc(WAVELENGTHS, spectrum, 290.0, 320.0)\n    uva_auc = trapezoid_auc(WAVELENGTHS, spectrum, 320.0, 400.0)\n    lambda_c = critical_wavelength(WAVELENGTHS, spectrum)\n    delta_descriptor = abs(q["fraction_csp3"] - d["fraction_csp3"]) + abs(q["charge_separation"] - d["charge_separation"]) * 0.25\n    energy = max(3.0, family_energy + 7.0 * delta_descriptor + 1.2 * d["aromatic_rings"] - 0.035 * d["mol_wt"] + _noise(smiles + "dh", 8.0))\n    specific = energy * 277.7777778 / max(1.0, d["mol_wt"])\n    log_half = family_log_half + 0.10 * d["aromatic_rings"] + 0.04 * d["logp"] - 0.025 * d["rotatable_bonds"] + _noise(smiles + "t", 0.24)\n    kp = -5.30 + 0.42 * d["logp"] - 0.012 * d["mol_wt"] - 0.018 * d["tpsa"] + _noise(smiles + "kp", 0.30)\n    photo_logit = -2.2 + 0.58 * d["aromatic_rings"] + 0.20 * max(0.0, d["logp"] - 3.0) + 0.15 * donor_acceptor\n    phototoxicity = 1.0 / (1.0 + math.exp(-photo_logit))\n    phototoxicity = min(0.98, max(0.02, phototoxicity + _noise(smiles + "pt", 0.08)))\n    result = {\n        "uvb_auc": uvb_auc,\n        "uva_auc": uva_auc,\n        "lambda_c_nm": lambda_c,\n        "energy_kj_mol": energy,\n        "specific_energy_wh_kg": specific,\n        "log_half_life_h": log_half,\n        "kp_log_cm_s": kp,\n        "phototoxicity_probability": phototoxicity,\n    }\n    result.update({f"abs_{w}": value for w, value in zip(WAVELENGTHS, spectrum)})\n    return result\n\n\ndef scaffold_split(scaffold: str) -> str:\n    bucket = int(stable_hash(scaffold, 8), 16) % 10\n    if bucket < 7:\n        return "train"\n    if bucket < 9:\n        return "validation"\n    return "test"\n\n\ndef enumerate_library(config: dict[str, Any]) -> list[dict[str, Any]]:\n    rows: list[dict[str, Any]] = []\n    prior_elements = set(config.get("production", {}).get("libinvent_supported_elements", config["elements"]))\n    synthon_ids = [\n        entry["id"] for entry in config["synthons"]\n        if {atom.GetSymbol() for atom in parse_smiles(entry["smiles"]).GetAtoms()} <= prior_elements\n    ]\n    for family in sorted(config["families"]):\n        for first in synthon_ids:\n            for second in synthon_ids:\n                try:\n                    pair = build_pair(config, family, first, second)\n                except ValueError:\n                    continue\n                veto = safety_veto(pair.smiles, config)\n                if veto.veto:\n                    continue\n                rows.append({\n                    "candidate_id": stable_hash(f"{family}|{first}|{second}", 20),\n                    "family": family,\n                    "synthon_a": first,\n                    "synthon_b": second,\n                    "smiles": pair.smiles,\n                    "charged_smiles": pair.charged_smiles,\n                    "formula": pair.formula,\n                    "scaffold": murcko_scaffold(pair.smiles),\n                })\n    # Canonical structure identity wins over alternate synthon encodings.\n    unique: dict[tuple[str, str], dict[str, Any]] = {}\n    for row in rows:\n        unique.setdefault((row["family"], row["smiles"]), row)\n    return list(unique.values())\n\n\ndef _ensure_split_coverage(rows: list[dict[str, Any]]) -> None:\n    """Keep scaffolds intact while ensuring every split is populated globally."""\n    scaffolds = sorted({row["scaffold"] for row in rows})\n    assignments = {scaffold: scaffold_split(scaffold) for scaffold in scaffolds}\n    present = set(assignments.values())\n    for wanted, index in (("validation", -2), ("test", -1)):\n        if wanted not in present and len(scaffolds) >= abs(index):\n            assignments[scaffolds[index]] = wanted\n    for row in rows:\n        row["split"] = assignments[row["scaffold"]]\n\n\ndef prepare_data(config: dict[str, Any], output_dir: str | Path, root: str | Path) -> dict[str, Any]:\n    destination = Path(output_dir)\n    destination.mkdir(parents=True, exist_ok=True)\n    root_path = Path(root)\n    library = enumerate_library(config)\n    if config["execution"].get("mode") != "smoke":\n        # Import lazily: the smoke fixture remains lightweight, while full and\n        # production modes are prohibited from manufacturing endpoint labels.\n        from .real_data import prepare_real_data\n\n        manifest = prepare_real_data(config, destination, root_path, library)\n        manifest["training_rows"] = sum(manifest["endpoint_rows"].values())\n        manifest["library_rows"] = manifest["reaction_library_rows"]\n        return manifest\n    per_family = int(config["training_rows_per_family"])\n    rng = random.Random(int(config["project"]["default_seed"]))\n    selected: list[dict[str, Any]] = []\n    for family in sorted(config["families"]):\n        members = [row for row in library if row["family"] == family]\n        rng.shuffle(members)\n        selected.extend(members[:per_family])\n    if len(selected) > int(config["project"]["max_training_structures"]):\n        raise ValueError("Prepared training corpus exceeds configured 30,000-row cap")\n    _ensure_split_coverage(selected)\n    source_by_family = {"nbd_qc": "M01", "dewar_pyrimidinone": "M03/M04", "spiropyran": "M01/M04"}\n    training_rows: list[dict[str, Any]] = []\n    for row in selected:\n        labels = reference_labels(row["smiles"], row["charged_smiles"], row["family"])\n        training_rows.append({\n            **row,\n            "source_id": source_by_family[row["family"]],\n            "source_kind": "deterministic_fixture",\n            "evidence_tier": "synthetic_smoke_only",\n            "state": "ground_and_charged_pair",\n            "temperature_k": 305.0,\n            "medium": "conditional_standard_medium",\n            "label_level": "molecule_state_condition",\n            **labels,\n        })\n    training_path = destination / "reviewer_training.csv"\n    write_csv(training_path, training_rows)\n    library_path = destination / "reaction_library.csv"\n    write_csv(library_path, library)\n    local_inputs = []\n    for name in ("database_matrix_MOST_UV_skin.xlsx", "Задание.docx", "Солнцезащитная плёнка с молекулярным накоплением солнечной энергии.pptx"):\n        path = root_path / name\n        if path.exists():\n            local_inputs.append({"path": str(path.resolve()), "bytes": path.stat().st_size, "sha256": sha256_file(path)})\n    provenance = {\n        "schema_version": "1.0",\n        "raw_inputs_mutated": False,\n        "derivative_data": str(training_path.resolve()),\n        "training_rows": len(training_rows),\n        "library_rows": len(library),\n        "local_inputs": local_inputs,\n        "source_registry": config["sources"],\n        "warning": "Fixture labels are synthetic and cannot support scientific, efficacy, or safety claims.",\n    }\n    (destination / "data_manifest.json").write_text(json.dumps(provenance, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    return provenance\n'
load_embedded('mostgen.data', _source, PROJECT_ROOT / 'mostgen' / 'data.py')
print('loaded mostgen.data')


loaded mostgen.data


### 3.18 Общий reviewer interface и smoke fixtures — `reviewers.py`


In [9]:
_source = 'from __future__ import annotations\n\nimport json\nimport math\nimport pickle\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom statistics import median\nfrom typing import Any, Iterable\n\nimport numpy as np\nfrom rdkit import DataStructs\nfrom sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor\nfrom sklearn.metrics import mean_absolute_error, root_mean_squared_error\n\nfrom .chemistry import descriptors, fingerprint, max_similarity\nfrom .data import WAVELENGTHS, read_csv\nfrom .provenance import stable_hash\n\n\nDESCRIPTOR_NAMES = (\n    "mol_wt", "logp", "tpsa", "hbd", "hba", "rings", "aromatic_rings",\n    "rotatable_bonds", "hetero_fraction", "fraction_csp3", "formal_charge",\n    "charge_separation", "heavy_atoms",\n)\nFAMILIES = ("nbd_qc", "dewar_pyrimidinone", "spiropyran")\nSPECTRAL_TARGETS = tuple(f"abs_{w}" for w in WAVELENGTHS)\nMOST_TARGETS = ("energy_kj_mol", "specific_energy_wh_kg", "log_half_life_h")\nSAFETY_TARGETS = ("kp_log_cm_s", "phototoxicity_probability")\n\n\ndef feature_vector(smiles: str, family: str, bits: int) -> np.ndarray:\n    fp = fingerprint(smiles, bits)\n    fp_array = np.zeros(bits, dtype=np.float32)\n    DataStructs.ConvertToNumpyArray(fp, fp_array)\n    desc = descriptors(smiles)\n    scaled = np.asarray([\n        desc["mol_wt"] / 500.0,\n        desc["logp"] / 6.0,\n        desc["tpsa"] / 150.0,\n        desc["hbd"] / 5.0,\n        desc["hba"] / 10.0,\n        desc["rings"] / 8.0,\n        desc["aromatic_rings"] / 6.0,\n        desc["rotatable_bonds"] / 12.0,\n        desc["hetero_fraction"],\n        desc["fraction_csp3"],\n        desc["formal_charge"] / 3.0,\n        desc["charge_separation"] / 4.0,\n        desc["heavy_atoms"] / 60.0,\n    ], dtype=np.float32)\n    one_hot = np.asarray([1.0 if family == item else 0.0 for item in FAMILIES], dtype=np.float32)\n    return np.concatenate([fp_array, scaled, one_hot])\n\n\n@dataclass\nclass Prediction:\n    mean: dict[str, float]\n    std: dict[str, float]\n\n\nclass ReviewerBundle:\n    """Serializable independent ensemble of spectrum, MOST, and safety models."""\n\n    def __init__(\n        self,\n        purpose: str,\n        bits: int,\n        spectrum_models: list[Any],\n        safety_models: list[Any],\n        most_models: dict[str, list[Any]],\n        references: dict[str, Any],\n        metrics: dict[str, Any],\n    ) -> None:\n        self.purpose = purpose\n        self.bits = bits\n        self.spectrum_models = spectrum_models\n        self.safety_models = safety_models\n        self.most_models = most_models\n        self.references = references\n        self.metrics = metrics\n\n    @staticmethod\n    def _ensemble_predict(models: list[Any], vector: np.ndarray, targets: tuple[str, ...]) -> Prediction:\n        member_values = []\n        sample = vector.reshape(1, -1)\n        for model in models:\n            member_values.append(np.asarray(model.predict(sample)[0], dtype=float).reshape(-1))\n        values = np.vstack(member_values)\n        return Prediction(\n            mean={key: float(value) for key, value in zip(targets, values.mean(axis=0))},\n            std={key: float(value) for key, value in zip(targets, values.std(axis=0, ddof=0))},\n        )\n\n    def predict(self, smiles: str, family: str) -> dict[str, Any]:\n        return self.predict_many([(smiles, family)])[0]\n\n    def predict_many(self, molecules: list[tuple[str, str]]) -> list[dict[str, Any]]:\n        if not molecules:\n            return []\n        x = np.vstack([feature_vector(smiles, family, self.bits) for smiles, family in molecules])\n\n        def batch(models: list[Any], targets: tuple[str, ...], indices: list[int] | None = None):\n            selected_x = x if indices is None else x[indices]\n            values = np.asarray([model.predict(selected_x) for model in models], dtype=float)\n            means, stds = values.mean(axis=0), values.std(axis=0, ddof=0)\n            return [\n                Prediction(\n                    mean={key: float(value) for key, value in zip(targets, means[index].reshape(-1))},\n                    std={key: float(value) for key, value in zip(targets, stds[index].reshape(-1))},\n                )\n                for index in range(len(selected_x))\n            ]\n\n        spectrum = batch(self.spectrum_models, SPECTRAL_TARGETS)\n        safety = batch(self.safety_models, SAFETY_TARGETS)\n        most: list[Prediction | None] = [None] * len(molecules)\n        for family in FAMILIES:\n            indices = [index for index, (_, item_family) in enumerate(molecules) if item_family == family]\n            if not indices:\n                continue\n            predictions = batch(self.most_models[family], MOST_TARGETS, indices)\n            for index, prediction in zip(indices, predictions):\n                most[index] = prediction\n        results = []\n        spectral_refs = self.references["spectral_smiles"]\n        for index, (smiles, family) in enumerate(molecules):\n            results.append({\n                "spectrum": spectrum[index], "most": most[index], "safety": safety[index],\n                "ad_spectral_similarity": max_similarity(smiles, spectral_refs, self.bits),\n                "ad_most_similarity": max_similarity(smiles, self.references["most_smiles_by_family"].get(family, []), self.bits),\n                "family_reference_medians": self.references["family_medians"][family],\n                "reviewer_purpose": self.purpose,\n            })\n        return results\n\n    def save(self, path: str | Path) -> None:\n        destination = Path(path)\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        with destination.open("wb") as handle:\n            pickle.dump(self, handle, protocol=pickle.HIGHEST_PROTOCOL)\n\n    @classmethod\n    def load(cls, path: str | Path) -> "ReviewerBundle":\n        with Path(path).open("rb") as handle:\n            bundle = pickle.load(handle)\n        if not isinstance(bundle, cls) and not (\n            hasattr(bundle, "predict_many") and hasattr(bundle, "purpose")\n        ):\n            raise TypeError("Reviewer artifact has an unexpected type")\n        return bundle\n\n\ndef _matrix(rows: list[dict[str, str]], bits: int) -> np.ndarray:\n    return np.vstack([feature_vector(row["smiles"], row["family"], bits) for row in rows])\n\n\ndef _targets(rows: list[dict[str, str]], names: tuple[str, ...]) -> np.ndarray:\n    return np.asarray([[float(row[name]) for name in names] for row in rows], dtype=np.float64)\n\n\ndef _new_model(kind: str, seed: int):\n    common = dict(n_estimators=28, random_state=seed, n_jobs=1, min_samples_leaf=2, max_features=0.65)\n    if kind == "reward":\n        return RandomForestRegressor(bootstrap=True, **common)\n    return ExtraTreesRegressor(bootstrap=False, **common)\n\n\ndef _fit_members(kind: str, count: int, seed: int, x: np.ndarray, y: np.ndarray) -> list[Any]:\n    models = []\n    for index in range(count):\n        model = _new_model(kind, seed + 1009 * index)\n        model.fit(x, y)\n        models.append(model)\n    return models\n\n\ndef _predict_mean(models: list[Any], x: np.ndarray) -> np.ndarray:\n    return np.mean([np.asarray(model.predict(x), dtype=float) for model in models], axis=0)\n\n\ndef _split_metrics(\n    rows: list[dict[str, str]],\n    bits: int,\n    spectrum_models: list[Any],\n    safety_models: list[Any],\n    most_models: dict[str, list[Any]],\n) -> dict[str, Any]:\n    results: dict[str, Any] = {}\n    for split in ("validation", "test"):\n        subset = [row for row in rows if row["split"] == split]\n        if not subset:\n            results[split] = {"n": 0, "status": "not_available"}\n            continue\n        x = _matrix(subset, bits)\n        result: dict[str, Any] = {"n": len(subset)}\n        for group, targets, models in (\n            ("spectrum", SPECTRAL_TARGETS, spectrum_models),\n            ("safety", SAFETY_TARGETS, safety_models),\n        ):\n            true = _targets(subset, targets)\n            pred = _predict_mean(models, x)\n            result[group] = {\n                "mae": float(mean_absolute_error(true, pred)),\n                "rmse": float(root_mean_squared_error(true, pred)),\n            }\n        most_errors = []\n        for family in FAMILIES:\n            members = [row for row in subset if row["family"] == family]\n            if not members:\n                continue\n            family_x = _matrix(members, bits)\n            true = _targets(members, MOST_TARGETS)\n            pred = _predict_mean(most_models[family], family_x)\n            most_errors.extend((true - pred).reshape(-1).tolist())\n        result["most"] = {\n            "mae": float(np.mean(np.abs(most_errors))) if most_errors else math.nan,\n            "rmse": float(np.sqrt(np.mean(np.square(most_errors)))) if most_errors else math.nan,\n        }\n        results[split] = result\n    return results\n\n\ndef _check_scaffold_leakage(rows: list[dict[str, str]]) -> None:\n    split_scaffolds: dict[str, set[str]] = {}\n    for row in rows:\n        split_scaffolds.setdefault(row["split"], set()).add(row["scaffold"])\n    names = sorted(split_scaffolds)\n    for index, first in enumerate(names):\n        for second in names[index + 1:]:\n            overlap = split_scaffolds[first] & split_scaffolds[second]\n            if overlap:\n                raise ValueError(f"Scaffold leakage between {first} and {second}: {sorted(overlap)[:3]}")\n\n\ndef _family_medians(rows: Iterable[dict[str, str]]) -> dict[str, dict[str, float]]:\n    rows = list(rows)\n    medians: dict[str, dict[str, float]] = {}\n    for family in FAMILIES:\n        members = [row for row in rows if row["family"] == family]\n        if not members:\n            raise ValueError(f"No training references for family {family}")\n        medians[family] = {\n            key: float(median(float(row[key]) for row in members))\n            for key in ("uvb_auc", "uva_auc", "energy_kj_mol", "specific_energy_wh_kg", "log_half_life_h")\n        }\n    return medians\n\n\ndef _train_bundle(rows: list[dict[str, str]], config: dict[str, Any], purpose: str) -> ReviewerBundle:\n    _check_scaffold_leakage(rows)\n    train = [row for row in rows if row["split"] == "train"]\n    if not train:\n        raise ValueError("Training split is empty")\n    bits = int(config["reviewers"]["fingerprint_bits"])\n    members = int(config["reviewers"]["ensemble_size"])\n    base_seed = int(config["project"]["default_seed"]) + (0 if purpose == "reward" else 500_009)\n    x = _matrix(train, bits)\n    spectrum_models = _fit_members(purpose, members, base_seed + 11, x, _targets(train, SPECTRAL_TARGETS))\n    safety_models = _fit_members(purpose, members, base_seed + 23, x, _targets(train, SAFETY_TARGETS))\n    most_models: dict[str, list[Any]] = {}\n    for offset, family in enumerate(FAMILIES):\n        family_rows = [row for row in train if row["family"] == family]\n        if len(family_rows) < 5:\n            raise ValueError(f"Insufficient family-aware training data for {family}")\n        most_models[family] = _fit_members(\n            purpose, members, base_seed + 101 + offset,\n            _matrix(family_rows, bits), _targets(family_rows, MOST_TARGETS),\n        )\n    metrics = _split_metrics(rows, bits, spectrum_models, safety_models, most_models)\n    references = {\n        "spectral_smiles": [row["smiles"] for row in train],\n        "most_smiles_by_family": {family: [row["smiles"] for row in train if row["family"] == family] for family in FAMILIES},\n        "family_medians": _family_medians(train),\n        "training_data_hash": stable_hash("\\n".join(sorted(row["candidate_id"] for row in train)), 32),\n        "training_scaffolds": sorted({row["scaffold"] for row in train}),\n    }\n    return ReviewerBundle(purpose, bits, spectrum_models, safety_models, most_models, references, metrics)\n\n\ndef train_reviewers(config: dict[str, Any], data_path: str | Path, output_dir: str | Path) -> dict[str, Any]:\n    rows = read_csv(data_path)\n    if rows and "endpoint" in rows[0]:\n        from .real_reviewers import train_real_reviewers\n\n        return train_real_reviewers(config, data_path, output_dir)\n    if len(rows) > int(config["project"]["max_training_structures"]):\n        raise ValueError("Reviewer training rows exceed configured cap")\n    tiers = {row.get("evidence_tier") for row in rows}\n    if config["execution"]["mode"] == "production" and tiers == {"synthetic_smoke_only"}:\n        raise RuntimeError("Production mode refuses synthetic-only reviewer data")\n    destination = Path(output_dir)\n    reward = _train_bundle(rows, config, "reward")\n    evaluator = _train_bundle(rows, config, "evaluator")\n    reward_path = destination / "reward" / "reviewers.pkl"\n    evaluator_path = destination / "evaluator" / "reviewers.pkl"\n    reward.save(reward_path)\n    evaluator.save(evaluator_path)\n    metadata = {\n        "schema_version": "1.0",\n        "rows": len(rows),\n        "evidence_tiers": sorted(str(item) for item in tiers),\n        "feature_schema": [f"morgan_{i}" for i in range(reward.bits)] + list(DESCRIPTOR_NAMES) + [f"family_{item}" for item in FAMILIES],\n        "reward": {"algorithm": "RandomForestRegressor ensemble", "path": str(reward_path.resolve()), "metrics": reward.metrics},\n        "evaluator": {"algorithm": "ExtraTreesRegressor ensemble", "path": str(evaluator_path.resolve()), "metrics": evaluator.metrics},\n        "independent_instances": reward_path.resolve() != evaluator_path.resolve(),\n        "limitations": [\n            "Synthetic smoke labels are not experimental observations.",\n            "Applicability domains are fingerprint-neighbourhood proxies.",\n            "Phototoxicity output is conservative triage, never proof of safety.",\n        ],\n    }\n    destination.mkdir(parents=True, exist_ok=True)\n    (destination / "model_cards.json").write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    return metadata\n'
load_embedded('mostgen.reviewers', _source, PROJECT_ROOT / 'mostgen' / 'reviewers.py')
print('loaded mostgen.reviewers')


loaded mostgen.reviewers


### 3.20 Парсеры реальных M/U endpoint-источников — `real_data.py`


In [10]:
_source = 'from __future__ import annotations\n\n"""Parsing of the versioned experimental sources used by production reviewers.\n\nEvery output row is long-form: one molecular endpoint under one set of\nconditions.  No missing experimental value is imputed and no lambda-max record\nis relabelled as a full spectrum.\n"""\n\nimport csv\nimport json\nimport math\nimport re\nimport xml.etree.ElementTree as ET\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nimport pandas as pd\nfrom rdkit import Chem\nfrom rdkit.Chem.Scaffolds import MurckoScaffold\n\nfrom .chemistry import ChemistryError, murcko_scaffold, standardize_smiles\nfrom .data import scaffold_split, write_csv\nfrom .numerics import critical_wavelength, trapezoid_auc\nfrom .provenance import sha256_file, stable_hash\n\n\ndef _finite(value: Any) -> float | None:\n    try:\n        number = float(value)\n    except (TypeError, ValueError):\n        return None\n    return number if math.isfinite(number) else None\n\n\ndef _standardize(value: Any) -> str | None:\n    if value is None or pd.isna(value) or str(value).strip().lower() in {"", "-", "nan", "none"}:\n        return None\n    try:\n        return standardize_smiles(str(value))\n    except (ChemistryError, TypeError, ValueError):\n        return None\n\n\ndef _row(smiles: str, endpoint: str, target: float, source: str, **metadata: Any) -> dict[str, Any]:\n    mol = Chem.MolFromSmiles(smiles)\n    scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=True) if mol is not None else ""\n    scaffold = scaffold or smiles\n    return {\n        "record_id": stable_hash(f"{source}|{endpoint}|{smiles}|{metadata}|{target}", 24),\n        "smiles": smiles,\n        "family": metadata.pop("family", "unassigned"),\n        "scaffold": scaffold,\n        "split": scaffold_split(scaffold),\n        "endpoint": endpoint,\n        "target": target,\n        "source_id": source,\n        "evidence_tier": "experimental",\n        **metadata,\n    }\n\n\ndef parse_m13(path: Path, max_rows: int = 25_000) -> list[dict[str, Any]]:\n    frame = pd.read_csv(path)\n    rows: list[dict[str, Any]] = []\n    seen: set[tuple[str, str, float, str]] = set()\n    for record in frame.to_dict("records"):\n        smiles = _standardize(record.get("smiles"))\n        peak = _finite(record.get("peakwavs_max"))\n        solvent = str(record.get("solvent") or "unknown")\n        origin = str(record.get("source") or "unknown").strip().lower()\n        # UV/VisML is the unified M13 distribution, but two of its constituent\n        # tables are the M11/M12 sources from the project registry.  Preserve\n        # that identity in every endpoint row instead of flattening all\n        # provenance to M13.\n        source_id = {"deep4chem": "M11", "cdex": "M12"}.get(origin, "M13")\n        if smiles is None or peak is None or not 180.0 <= peak <= 900.0:\n            continue\n        key = (smiles, solvent, round(peak, 4), origin)\n        if key in seen:\n            continue\n        seen.add(key)\n        rows.append(_row(smiles, "lambda_max_nm", peak, source_id, solvent=solvent, original_source=origin,\n                         state="unspecified", label_level="molecule_condition"))\n    # A content-derived selection is deterministic and independent of source order.\n    rows.sort(key=lambda item: stable_hash(item["record_id"], 32))\n    return rows[:max_rows]\n\n\ndef parse_m01(path: Path) -> list[dict[str, Any]]:\n    frame = pd.read_csv(path)\n    rows: list[dict[str, Any]] = []\n    peak_columns = {\n        "E isomer pi-pi* wavelength in nm": "E_pi_pi",\n        "E isomer n-pi* wavelength in nm": "E_n_pi",\n        "Z isomer pi-pi* wavelength in nm": "Z_pi_pi",\n        "Z isomer n-pi* wavelength in nm": "Z_n_pi",\n    }\n    for record in frame.to_dict("records"):\n        smiles = _standardize(record.get("SMILES"))\n        if smiles is None:\n            continue\n        solvent = str(record.get("Irradiation solvent") or "unknown")\n        for column, state in peak_columns.items():\n            peak = _finite(record.get(column))\n            if peak is not None and 180.0 <= peak <= 900.0:\n                rows.append(_row(smiles, "lambda_max_nm", peak, "M01", solvent=solvent,\n                                 state=state, label_level="molecule_state_condition"))\n        rate = _finite(record.get("rate of thermal isomerisation from Z-E in s-1"))\n        if rate is not None and rate > 0:\n            half_life_h = math.log(2.0) / rate / 3600.0\n            rows.append(_row(smiles, "log_half_life_h", math.log10(half_life_h), "M01",\n                             solvent=str(record.get("Solvent used for thermal isomerisation rates") or "unknown"),\n                             temperature_k="not_reported_in_table", state="Z_to_E",\n                             label_level="molecule_state_condition"))\n    return rows\n\n\ndef parse_u12_skinpix(path: Path) -> list[dict[str, Any]]:\n    # The workbook has an incorrect A1 dimension; non-read-only pandas/openpyxl\n    # parsing still exposes all 202 records.\n    frame = pd.read_excel(path, sheet_name="default_1")\n    rows = []\n    for record in frame.to_dict("records"):\n        smiles = _standardize(record.get("SMILES"))\n        target = _finite(record.get("logKp (cm/s) (converted)"))\n        if smiles is None or target is None:\n            continue\n        rows.append(_row(\n            smiles, "kp_log_cm_s", target, "U12",\n            cas=str(record.get("CAS number") or ""), compound_name=str(record.get("compound name") or ""),\n            donor=str(record.get("category donor type") or "unknown"),\n            acceptor=str(record.get("category acceptor type") or "unknown"),\n            skin=str(record.get("used layer") or "unknown"),\n            temperature_c=record.get("donor/skin surface temperature (°C)"),\n            relation=str(record.get("Kp relation") or "="),\n            doi=str(record.get("doi") or ""), label_level="molecule_condition",\n        ))\n    return rows\n\n\ndef parse_u13_permeation(path: Path, u09_path: Path, u12_path: Path) -> list[dict[str, Any]]:\n    """Parse U13 and retain only exact CAS structures available locally.\n\n    U13 reports log10(Kp / cm h-1) but no SMILES.  Structures are joined by\n    exact CAS to the two already versioned workbooks and the target is converted\n    to log10(cm s-1) by subtracting log10(3600).  Unmapped CAS are not guessed.\n    """\n    mappings: dict[str, str] = {}\n    hppt = pd.read_excel(u09_path, sheet_name="HPPT Data")\n    for record in hppt.to_dict("records"):\n        cas = str(record.get("CASRN") or "").strip()\n        smiles = _standardize(record.get("QSAR.Ready.SMILES") or record.get("SMILES"))\n        if cas and smiles:\n            mappings[cas] = smiles\n    skinpix = pd.read_excel(u12_path, sheet_name="default_1")\n    for record in skinpix.to_dict("records"):\n        cas = str(record.get("CAS number") or "").strip()\n        smiles = _standardize(record.get("SMILES"))\n        if cas and smiles:\n            mappings[cas] = smiles\n    frame = pd.read_excel(path, sheet_name="Sheet1")\n    rows = []\n    for record in frame.to_dict("records"):\n        cas = str(record.get("CAS No") or "").strip()\n        smiles = mappings.get(cas)\n        log_kp_cm_h = _finite(record.get("logkpl"))\n        if not smiles or log_kp_cm_h is None:\n            continue\n        rows.append(_row(\n            smiles, "kp_log_cm_s", log_kp_cm_h - math.log10(3600.0), "U13",\n            cas=cas, compound_name=str(record.get("Compound") or ""),\n            original_value=log_kp_cm_h, original_unit="log10(cm/h)",\n            conversion_rule="log10(cm/s)=log10(cm/h)-log10(3600)",\n            experimental_temperature_k=record.get("Texpi"),\n            skin_thickness=record.get("Skin thicknessj"),\n            skin_integrity_test=str(record.get("Skin Integrity testk") or ""),\n            original_set=str(record.get("set") or ""), reference=str(record.get("Reference") or ""),\n            label_level="molecule_condition",\n        ))\n    return rows\n\n\ndef parse_u09_hppt(path: Path) -> list[dict[str, Any]]:\n    frame = pd.read_excel(path, sheet_name="HPPT Data")\n    rows = []\n    for record in frame.to_dict("records"):\n        smiles = _standardize(record.get("QSAR.Ready.SMILES") or record.get("SMILES"))\n        call = str(record.get("Call") or "").strip().lower()\n        if smiles is None or call not in {"active", "inactive"}:\n            continue\n        rows.append(_row(\n            smiles, "skin_sensitization", 1.0 if call == "active" else 0.0, "U09",\n            cas=str(record.get("CASRN") or ""), dtxsid=str(record.get("DTXSID") or ""),\n            test_type=str(record.get("Test.Type") or ""), vehicle=str(record.get("Vehicle") or ""),\n            concentration_percent=record.get("Conc.(%)"), dose_ug_cm2=record.get("DSA.(μg/cm2)"),\n            n_subjects=record.get("No.Test.Subjects"), label_level="molecule_condition_human",\n        ))\n    return rows\n\n\ndef parse_u07_irritation(path: Path, structure_source: Path) -> list[dict[str, Any]]:\n    """Join NICE calls to structures by DTXSID without guessing missing IDs.\n\n    The U07 workbook intentionally carries assay records rather than SMILES.\n    U09 is an official NICE workbook containing DTXSID and QSAR-ready SMILES;\n    only exact identifiers present in both sources are retained.\n    """\n    structures = pd.read_excel(structure_source, sheet_name="HPPT Data")\n    dtxsid_to_smiles = {}\n    for record in structures.to_dict("records"):\n        dtxsid = str(record.get("DTXSID") or "").strip()\n        smiles = _standardize(record.get("QSAR.Ready.SMILES") or record.get("SMILES"))\n        if dtxsid and smiles:\n            dtxsid_to_smiles[dtxsid] = smiles\n    frame = pd.read_excel(path, sheet_name="Data")\n    rows = []\n    for record in frame.to_dict("records"):\n        if str(record.get("Endpoint") or "").strip().lower() != "call":\n            continue\n        call = str(record.get("Response") or "").strip().lower()\n        if call not in {"active", "inactive"}:\n            continue\n        dtxsid = str(record.get("DTXSID") or "").strip()\n        smiles = dtxsid_to_smiles.get(dtxsid)\n        if not smiles:\n            continue\n        rows.append(_row(\n            smiles, "skin_irritation", 1.0 if call == "active" else 0.0, "U07",\n            dtxsid=dtxsid, cas=str(record.get("CASRN") or ""),\n            chemical_name=str(record.get("Chemical Name") or ""),\n            assay=str(record.get("Assay") or ""), species=str(record.get("Species") or ""),\n            route=str(record.get("Route") or ""), concentration=record.get("Concentration"),\n            concentration_unit=str(record.get("Concentration Units") or ""),\n            reference=str(record.get("Reference") or ""),\n            label_level="molecule_condition_assay",\n        ))\n    return rows\n\n\ndef parse_u16_qsardb(directory: Path) -> list[dict[str, Any]]:\n    namespace = {"q": "http://www.qsardb.org/QDB"}\n    registry = ET.parse(directory / "compounds" / "compounds.xml").getroot()\n    compounds: dict[int, dict[str, str]] = {}\n    for node in registry.findall("q:Compound", namespace):\n        cid = int(node.findtext("q:Id", namespaces=namespace))\n        inchi = node.findtext("q:InChI", default="", namespaces=namespace)\n        mol = Chem.MolFromInchi(inchi) if inchi else None\n        smiles = _standardize(Chem.MolToSmiles(mol, isomericSmiles=True)) if mol is not None else None\n        if smiles:\n            compounds[cid] = {\n                "smiles": smiles,\n                "name": node.findtext("q:Name", default="", namespaces=namespace),\n                "cas": node.findtext("q:Cas", default="", namespaces=namespace),\n            }\n    values = directory / "properties" / "3T3_NRU_Phototoxicity" / "values"\n    rows = []\n    with values.open(encoding="utf-8") as handle:\n        next(handle)\n        for line in handle:\n            fields = line.rstrip().split("\\t")\n            if len(fields) != 2 or int(fields[0]) not in compounds:\n                continue\n            cid, label = int(fields[0]), fields[1]\n            compound = compounds[cid]\n            rows.append(_row(\n                compound["smiles"], "phototoxicity", 1.0 if label == "P" else 0.0, "U16",\n                qsardb_compound_id=cid, cas=compound["cas"], compound_name=compound["name"],\n                assay="3T3_NRU", species="Mus musculus", label_level="molecule_assay",\n                original_split="validation" if cid >= 51 else "train",\n            ))\n    return rows\n\n\ndef _decimal(value: str) -> float | None:\n    return _finite(value.strip().replace(",", "."))\n\n\ndef _read_instrument_csv(path: Path) -> tuple[list[str], list[list[str]]]:\n    text = path.read_text(encoding="latin-1")\n    records = list(csv.reader(text.splitlines()))\n    for index, fields in enumerate(records):\n        first = fields[0].strip().lower() if fields else ""\n        if first in {"nm", "wavelength (nm)"}:\n            return fields, records[index + 1:]\n    return [], []\n\n\ndef parse_m05_full_spectra(directory: Path, pubchem_directory: Path) -> list[dict[str, Any]]:\n    mappings = {}\n    for cas, token in (("1498-88-0", "NO2_SP"), ("1498-89-1", "MeO_NO2_SP")):\n        metadata = json.loads((pubchem_directory / f"{cas}.json").read_text())\n        properties = metadata["PropertyTable"]["Properties"][0]\n        mappings[token] = (standardize_smiles(properties.get("SMILES") or properties["ConnectivitySMILES"]), cas)\n    rows = []\n    for path in sorted(directory.rglob("*.csv")):\n        header, records = _read_instrument_csv(path)\n        if not header or len(header) < 2:\n            continue\n        token = "MeO_NO2_SP" if "MeO_NO2_SP" in path.name or "MeO_NO2_SP" in str(path.parent) else "NO2_SP"\n        smiles, cas = mappings[token]\n        curves: list[list[tuple[float, float]]] = [[] for _ in header[1:]]\n        for record in records:\n            wavelength = _decimal(record[0]) if record else None\n            if wavelength is None:\n                continue\n            for index in range(min(len(curves), len(record) - 1)):\n                absorbance = _decimal(record[index + 1])\n                if absorbance is not None:\n                    curves[index].append((wavelength, max(0.0, absorbance)))\n        for curve_index in sorted({0, len(curves) - 1}):\n            curve = curves[curve_index]\n            wavelengths = [item[0] for item in curve if 290.0 <= item[0] <= 400.0]\n            absorbance = [item[1] for item in curve if 290.0 <= item[0] <= 400.0]\n            if len(wavelengths) < 20 or max(absorbance, default=0.0) <= 0:\n                continue\n            peak = wavelengths[max(range(len(absorbance)), key=absorbance.__getitem__)]\n            rows.append({\n                **_row(smiles, "full_spectrum", peak, "M05", family="spiropyran", cas=cas,\n                       condition_file=str(path.relative_to(directory)), curve_index=curve_index,\n                       label_level="molecule_state_condition_spectrum"),\n                "wavelength_min_nm": min(wavelengths), "wavelength_max_nm": max(wavelengths),\n                "uvb_auc": trapezoid_auc(wavelengths, absorbance, 290.0, 320.0),\n                "uva_auc": trapezoid_auc(wavelengths, absorbance, 320.0, 400.0),\n                "lambda_c_nm": critical_wavelength(wavelengths, absorbance),\n                "spectrum_json": json.dumps([[w, a] for w, a in zip(wavelengths, absorbance)]),\n            })\n    return rows\n\n\ndef parse_source_catalog(path: Path) -> list[dict[str, Any]]:\n    rows: list[dict[str, Any]] = []\n    workbook = pd.ExcelFile(path)\n    for sheet in workbook.sheet_names:\n        frame = pd.read_excel(path, sheet_name=sheet)\n        for index, record in enumerate(frame.to_dict("records"), start=2):\n            clean = {str(key): value for key, value in record.items() if not pd.isna(value)}\n            rows.append({"sheet": sheet, "excel_row": index, "record_json": json.dumps(clean, ensure_ascii=False, default=str)})\n    return rows\n\n\ndef _source_status(path: Path, source_id: str, role: str) -> dict[str, Any]:\n    return {\n        "source_id": source_id, "role": role, "path": str(path.resolve()), "available": path.exists(),\n        "bytes": path.stat().st_size if path.exists() else 0,\n        "sha256": sha256_file(path) if path.is_file() else None,\n    }\n\n\ndef prepare_real_data(config: dict[str, Any], output_dir: str | Path, root: str | Path,\n                      reaction_library: Iterable[dict[str, Any]]) -> dict[str, Any]:\n    destination = Path(output_dir)\n    destination.mkdir(parents=True, exist_ok=True)\n    root = Path(root)\n    inputs = {\n        "M13": root / "data/raw/M13_uvvisml/uvvisml/data/processed/all_lambda_max_abs_including_duplicates.csv",\n        "M01": root / "data/raw/M01_photoswitch/dataset/photoswitches.csv",\n        "M05": root / "data/raw/M05_zenodo/files/data",\n        "U09": root / "data/raw/U09_hppt/hppt_database_14feb2023.xlsx",\n        "U07": root / "data/raw/U07_nice/Skin_Irritation_Corrosion.xlsx",\n        "U12": root / "data/raw/U12_skinpix/20230620_cleanedDB.xlsx",\n        "U16": root / "data/raw/U16_qsardb/files",\n        "U13": root / "data/raw/U13_mendeley/Table_1.xlsx",\n        "catalog": root / "database_matrix_MOST_UV_skin.xlsx",\n    }\n    missing = [key for key, path in inputs.items() if not path.exists()]\n    if missing:\n        raise FileNotFoundError(f"Required real data sources are missing: {missing}")\n    rows = []\n    rows.extend(parse_m13(inputs["M13"], int(config.get("data", {}).get("m13_max_rows", 25_000))))\n    rows.extend(parse_m01(inputs["M01"]))\n    rows.extend(parse_u09_hppt(inputs["U09"]))\n    rows.extend(parse_u07_irritation(inputs["U07"], inputs["U09"]))\n    rows.extend(parse_u12_skinpix(inputs["U12"]))\n    rows.extend(parse_u13_permeation(inputs["U13"], inputs["U09"], inputs["U12"]))\n    rows.extend(parse_u16_qsardb(inputs["U16"]))\n    full_spectra = parse_m05_full_spectra(inputs["M05"], root / "data/raw/M05_zenodo/pubchem")\n    write_csv(destination / "reviewer_endpoints.csv", rows)\n    write_csv(destination / "full_spectra.csv", full_spectra)\n    write_csv(destination / "reaction_library.csv", reaction_library)\n    catalog = parse_source_catalog(inputs["catalog"])\n    write_csv(destination / "source_catalog_from_initial_excel.csv", catalog)\n    positives = [row for row in rows if row["endpoint"] == "phototoxicity" and float(row["target"]) == 1.0]\n    write_csv(destination / "known_phototoxic_registry.csv", positives)\n    unique_structures = {row["smiles"] for row in rows}\n    if len(unique_structures) > int(config["project"]["max_training_structures"]):\n        raise ValueError("Real endpoint corpus exceeds the configured unique-structure cap")\n    counts: dict[str, int] = defaultdict(int)\n    structures_by_endpoint: dict[str, set[str]] = defaultdict(set)\n    for row in rows:\n        counts[row["endpoint"]] += 1\n        structures_by_endpoint[row["endpoint"]].add(row["smiles"])\n    statuses = [\n        _source_status(root / "data/raw/M13_uvvisml/uvvisml/data/original/joung/DB_for_chromophore_Sci_Data_rev02.csv",\n                       "M11", "Deep4Chem lambda-max constituent of the M13 unified table"),\n        _source_status(root / "data/raw/M13_uvvisml/uvvisml/data/original/jcole/paper_allDB.csv",\n                       "M12", "CDEx lambda-max constituent of the M13 unified table"),\n        _source_status(inputs["M13"], "M13", "unified UV/VisML lambda-max transfer table"),\n        _source_status(inputs["M01"], "M01", "photoswitch lambda-max and thermal kinetics"),\n        _source_status(root / "data/raw/M05_zenodo/data.zip", "M05", "condition-rich full spiropyran spectra"),\n        _source_status(inputs["U09"], "U09", "human patch-test sensitization"),\n        _source_status(inputs["U07"], "U07", "NICE irritation/corrosion calls joined by exact DTXSID"),\n        _source_status(inputs["U12"], "U12", "skin permeation Kp"),\n        _source_status(inputs["U13"], "U13", "human epidermis Kp/Jmax; exact CAS join and unit conversion"),\n        _source_status(root / "data/raw/U16_qsardb/2011TIV324.qdb.zip", "U16", "3T3 NRU phototoxicity"),\n        _source_status(inputs["catalog"], "initial_excel", "source registry and routing only"),\n    ]\n    manifest = {\n        "schema_version": "2.0", "raw_inputs_mutated": False,\n        "training_mode": "real_endpoint_specific", "endpoint_rows": dict(sorted(counts.items())),\n        "unique_structures_by_endpoint": {key: len(value) for key, value in sorted(structures_by_endpoint.items())},\n        "unique_structures_total": len(unique_structures), "full_spectrum_records": len(full_spectra),\n        "reaction_library_rows": len(list(reaction_library)), "source_catalog_rows": len(catalog),\n        "sources": statuses,\n        "initial_excel_usage": "Used as a literature/source catalog and provenance input; it contains no molecular training labels.",\n        "unavailable_labels": {\n            "M03": "Metadata available, but Dryad file endpoint returned HTTP 401 without bearer token; empty partial file excluded.",\n            "energy_kj_mol": "No sufficiently identified open molecular training table is available; no synthetic replacement is made.",\n        },\n        "scientific_limitations": [\n            "Lambda-max observations are not full absorption spectra.",\n            "M05 full spectra cover two spiropyran structures and are calibration/validation evidence, not a broad training corpus.",\n            "M01 thermal kinetics are dominated by azo photoswitches and constitute class-shift evidence for MOST families.",\n            "Safety models are triage only and do not establish cosmetic safety.",\n        ],\n    }\n    (destination / "data_manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    return manifest\n'
load_embedded('mostgen.real_data', _source, PROJECT_ROOT / 'mostgen' / 'real_data.py')
print('loaded mostgen.real_data')


loaded mostgen.real_data


### 3.22 Endpoint ensembles, conformal uncertainty и AD — `real_reviewers.py`


In [11]:
_source = 'from __future__ import annotations\n\n"""Endpoint-specific reviewers trained only on parsed experimental records."""\n\nimport json\nimport math\nimport pickle\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nfrom sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, RandomForestClassifier, RandomForestRegressor\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, mean_absolute_error, root_mean_squared_error, roc_auc_score\n\nfrom .chemistry import max_similarity\nfrom .data import WAVELENGTHS, read_csv\nfrom .numerics import critical_wavelength, trapezoid_auc\nfrom .provenance import stable_hash\nfrom .reviewers import FAMILIES, Prediction, feature_vector\n\n\nREGRESSION_ENDPOINTS = ("lambda_max_nm", "log_half_life_h", "kp_log_cm_s")\nCLASSIFICATION_ENDPOINTS = ("phototoxicity", "skin_sensitization", "skin_irritation")\nALL_ENDPOINTS = REGRESSION_ENDPOINTS + CLASSIFICATION_ENDPOINTS\n\n\n@dataclass\nclass EndpointEnsemble:\n    endpoint: str\n    task: str\n    models: list[Any]\n    conformal_q90: float\n    reference_smiles: list[str]\n    metrics: dict[str, Any]\n\n    def members(self, x: np.ndarray) -> np.ndarray:\n        if self.task == "classification":\n            outputs = []\n            for model in self.models:\n                classes = list(model.classes_)\n                probabilities = model.predict_proba(x)\n                outputs.append(probabilities[:, classes.index(1.0)] if 1.0 in classes else np.zeros(len(x)))\n            return np.asarray(outputs, dtype=float)\n        return np.asarray([model.predict(x) for model in self.models], dtype=float)\n\n    def predict(self, x: np.ndarray, z: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\n        values = self.members(x)\n        mean = values.mean(axis=0)\n        ensemble_std = values.std(axis=0, ddof=0)\n        calibrated = np.maximum(ensemble_std, self.conformal_q90 / max(z, 1e-8))\n        return mean, calibrated, values\n\n\nclass RealReviewerBundle:\n    """Serializable bundle with independent endpoint models and provenance."""\n\n    data_mode = "real_endpoint_specific"\n\n    def __init__(self, purpose: str, bits: int, z: float, endpoints: dict[str, EndpointEnsemble],\n                 references: dict[str, Any], metrics: dict[str, Any]) -> None:\n        self.purpose = purpose\n        self.bits = bits\n        self.z = z\n        self.endpoints = endpoints\n        self.references = references\n        self.metrics = metrics\n\n    def save(self, path: str | Path) -> None:\n        destination = Path(path)\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        with destination.open("wb") as handle:\n            pickle.dump(self, handle, protocol=pickle.HIGHEST_PROTOCOL)\n\n    @classmethod\n    def load(cls, path: str | Path) -> "RealReviewerBundle":\n        with Path(path).open("rb") as handle:\n            model = pickle.load(handle)\n        if not isinstance(model, cls):\n            raise TypeError("Not a real endpoint-specific reviewer bundle")\n        return model\n\n    def predict(self, smiles: str, family: str) -> dict[str, Any]:\n        return self.predict_many([(smiles, family)])[0]\n\n    def predict_many(self, molecules: list[tuple[str, str]]) -> list[dict[str, Any]]:\n        if not molecules:\n            return []\n        x = np.vstack([feature_vector(smiles, family, self.bits) for smiles, family in molecules])\n        predictions = {name: model.predict(x, self.z) for name, model in self.endpoints.items()}\n        output = []\n        for index, (smiles, family) in enumerate(molecules):\n            lambda_mean, lambda_std, lambda_members = predictions["lambda_max_nm"]\n            member_curves = []\n            for peak in lambda_members[:, index]:\n                # This deliberately remains a declared band-shape proxy.  A\n                # lambda-max label cannot identify an experimental spectrum.\n                member_curves.append(np.asarray([\n                    math.exp(-0.5 * ((wavelength - float(peak)) / 34.0) ** 2)\n                    for wavelength in WAVELENGTHS\n                ]))\n            curves = np.vstack(member_curves)\n            spectrum = Prediction(\n                mean={f"abs_{w}": float(v) for w, v in zip(WAVELENGTHS, curves.mean(axis=0))},\n                std={f"abs_{w}": float(v) for w, v in zip(WAVELENGTHS, curves.std(axis=0))},\n            )\n\n            def pred(endpoint: str, output_name: str) -> Prediction:\n                mean, spread, _ = predictions[endpoint]\n                return Prediction({output_name: float(mean[index])}, {output_name: float(spread[index])})\n\n            half = pred("log_half_life_h", "log_half_life_h")\n            kp = pred("kp_log_cm_s", "kp_log_cm_s")\n            photo = pred("phototoxicity", "phototoxicity_probability")\n            sensitization = pred("skin_sensitization", "sensitization_probability")\n            irritation = pred("skin_irritation", "irritation_probability")\n            safety = Prediction(\n                mean={**kp.mean, **photo.mean, **sensitization.mean, **irritation.mean},\n                std={**kp.std, **photo.std, **sensitization.std, **irritation.std},\n            )\n            most = Prediction(\n                mean={"energy_kj_mol": math.nan, "specific_energy_wh_kg": math.nan, **half.mean},\n                std={"energy_kj_mol": math.nan, "specific_energy_wh_kg": math.nan, **half.std},\n            )\n            endpoint_ad = {\n                name: max_similarity(smiles, model.reference_smiles, self.bits)\n                for name, model in self.endpoints.items()\n            }\n            output.append({\n                "spectrum": spectrum, "most": most, "safety": safety,\n                "ad_spectral_similarity": endpoint_ad["lambda_max_nm"],\n                # Energy is the defining MOST endpoint; without molecular\n                # energy labels it is out of domain until the physical oracle.\n                "ad_most_similarity": 0.0,\n                "endpoint_ad": endpoint_ad,\n                "family_reference_medians": self.references["family_medians"][family],\n                "reviewer_purpose": self.purpose,\n                "data_mode": self.data_mode,\n                "evidence": {\n                    "spectrum": "lambda_max_gaussian_band_proxy",\n                    "half_life": "M01_class_shift_model",\n                    "energy": "missing_requires_gfn2_xtb_or_experiment",\n                    "kp": "U12_SkinPiX_plus_U13_human_epidermis",\n                    "phototoxicity": "U16_3T3_NRU",\n                    "sensitization": "U09_human_patch_test",\n                    "irritation": "U07_NICE_exact_DTXSID_join",\n                },\n                "lambda_max_nm": float(lambda_mean[index]),\n                "lambda_max_uncertainty": float(lambda_std[index]),\n            })\n        return output\n\n\ndef _matrix(rows: list[dict[str, str]], bits: int) -> np.ndarray:\n    return np.vstack([feature_vector(row["smiles"], row.get("family", "unassigned"), bits) for row in rows])\n\n\ndef _deterministic_refs(rows: list[dict[str, str]], limit: int = 768) -> list[str]:\n    unique = sorted({row["smiles"] for row in rows}, key=lambda smiles: stable_hash(smiles, 32))\n    return unique[:limit]\n\n\ndef _new_model(purpose: str, task: str, seed: int, trees: int):\n    common = dict(n_estimators=trees, random_state=seed, n_jobs=1, min_samples_leaf=2, max_features=0.65)\n    if task == "classification":\n        cls = RandomForestClassifier if purpose == "reward" else ExtraTreesClassifier\n        return cls(class_weight="balanced", bootstrap=purpose == "reward", **common)\n    cls = RandomForestRegressor if purpose == "reward" else ExtraTreesRegressor\n    return cls(bootstrap=purpose == "reward", **common)\n\n\ndef _fit_endpoint(rows: list[dict[str, str]], endpoint: str, purpose: str, bits: int,\n                  members: int, trees: int, seed: int, z: float) -> EndpointEnsemble:\n    endpoint_rows = [row for row in rows if row["endpoint"] == endpoint]\n    task = "classification" if endpoint in CLASSIFICATION_ENDPOINTS else "regression"\n    train = [row for row in endpoint_rows if row["split"] == "train"]\n    calibration = [row for row in endpoint_rows if row["split"] == "validation"]\n    test = [row for row in endpoint_rows if row["split"] == "test"]\n    if task == "classification" and len({float(row["target"]) for row in train}) < 2:\n        # Deterministic group split can be class-imbalanced for a 53-molecule\n        # source. Move complete scaffolds, never individual duplicate rows.\n        train = [row for row in endpoint_rows if row["split"] != "test"]\n        calibration = test\n    if len(train) < 8:\n        raise ValueError(f"Insufficient real training records for {endpoint}: {len(train)}")\n    x_train = _matrix(train, bits)\n    y_train = np.asarray([float(row["target"]) for row in train])\n    models = []\n    for index in range(members):\n        model = _new_model(purpose, task, seed + index * 1009, trees)\n        model.fit(x_train, y_train)\n        models.append(model)\n    provisional = EndpointEnsemble(endpoint, task, models, 0.0, _deterministic_refs(train), {})\n    calibration_residuals = []\n    if calibration:\n        y_cal = np.asarray([float(row["target"]) for row in calibration])\n        member_values = provisional.members(_matrix(calibration, bits))\n        calibration_residuals = np.abs(y_cal - member_values.mean(axis=0)).tolist()\n    q90 = float(np.quantile(calibration_residuals, 0.90, method="higher")) if calibration_residuals else (0.25 if task == "classification" else 1.0)\n    metrics: dict[str, Any] = {\n        "n_total": len(endpoint_rows), "n_train": len(train), "n_calibration": len(calibration), "n_test": len(test),\n        "unique_structures": len({row["smiles"] for row in endpoint_rows}), "conformal_q90": q90,\n    }\n    if test:\n        y_test = np.asarray([float(row["target"]) for row in test])\n        pred = provisional.members(_matrix(test, bits)).mean(axis=0)\n        if task == "classification":\n            labels = (pred >= 0.5).astype(float)\n            metrics.update({"accuracy": float(accuracy_score(y_test, labels)),\n                            "balanced_accuracy": float(balanced_accuracy_score(y_test, labels))})\n            if len(set(y_test)) == 2:\n                metrics["roc_auc"] = float(roc_auc_score(y_test, pred))\n        else:\n            metrics.update({"mae": float(mean_absolute_error(y_test, pred)),\n                            "rmse": float(root_mean_squared_error(y_test, pred)),\n                            "interval_90_coverage": float(np.mean(np.abs(y_test - pred) <= q90))})\n    provisional.conformal_q90 = q90\n    provisional.metrics = metrics\n    return provisional\n\n\ndef _family_medians(rows: list[dict[str, str]]) -> dict[str, dict[str, float]]:\n    peaks = [float(row["target"]) for row in rows if row["endpoint"] == "lambda_max_nm"]\n    peak = float(np.median(peaks))\n    curve = [math.exp(-0.5 * ((wavelength - peak) / 34.0) ** 2) for wavelength in WAVELENGTHS]\n    reference = {\n        "uvb_auc": trapezoid_auc(WAVELENGTHS, curve, 290.0, 320.0),\n        "uva_auc": trapezoid_auc(WAVELENGTHS, curve, 320.0, 400.0),\n        "lambda_c_nm": critical_wavelength(WAVELENGTHS, curve),\n        "energy_kj_mol": math.nan, "specific_energy_wh_kg": math.nan,\n        "log_half_life_h": math.log10(8.0),\n    }\n    return {family: dict(reference) for family in FAMILIES}\n\n\ndef train_real_reviewers(config: dict[str, Any], data_path: str | Path, output_dir: str | Path) -> dict[str, Any]:\n    rows = read_csv(data_path)\n    endpoints_present = {row.get("endpoint") for row in rows}\n    missing = set(ALL_ENDPOINTS) - endpoints_present\n    if missing:\n        raise ValueError(f"Missing required real endpoints: {sorted(missing)}")\n    unique = {row["smiles"] for row in rows}\n    if len(unique) > int(config["project"]["max_training_structures"]):\n        raise ValueError("Reviewer training structures exceed configured cap")\n    bits = int(config["reviewers"]["fingerprint_bits"])\n    members = int(config["reviewers"]["ensemble_size"])\n    trees = int(config["reviewers"].get("trees_per_member", 28))\n    z = float(config["reviewers"]["confidence_z"])\n    destination = Path(output_dir)\n    cards = {}\n    for purpose, shift in (("reward", 0), ("evaluator", 500_009)):\n        endpoint_models = {\n            endpoint: _fit_endpoint(rows, endpoint, purpose, bits, members, trees,\n                                    int(config["project"]["default_seed"]) + shift + 97 * idx, z)\n            for idx, endpoint in enumerate(ALL_ENDPOINTS)\n        }\n        references = {\n            "family_medians": _family_medians(rows),\n            "training_data_hash": stable_hash("\\n".join(sorted(row["record_id"] for row in rows)), 32),\n            "energy_model_status": "unavailable_no_open_molecular_labels",\n        }\n        metrics = {name: model.metrics for name, model in endpoint_models.items()}\n        bundle = RealReviewerBundle(purpose, bits, z, endpoint_models, references, metrics)\n        path = destination / purpose / "reviewers.pkl"\n        bundle.save(path)\n        cards[purpose] = {"path": str(path.resolve()), "algorithm": ("RandomForest" if purpose == "reward" else "ExtraTrees") + " endpoint ensembles",\n                          "metrics": metrics}\n    metadata = {\n        "schema_version": "2.0", "rows": len(rows), "unique_structures": len(unique),\n        "evidence_tiers": ["experimental"], "data_mode": "real_endpoint_specific",\n        **cards, "independent_instances": True,\n        "uncertainty": "90% split-conformal absolute-residual radius, lower-bounded by ensemble spread",\n        "energy_model_status": "not_trained; physical GFN2-xTB oracle required",\n        "limitations": [\n            "The spectral reward is a declared Gaussian band proxy around predicted lambda-max, not a measured spectrum.",\n            "M01 kinetics are class-shift evidence and are outside-domain for most generated MOST scaffolds.",\n            "U16 is small; uncertain phototoxicity predictions fail closed.",\n        ],\n    }\n    destination.mkdir(parents=True, exist_ok=True)\n    (destination / "model_cards.json").write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    return metadata\n'
load_embedded('mostgen.real_reviewers', _source, PROJECT_ROOT / 'mostgen' / 'real_reviewers.py')
print('loaded mostgen.real_reviewers')


loaded mostgen.real_reviewers


### 3.24 Прозрачные компоненты reward и hard gates — `scoring.py`


In [12]:
_source = 'from __future__ import annotations\n\nimport json\nimport math\nfrom dataclasses import dataclass\nfrom typing import Any, Sequence\n\nfrom .chemistry import ChemistryError, descriptors, safety_veto, synthetic_accessibility\nfrom .data import WAVELENGTHS\nfrom .numerics import (\n    beer_lambert_transmittance,\n    critical_wavelength,\n    interval_score,\n    lower_confidence_bound,\n    sigmoid,\n    trapezoid_auc,\n    upper_confidence_bound,\n    weighted_geometric_mean,\n)\nfrom .reviewers import ReviewerBundle\n\n\ndef _auc_uncertainty(wavelengths: Sequence[float], std: Sequence[float], low: float, high: float) -> float:\n    # Independent-bin propagation is a conservative-enough transparent proxy\n    # for the smoke ensemble. Production models should retain member curves.\n    selected = [(w, s) for w, s in zip(wavelengths, std) if low <= w <= high]\n    if len(selected) < 2:\n        return 0.0\n    variance = 0.0\n    for (x1, s1), (x2, s2) in zip(selected, selected[1:]):\n        dx = x2 - x1\n        variance += (0.5 * dx) ** 2 * (s1**2 + s2**2)\n    return math.sqrt(variance)\n\n\ndef _lambda_uncertainty(wavelengths: Sequence[float], mean: Sequence[float], std: Sequence[float]) -> float:\n    centre = critical_wavelength(wavelengths, mean)\n    lower_curve = [max(0.0, value - spread) for value, spread in zip(mean, std)]\n    upper_curve = [max(0.0, value + spread) for value, spread in zip(mean, std)]\n    return max(abs(centre - critical_wavelength(wavelengths, lower_curve)), abs(centre - critical_wavelength(wavelengths, upper_curve)))\n\n\n@dataclass\nclass ScoringContext:\n    config: dict[str, Any]\n    reviewers: ReviewerBundle\n\n    def review_candidate(\n        self, candidate: dict[str, Any], method_id: str, seed: int,\n        raw_prediction: dict[str, Any] | None = None,\n    ) -> dict[str, Any]:\n        smiles = candidate["smiles"]\n        charged = candidate["charged_smiles"]\n        family = candidate["family"]\n        veto = safety_veto(smiles, self.config)\n        base: dict[str, Any] = {\n            **candidate,\n            "method_id": method_id,\n            "seed": seed,\n            "valid": not veto.veto,\n            "psoralen_alert": veto.psoralen_alert,\n            "known_phototoxic_match": veto.known_phototoxic_match,\n            "psoralen_similarity": veto.psoralen_similarity,\n            "psoralen_similarity_warning": veto.psoralen_similarity_warning,\n            "reactive_alerts": ";".join(veto.reactive_alerts),\n            "unsupported_elements": ";".join(veto.unsupported_elements),\n            "safety_veto": veto.veto,\n            "not_iso_certified": True,\n            "selected": False,\n            "reviewer_purpose": self.reviewers.purpose,\n            "reviewer_data_mode": getattr(self.reviewers, "data_mode", "synthetic_smoke_only"),\n        }\n        if veto.veto:\n            base.update({\n                "reward": 0.0,\n                "reward_pre_diversity": 0.0,\n                "joint_uv_pass": False,\n                "most_pass": False,\n                "safety_pass": False,\n                "joint_pass": False,\n                "phototoxicity_uncertain": True,\n                "ad_spectral": False,\n                "ad_most": False,\n                "failure_reasons": ";".join(veto.reasons),\n                "spectrum_json": "[]",\n            })\n            return base\n        try:\n            raw = raw_prediction if raw_prediction is not None else self.reviewers.predict(smiles, family)\n        except (ChemistryError, ValueError) as exc:\n            base.update({\n                "valid": False, "reward": 0.0, "joint_uv_pass": False,\n                "most_pass": False, "safety_pass": False, "joint_pass": False,\n                "phototoxicity_uncertain": True, "ad_spectral": False,\n                "ad_most": False, "failure_reasons": f"reviewer_error:{exc}",\n                "spectrum_json": "[]",\n            })\n            return base\n\n        spectrum_mean = [max(0.0, raw["spectrum"].mean[f"abs_{w}"]) for w in WAVELENGTHS]\n        spectrum_std = [max(0.0, raw["spectrum"].std[f"abs_{w}"]) for w in WAVELENGTHS]\n        uvb = trapezoid_auc(WAVELENGTHS, spectrum_mean, 290.0, 320.0)\n        uva = trapezoid_auc(WAVELENGTHS, spectrum_mean, 320.0, 400.0)\n        uvb_std = _auc_uncertainty(WAVELENGTHS, spectrum_std, 290.0, 320.0)\n        uva_std = _auc_uncertainty(WAVELENGTHS, spectrum_std, 320.0, 400.0)\n        lambda_c = critical_wavelength(WAVELENGTHS, spectrum_mean)\n        lambda_std = _lambda_uncertainty(WAVELENGTHS, spectrum_mean, spectrum_std)\n        z = float(self.config["reviewers"]["confidence_z"])\n        uvb_lcb = lower_confidence_bound(uvb, uvb_std, z)\n        uva_lcb = lower_confidence_bound(uva, uva_std, z)\n        lambda_lcb = lower_confidence_bound(lambda_c, lambda_std, z)\n        loading = float(self.config["reviewers"].get("beer_lambert_loading_scale", 1.0))\n        uvb_indices = [index for index, wavelength in enumerate(WAVELENGTHS) if 290 <= wavelength <= 320]\n        uva_indices = [index for index, wavelength in enumerate(WAVELENGTHS) if 320 <= wavelength <= 400]\n        uvb_transmittance = beer_lambert_transmittance([spectrum_mean[index] for index in uvb_indices], loading)\n        uva_transmittance = beer_lambert_transmittance([spectrum_mean[index] for index in uva_indices], loading)\n        # Lower absorbance is the conservative side of the uncertainty band\n        # because it produces the upper (worst) transmittance proxy.\n        uvb_transmittance_ucb = beer_lambert_transmittance(\n            [max(0.0, spectrum_mean[index] - z * spectrum_std[index]) for index in uvb_indices], loading,\n        )\n        uva_transmittance_ucb = beer_lambert_transmittance(\n            [max(0.0, spectrum_mean[index] - z * spectrum_std[index]) for index in uva_indices], loading,\n        )\n        film_proxy_pass = (\n            uvb_transmittance_ucb <= float(self.config["reviewers"].get("uvb_transmittance_max", 1.0))\n            and uva_transmittance_ucb <= float(self.config["reviewers"].get("uva_transmittance_max", 1.0))\n        )\n        medians = raw["family_reference_medians"]\n        joint_uv = (\n            uvb_lcb >= medians["uvb_auc"]\n            and uva_lcb >= medians["uva_auc"]\n            and lambda_lcb >= float(self.config["reviewers"]["lambda_c_min_nm"])\n            and film_proxy_pass\n        )\n\n        most_mean, most_std = raw["most"].mean, raw["most"].std\n        energy = most_mean["energy_kj_mol"]\n        energy_std = most_std["energy_kj_mol"]\n        energy_available = math.isfinite(energy) and math.isfinite(energy_std)\n        energy_lcb = lower_confidence_bound(energy, energy_std, z) if energy_available else math.nan\n        specific = most_mean["specific_energy_wh_kg"]\n        specific_std = most_std["specific_energy_wh_kg"]\n        specific_available = math.isfinite(specific) and math.isfinite(specific_std)\n        specific_lcb = lower_confidence_bound(specific, specific_std, z) if specific_available else math.nan\n        half_log = most_mean["log_half_life_h"]\n        half_log_std = most_std["log_half_life_h"]\n        half_available = math.isfinite(half_log) and math.isfinite(half_log_std)\n        half_hours = 10.0 ** half_log if half_available else math.nan\n        half_low = 10.0 ** lower_confidence_bound(half_log, half_log_std, z) if half_available else math.nan\n        half_high = 10.0 ** upper_confidence_bound(half_log, half_log_std, z) if half_available else math.nan\n        half_window = self.config["reviewers"]["half_life_window_hours"]\n        most_pass = (\n            energy_available\n            and specific_available\n            and half_available\n            and energy_lcb > 0.0\n            and energy_lcb >= medians["energy_kj_mol"]\n            and specific_lcb >= float(self.config["reviewers"].get("specific_energy_min_wh_kg", 0.0))\n            and half_window[0] <= half_hours <= half_window[1]\n        )\n\n        safety_mean, safety_std = raw["safety"].mean, raw["safety"].std\n        photo = min(1.0, max(0.0, safety_mean["phototoxicity_probability"]))\n        photo_std = safety_std["phototoxicity_probability"]\n        photo_upper = min(1.0, upper_confidence_bound(photo, photo_std, z))\n        uncertain_band = self.config["reviewers"]["uncertain_probability_band"]\n        photo_uncertain = (\n            uncertain_band[0] <= photo <= uncertain_band[1]\n            or (photo - z * photo_std) <= uncertain_band[1] <= photo_upper\n        )\n        kp = safety_mean["kp_log_cm_s"]\n        kp_std = safety_std["kp_log_cm_s"]\n        kp_upper = upper_confidence_bound(kp, kp_std, z)\n        sensitization = safety_mean.get("sensitization_probability", math.nan)\n        sensitization_std = safety_std.get("sensitization_probability", math.nan)\n        sensitization_upper = (\n            min(1.0, upper_confidence_bound(sensitization, sensitization_std, z))\n            if math.isfinite(sensitization) and math.isfinite(sensitization_std) else math.nan\n        )\n        irritation = safety_mean.get("irritation_probability", math.nan)\n        irritation_std = safety_std.get("irritation_probability", math.nan)\n        irritation_upper = (\n            min(1.0, upper_confidence_bound(irritation, irritation_std, z))\n            if math.isfinite(irritation) and math.isfinite(irritation_std) else math.nan\n        )\n        sa = synthetic_accessibility(smiles)\n        ad_threshold = float(self.config["reviewers"]["ad_similarity_threshold"])\n        ad_spectral = raw["ad_spectral_similarity"] >= ad_threshold\n        ad_most = raw["ad_most_similarity"] >= ad_threshold\n        safety_pass = (\n            not photo_uncertain\n            and photo_upper <= float(self.config["reviewers"]["phototoxicity_max_probability"])\n            and kp_upper <= float(self.config["reviewers"]["kp_max_log_cm_s"])\n            and (not math.isfinite(sensitization_upper) or sensitization_upper <= float(self.config["reviewers"].get("sensitization_max_probability", 0.50)))\n            and (not math.isfinite(irritation_upper) or irritation_upper <= float(self.config["reviewers"].get("irritation_max_probability", 0.50)))\n            and sa <= 5.0\n        )\n\n        floor = float(self.config["reward"]["floor"])\n        components = {\n            "uvb": sigmoid(uvb_lcb, medians["uvb_auc"], max(0.5, medians["uvb_auc"] * 0.12)),\n            "uva": sigmoid(uva_lcb, medians["uva_auc"], max(0.5, medians["uva_auc"] * 0.12)),\n            "lambda_c": sigmoid(lambda_lcb, float(self.config["reviewers"]["lambda_c_min_nm"]), 5.0),\n            "film_uvb": sigmoid(float(self.config["reviewers"].get("uvb_transmittance_max", 1.0)) - uvb_transmittance_ucb, 0.0, 0.08),\n            "film_uva": sigmoid(float(self.config["reviewers"].get("uva_transmittance_max", 1.0)) - uva_transmittance_ucb, 0.0, 0.08),\n            # Keep curriculum rewards dense without inventing an energy\n            # prediction. Missing energy receives an explicit low prior and\n            # can never pass the hard MOST gate.\n            "energy": sigmoid(energy_lcb, medians["energy_kj_mol"], max(2.0, medians["energy_kj_mol"] * 0.10)) if energy_available and math.isfinite(medians["energy_kj_mol"]) else 0.20,\n            "specific_energy": sigmoid(specific_lcb, float(self.config["reviewers"].get("specific_energy_min_wh_kg", 0.0)), 15.0) if specific_available else 0.20,\n            "half_life": interval_score(half_log, math.log10(half_window[0]), math.log10(half_window[1]), 0.13) if half_available else 0.20,\n            "phototoxicity": sigmoid(float(self.config["reviewers"]["phototoxicity_max_probability"]) - photo_upper, 0.0, 0.07),\n            "permeation": sigmoid(float(self.config["reviewers"]["kp_max_log_cm_s"]) - kp_upper, 0.0, 0.25),\n            "sensitization": sigmoid(float(self.config["reviewers"].get("sensitization_max_probability", 0.50)) - sensitization_upper, 0.0, 0.07) if math.isfinite(sensitization_upper) else 0.20,\n            "irritation": sigmoid(float(self.config["reviewers"].get("irritation_max_probability", 0.50)) - irritation_upper, 0.0, 0.07) if math.isfinite(irritation_upper) else 0.20,\n            "sa": sigmoid(5.0 - sa, 0.0, 0.75),\n            "ad": min(1.0, min(raw["ad_spectral_similarity"], raw["ad_most_similarity"]) / ad_threshold),\n        }\n        weights = self.config["reward"]["weights"]\n        reward = weighted_geometric_mean(components, weights, floor)\n        stage_rewards = {\n            "chemistry": 1.0,\n            "spectrum": weighted_geometric_mean({k: components[k] for k in ("uvb", "uva", "lambda_c", "film_uvb", "film_uva")}, weights, floor),\n            "most": weighted_geometric_mean({k: components[k] for k in ("energy", "specific_energy", "half_life")}, weights, floor),\n            "safety": weighted_geometric_mean({k: components[k] for k in ("phototoxicity", "permeation", "sensitization", "irritation", "sa", "ad")}, weights, floor),\n        }\n        failures = []\n        if not joint_uv:\n            failures.append("uv_joint_lcb_or_lambda")\n        if not film_proxy_pass:\n            failures.append("beer_lambert_film_proxy")\n        if not most_pass:\n            failures.append("most_energy_or_half_life")\n        if not energy_available:\n            failures.append("energy_model_unavailable_requires_physical_oracle")\n        if not ad_spectral:\n            failures.append("spectral_out_of_domain")\n        if not ad_most:\n            failures.append("most_out_of_domain")\n        if photo_uncertain:\n            failures.append("phototoxicity_uncertain")\n        elif photo_upper > float(self.config["reviewers"]["phototoxicity_max_probability"]):\n            failures.append("phototoxicity_risk")\n        if kp_upper > float(self.config["reviewers"]["kp_max_log_cm_s"]):\n            failures.append("permeation_risk")\n        if sa > 5.0:\n            failures.append("sa_proxy")\n        if math.isfinite(sensitization_upper) and sensitization_upper > float(self.config["reviewers"].get("sensitization_max_probability", 0.50)):\n            failures.append("sensitization_risk")\n        if math.isfinite(irritation_upper) and irritation_upper > float(self.config["reviewers"].get("irritation_max_probability", 0.50)):\n            failures.append("irritation_risk")\n        joint_pass = bool(joint_uv and most_pass and safety_pass and ad_spectral and ad_most)\n        d = descriptors(smiles)\n        base.update({\n            "reference_uvb_median": medians["uvb_auc"], "reference_uva_median": medians["uva_auc"],\n            "reference_energy_median": medians["energy_kj_mol"],\n            "uvb_auc": uvb, "uvb_auc_uncertainty": uvb_std, "uvb_auc_lcb": uvb_lcb,\n            "uva_auc": uva, "uva_auc_uncertainty": uva_std, "uva_auc_lcb": uva_lcb,\n            "lambda_c_nm": lambda_c, "lambda_c_uncertainty": lambda_std, "lambda_c_lcb": lambda_lcb,\n            "uvb_transmittance": uvb_transmittance, "uvb_transmittance_ucb": uvb_transmittance_ucb,\n            "uva_transmittance": uva_transmittance, "uva_transmittance_ucb": uva_transmittance_ucb,\n            "film_proxy_pass": film_proxy_pass, "beer_lambert_loading_scale": loading,\n            "energy_kj_mol": energy, "energy_uncertainty": energy_std, "energy_lcb": energy_lcb,\n            "specific_energy_wh_kg": specific, "specific_energy_uncertainty": specific_std,\n            "specific_energy_lcb": specific_lcb,\n            "log_half_life_h": half_log, "half_life_uncertainty_log": half_log_std,\n            "half_life_h": half_hours, "half_life_lcb_h": half_low, "half_life_ucb_h": half_high,\n            "kp_log_cm_s": kp, "kp_uncertainty": kp_std, "kp_ucb": kp_upper,\n            "phototoxicity_probability": photo, "phototoxicity_uncertainty": photo_std,\n            "phototoxicity_ucb": photo_upper, "phototoxicity_uncertain": photo_uncertain,\n            "sensitization_probability": sensitization, "sensitization_uncertainty": sensitization_std,\n            "sensitization_ucb": sensitization_upper,\n            "irritation_probability": irritation, "irritation_uncertainty": irritation_std,\n            "irritation_ucb": irritation_upper,\n            "similarity_D_A": raw["ad_spectral_similarity"], "similarity_D_B": raw["ad_most_similarity"],\n            "ad_spectral": ad_spectral, "ad_most": ad_most,\n            "sa_score": sa, "mol_wt": d["mol_wt"], "logp": d["logp"], "tpsa": d["tpsa"],\n            "joint_uv_pass": joint_uv, "most_pass": most_pass, "safety_pass": safety_pass,\n            "joint_pass": joint_pass, "reward": reward, "reward_pre_diversity": reward,\n            "energy_model_available": energy_available,\n            "evidence_complete": bool(energy_available and specific_available and half_available),\n            "spectrum_evidence": raw.get("evidence", {}).get("spectrum", "synthetic_smoke_curve"),\n            "evidence_json": json.dumps(raw.get("evidence", {}), sort_keys=True),\n            "reward_components_json": json.dumps(components, sort_keys=True),\n            "stage_rewards_json": json.dumps(stage_rewards, sort_keys=True),\n            "failure_reasons": ";".join(failures),\n            "spectrum_json": json.dumps([{"wavelength_nm": w, "absorbance": round(a, 6), "uncertainty": round(s, 6)} for w, a, s in zip(WAVELENGTHS, spectrum_mean, spectrum_std)]),\n        })\n        return base\n\n    def review_candidates(self, candidates: list[dict[str, Any]], method_id: str, seed: int) -> list[dict[str, Any]]:\n        safe_indices = []\n        safe_molecules = []\n        for index, candidate in enumerate(candidates):\n            if not safety_veto(candidate["smiles"], self.config).veto:\n                safe_indices.append(index)\n                safe_molecules.append((candidate["smiles"], candidate["family"]))\n        predictions = self.reviewers.predict_many(safe_molecules)\n        prediction_by_index = dict(zip(safe_indices, predictions))\n        return [\n            self.review_candidate(candidate, method_id, seed, prediction_by_index.get(index))\n            for index, candidate in enumerate(candidates)\n        ]\n'
load_embedded('mostgen.scoring', _source, PROJECT_ROOT / 'mostgen' / 'scoring.py')
print('loaded mostgen.scoring')


loaded mostgen.scoring


### 3.26 Baselines и настоящий REINVENT4/LibInvent — `search.py`


In [13]:
_source = 'from __future__ import annotations\n\nimport json\nimport math\nimport random\nimport subprocess\nimport sys\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nfrom typing import Any\n\nfrom rdkit import DataStructs\n\nfrom .chemistry import fingerprint, parse_smiles\nfrom .data import read_csv, write_csv\nfrom .provenance import stable_hash\nfrom .reviewers import ReviewerBundle\nfrom .scoring import ScoringContext\n\n\nMETHODS = ("prior_random", "weighted_retraining", "libinvent_rl")\n\n\ndef _update_budget_ledger(directory: str | Path, entries: dict[str, dict[str, Any]]) -> Path:\n    """Atomically merge measured reward-evaluation counts for method/seed runs."""\n    path = Path(directory) / "search_budget_ledger.json"\n    payload: dict[str, Any] = {"schema_version": "1.0", "runs": {}}\n    if path.exists():\n        payload = json.loads(path.read_text(encoding="utf-8"))\n    payload.setdefault("runs", {}).update(entries)\n    temporary = path.with_suffix(".json.tmp")\n    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n    return path\n\n\ndef _family_quotas(total: int, families: list[str]) -> dict[str, int]:\n    base, remainder = divmod(total, len(families))\n    return {family: base + (1 if index < remainder else 0) for index, family in enumerate(families)}\n\n\ndef _weighted_choice(rng: random.Random, rows: list[dict[str, Any]], weights: list[float]) -> dict[str, Any]:\n    total = sum(weights)\n    if total <= 0.0:\n        return rng.choice(rows)\n    point = rng.random() * total\n    cumulative = 0.0\n    for row, weight in zip(rows, weights):\n        cumulative += weight\n        if cumulative >= point:\n            return row\n    return rows[-1]\n\n\ndef _diversity_clusters(rows: list[dict[str, Any]], bits: int, threshold: float = 0.58) -> None:\n    centroids = []\n    counts: Counter[int] = Counter()\n    for row in rows:\n        fp = fingerprint(row["smiles"], bits)\n        cluster = None\n        if centroids:\n            similarities = DataStructs.BulkTanimotoSimilarity(fp, centroids)\n            best = max(range(len(similarities)), key=similarities.__getitem__)\n            if similarities[best] >= threshold:\n                cluster = best\n        if cluster is None:\n            cluster = len(centroids)\n            centroids.append(fp)\n        prior_count = counts[cluster]\n        counts[cluster] += 1\n        row["ecfp_cluster"] = cluster\n        row["cluster_prior_count"] = prior_count\n\n\ndef _apply_diversity_penalty(rows: list[dict[str, Any]], config: dict[str, Any]) -> None:\n    _diversity_clusters(rows, int(config["reviewers"]["fingerprint_bits"]))\n    penalty = float(config["reward"]["diversity_penalty"])\n    for row in rows:\n        base = float(row.get("reward_pre_diversity", 0.0))\n        row["reward"] = base / (1.0 + penalty * int(row["cluster_prior_count"]))\n\n\ndef _prior_candidates(rng: random.Random, pool: list[dict[str, Any]], count: int) -> list[dict[str, Any]]:\n    return rng.sample(pool, min(count, len(pool)))\n\n\ndef _adaptive_candidates(\n    method: str,\n    rng: random.Random,\n    pool: list[dict[str, Any]],\n    count: int,\n    context: ScoringContext,\n    seed: int,\n) -> list[dict[str, Any]]:\n    remaining = list(pool)\n    synthon_value: dict[str, float] = defaultdict(lambda: 0.5)\n    synthon_visits: Counter[str] = Counter()\n    selected: list[dict[str, Any]] = []\n    warmup = min(max(8, count // 8), count)\n    stage_names = [entry["name"] for entry in context.config["reward"]["stages"]]\n    batch_size = min(24, max(4, count // 20))\n    while remaining and len(selected) < count:\n        pending = []\n        pending_stages = []\n        for _ in range(min(batch_size, count - len(selected), len(remaining))):\n            progress = (len(selected) + len(pending)) / max(1, count)\n            stage = stage_names[min(len(stage_names) - 1, int(progress * len(stage_names)))]\n            if len(selected) + len(pending) < warmup:\n                candidate = rng.choice(remaining)\n            else:\n                weights = []\n                for row in remaining:\n                    q = 0.5 * (synthon_value[row["synthon_a"]] + synthon_value[row["synthon_b"]])\n                    if method == "weighted_retraining":\n                        weight = 0.05 + q**3\n                    else:\n                        visits = 1 + synthon_visits[row["synthon_a"]] + synthon_visits[row["synthon_b"]]\n                        bonus = math.sqrt(math.log(2 + len(selected) + len(pending)) / visits)\n                        weight = 0.03 + math.exp(min(4.0, 2.2 * q + 0.35 * bonus))\n                    weights.append(weight)\n                candidate = _weighted_choice(rng, remaining, weights)\n            remaining.remove(candidate)\n            pending.append(candidate)\n            pending_stages.append(stage)\n        reviewed_batch = context.review_candidates(pending, method, seed)\n        for reviewed, stage, candidate in zip(reviewed_batch, pending_stages, pending):\n            reviewed["curriculum_stage"] = stage\n            stage_rewards = json.loads(reviewed.get("stage_rewards_json", "{}") or "{}")\n            signal = float(stage_rewards.get(stage, reviewed.get("reward_pre_diversity", 0.0)))\n            for synthon in (candidate["synthon_a"], candidate["synthon_b"]):\n                visits = synthon_visits[synthon]\n                synthon_value[synthon] = (synthon_value[synthon] * visits + signal) / (visits + 1)\n                synthon_visits[synthon] += 1\n            selected.append(reviewed)\n    return selected\n\n\ndef run_method(\n    config: dict[str, Any],\n    library_path: str | Path,\n    reviewer_path: str | Path,\n    method: str,\n    seed: int,\n    budget: int | None = None,\n) -> list[dict[str, Any]]:\n    if method not in METHODS:\n        raise ValueError(f"Unknown search method: {method}")\n    pool = read_csv(library_path)\n    reviewers = ReviewerBundle.load(reviewer_path)\n    context = ScoringContext(config, reviewers)\n    calls = int(budget or config["execution"]["reviewer_budget_per_run"])\n    if calls < int(config["execution"]["n_per_run"]):\n        raise ValueError("Reviewer call budget cannot produce the requested unique count")\n    families = sorted(config["families"])\n    quotas = _family_quotas(calls, families)\n    rng = random.Random(seed ^ int(stable_hash(method, 8), 16))\n    results: list[dict[str, Any]] = []\n    for family in families:\n        members = [row for row in pool if row["family"] == family]\n        quota = quotas[family]\n        if len(members) < quota:\n            raise ValueError(f"Family {family} has only {len(members)} candidates for quota {quota}")\n        if method == "prior_random":\n            candidates = _prior_candidates(rng, members, quota)\n            results.extend(context.review_candidates(candidates, method, seed))\n        else:\n            results.extend(_adaptive_candidates(method, rng, members, quota, context, seed))\n    if len(results) != calls or len({row["smiles"] for row in results}) != len(results):\n        raise RuntimeError("Search failed the exact-budget/uniqueness invariant")\n    for reviewer_call, row in enumerate(results, start=1):\n        row["reviewer_call"] = reviewer_call\n    _apply_diversity_penalty(results, config)\n    retained: list[dict[str, Any]] = []\n    output_quotas = _family_quotas(int(config["execution"]["n_per_run"]), families)\n    for family in families:\n        members = [row for row in results if row["family"] == family]\n        members.sort(key=lambda row: (-float(row.get("reward", 0.0)), row["smiles"]))\n        retained.extend(members[:output_quotas[family]])\n    retained.sort(key=lambda row: (row["family"], -float(row.get("reward", 0.0)), row["smiles"]))\n    for rank, row in enumerate(retained, start=1):\n        row["method_rank"] = rank\n        row["candidate_reward_evaluations_in_run"] = calls\n    if len(retained) != int(config["execution"]["n_per_run"]):\n        raise RuntimeError("Search failed to retain the requested family-balanced output quota")\n    return retained\n\n\ndef run_methods(\n    config: dict[str, Any],\n    library_path: str | Path,\n    reviewer_path: str | Path,\n    output_path: str | Path,\n    methods: list[str],\n) -> dict[str, Any]:\n    all_rows: list[dict[str, Any]] = []\n    run_counts: dict[str, int] = {}\n    ledger_entries: dict[str, dict[str, Any]] = {}\n    for method in methods:\n        for seed in config["execution"]["seeds"]:\n            rows = run_method(config, library_path, reviewer_path, method, int(seed))\n            key = f"{method}:{seed}"\n            run_counts[key] = len(rows)\n            ledger_entries[key] = {\n                "method_id": method, "seed": int(seed),\n                "candidate_reward_evaluations": int(config["execution"]["reviewer_budget_per_run"]),\n                "retained_candidates": len(rows),\n                "source": "reaction_library_search",\n            }\n            all_rows.extend(rows)\n    write_csv(output_path, all_rows)\n    ledger_path = _update_budget_ledger(Path(output_path).parent, ledger_entries)\n    return {\n        "path": str(Path(output_path).resolve()),\n        "rows": len(all_rows),\n        "run_counts": run_counts,\n        "reviewer_call_counts": {key: value["candidate_reward_evaluations"] for key, value in ledger_entries.items()},\n        "budget_ledger": str(ledger_path.resolve()),\n        "methods": methods,\n        "seeds": config["execution"]["seeds"],\n    }\n\n\ndef write_generator_manifests(config: dict[str, Any], output_dir: str | Path) -> dict[str, Any]:\n    destination = Path(output_dir)\n    destination.mkdir(parents=True, exist_ok=True)\n    manifests = []\n    toml_configs = []\n    root = destination.parent\n    scoring_config = destination / "scoring_config.resolved.json"\n    scoring_config.write_text(\n        json.dumps({key: value for key, value in config.items() if not key.startswith("_")}, indent=2, sort_keys=True) + "\\n",\n        encoding="utf-8",\n    )\n    scorer = Path(__file__).resolve().parents[1] / "scripts" / "reinvent_external_score.py"\n    prior_value = config["production"]["reaction_prior_path"] or "PRIOR_PATH_REQUIRED"\n    prior_path = Path(prior_value)\n    if not prior_path.is_absolute():\n        prior_path = Path(__file__).resolve().parents[1] / prior_path\n    prior = str(prior_path.resolve())\n    per_family_budget = math.ceil(int(config["execution"]["reviewer_budget_per_run"]) / len(config["families"]))\n    steps_per_stage = int(config["production"].get("rl_steps_per_stage", 2))\n    batch_size = int(config["production"].get("rl_batch_size", max(1, math.ceil(per_family_budget / (4 * steps_per_stage)))))\n    library_rows = read_csv(root / "data" / "reaction_library.csv") if (root / "data" / "reaction_library.csv").exists() else []\n    synthons = {item["id"]: item["smiles"] for item in config["synthons"]}\n    supported = set(config["production"].get("libinvent_supported_elements", config["elements"]))\n    compatible_ids = {identifier for identifier, smiles in synthons.items()\n                      if {atom.GetSymbol() for atom in parse_smiles(smiles).GetAtoms()} <= supported}\n    for family, family_config in config["families"].items():\n        scaffold = family_config["libinvent_scaffold"]\n        scaffold_path = destination / f"scaffold_{family}.smi"\n        scaffold_path.write_text(scaffold + "\\n", encoding="utf-8")\n        tl_rows = []\n        for row in library_rows:\n            if row["family"] != family:\n                continue\n            if row["synthon_a"] not in compatible_ids or row["synthon_b"] not in compatible_ids:\n                continue\n            first = "*" + synthons[row["synthon_a"]]\n            second = "*" + synthons[row["synthon_b"]]\n            tl_scaffold = scaffold.replace("[*:0]", "[*]").replace("[*:1]", "[*]")\n            tl_rows.append(f"{tl_scaffold}\\t{first}|{second}")\n        tl_train = destination / f"tl_{family}.csv"\n        tl_validation = destination / f"tl_{family}_validation.csv"\n        split = max(1, int(len(tl_rows) * 0.9)) if tl_rows else 0\n        tl_train.write_text("\\n".join(tl_rows[:split]) + "\\n", encoding="utf-8")\n        tl_validation.write_text("\\n".join(tl_rows[split:] or tl_rows[-1:]) + "\\n", encoding="utf-8")\n        tl_agent = destination / f"libinvent_{family}.agent"\n        tl_lines = [\n            \'run_type = "transfer_learning"\', \'device = "cpu"\',\n            f\'tb_logdir = "{(destination / ("tb_tl_" + family)).resolve().as_posix()}"\',\n            f\'json_out_config = "{(destination / ("tl_resolved_" + family + ".json")).resolve().as_posix()}"\',\n            "", "[parameters]", f"num_epochs = {int(config[\'production\'].get(\'transfer_learning_epochs\', 1))}",\n            f"save_every_n_epochs = {int(config[\'production\'].get(\'transfer_learning_epochs\', 1))}",\n            "batch_size = 96", "num_refs = 0", "sample_batch_size = 100", "tb_isim = false",\n            f\'input_model_file = "{prior}"\', f\'smiles_file = "{tl_train.resolve().as_posix()}"\',\n            f\'validation_smiles_file = "{tl_validation.resolve().as_posix()}"\',\n            f\'output_model_file = "{tl_agent.resolve().as_posix()}"\',\n        ]\n        tl_config = destination / f"transfer_learning_{family}.toml"\n        tl_config.write_text("\\n".join(tl_lines) + "\\n", encoding="utf-8")\n        model_path = root / "models" / "reward" / "reviewers.pkl"\n        library_path = root / "data" / "reaction_library.csv"\n        family_toml_configs = []\n        for run_seed in config["execution"]["seeds"]:\n            run_seed = int(run_seed)\n            lines = [\n                \'run_type = "staged_learning"\', \'device = "cpu"\', f"seed = {run_seed}",\n                f\'tb_logdir = "{(destination / ("tb_" + family + "_" + str(run_seed))).resolve().as_posix()}"\',\n                f\'json_out_config = "{(destination / ("resolved_" + family + "_" + str(run_seed) + ".json")).resolve().as_posix()}"\',\n                "", "[parameters]",\n                f\'prior_file = "{prior}"\', f\'agent_file = "{tl_agent.resolve().as_posix()}"\',\n                f\'smiles_file = "{scaffold_path.resolve().as_posix()}"\',\n                f\'summary_csv_prefix = "{(destination / ("rl_" + family + "_" + str(run_seed))).resolve().as_posix()}"\',\n                f"batch_size = {batch_size}", "randomize_smiles = true", "use_checkpoint = false", "purge_memories = false",\n                "", "[learning_strategy]", \'type = "dap"\', "sigma = 128", "rate = 0.0001",\n                "", "[diversity_filter]", \'type = "PenalizeSameSmiles"\', "bucket_size = 25", "minscore = 0.35", "penalty_multiplier = 0.5",\n            ]\n            for index, stage in enumerate(config["reward"]["stages"], start=1):\n                stage_name = stage["name"]\n                args = (\n                    f\'{scorer.resolve().as_posix()} --config {scoring_config.resolve().as_posix()} \'\n                    f\'--model {model_path.resolve().as_posix()} --library {library_path.resolve().as_posix()} \'\n                    f\'--family {family} --stage {stage_name}\'\n                )\n                lines.extend([\n                    "", "[[stage]]",\n                    f\'chkpt_file = "{(destination / (family + "_" + str(run_seed) + "_stage" + str(index) + ".chkpt")).resolve().as_posix()}"\',\n                    \'termination = "simple"\', "max_score = 0.0", f"min_steps = {max(0, steps_per_stage - 2)}", f"max_steps = {steps_per_stage + 2}",\n                    "", "[stage.scoring]", \'type = "geometric_mean"\',\n                    "", "[[stage.scoring.component]]", "[stage.scoring.component.ExternalProcess]",\n                    "[[stage.scoring.component.ExternalProcess.endpoint]]", f\'name = "MOSTGen {stage_name}"\', "weight = 1.0",\n                    f\'params.executable = "{Path(sys.executable).absolute().as_posix()}"\', f\'params.args = "{args}"\',\n                    \'params.property = "stage_score"\',\n                ])\n            toml_path = destination / f"reinvent4_{family}_{run_seed}.toml"\n            toml_path.write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n            family_toml_configs.append(str(toml_path.resolve()))\n            toml_configs.append(str(toml_path.resolve()))\n        manifest = {\n            "schema_version": "1.0",\n            "family": family,\n            "generator": "REINVENT4 LibInvent",\n            "backend_status": "surrogate_smoke" if config["execution"]["backend"] == "surrogate_smoke" else "external_required",\n            "repository": config["production"]["reinvent4_repository"],\n            "revision": config["production"]["reinvent4_revision"],\n            "version": config["production"]["reinvent4_version"],\n            "prior": config["production"]["reaction_prior_path"] or "NOT_CONFIGURED",\n            "reaction_smarts": family_config["reaction_smarts"],\n            "allowed_positions": family_config["allowed_positions"],\n            "curriculum": config["reward"]["stages"],\n            "reward_weights": config["reward"]["weights"],\n            "diversity_filter": {"type": "ECFP centroid", "penalty": config["reward"]["diversity_penalty"]},\n            "seed_runs": config["execution"]["seeds"],\n            "reviewer_budget_per_run": config["execution"]["reviewer_budget_per_run"],\n            "toml_configs": family_toml_configs,\n            "scaffold_file": str(scaffold_path.resolve()),\n            "external_scorer": str(scorer.resolve()),\n            "transfer_learning_config": str(tl_config.resolve()),\n            "transfer_learning_rows": len(tl_rows),\n            "trained_agent": str(tl_agent.resolve()),\n        }\n        path = destination / f"reinvent4_{family}.json"\n        path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n        manifests.append(str(path.resolve()))\n    policy = {\n        "status": "manifest_only" if config["execution"]["mode"] == "smoke" else "ready_to_train_reinvent4",\n        "family_manifests": manifests,\n        "reinvent_toml_configs": toml_configs,\n        "reinvent_command": "reinvent -l <family>.log <family>.toml",\n        "note": "The smoke policy is an adaptive reaction-library search, not a trained REINVENT neural prior.",\n    }\n    (destination / "generator_policy.json").write_text(json.dumps(policy, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    return policy\n\n\ndef execute_generator_training(config: dict[str, Any], output_dir: str | Path) -> dict[str, Any]:\n    """Run real LibInvent transfer learning with the pinned public prior."""\n    destination = Path(output_dir)\n    executable = Path(config["production"].get("reinvent_executable", "reinvent"))\n    if not executable.is_absolute():\n        executable = Path(__file__).resolve().parents[1] / executable\n    if not executable.is_file():\n        raise FileNotFoundError(f"REINVENT4 executable is unavailable: {executable}")\n    runs = []\n    for family in sorted(config["families"]):\n        toml = destination / f"transfer_learning_{family}.toml"\n        log = destination / f"transfer_learning_{family}.log"\n        completed = subprocess.run([str(executable), "-l", str(log), str(toml)], cwd=Path(__file__).resolve().parents[1],\n                                   text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False)\n        agent = destination / f"libinvent_{family}.agent"\n        runs.append({"family": family, "returncode": completed.returncode, "agent": str(agent.resolve()),\n                     "agent_exists": agent.exists(), "log": str(log.resolve())})\n        if completed.returncode != 0 or not agent.exists():\n            tail = completed.stdout[-1200:] if completed.stdout else ""\n            raise RuntimeError(f"LibInvent transfer learning failed for {family}; inspect {log}; {tail}")\n    result = {"status": "trained", "generator": "REINVENT4 LibInvent", "runs": runs}\n    (destination / "training_result.json").write_text(json.dumps(result, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    return result\n\n\ndef execute_reinvent_curriculum(config: dict[str, Any], output_dir: str | Path) -> dict[str, Any]:\n    """Execute the real four-stage REINVENT4 curriculum for each family."""\n    destination = Path(output_dir)\n    executable = Path(config["production"].get("reinvent_executable", "reinvent"))\n    if not executable.is_absolute():\n        executable = Path(__file__).resolve().parents[1] / executable\n    runs = []\n    for run_seed in config["execution"]["seeds"]:\n        run_seed = int(run_seed)\n        for family in sorted(config["families"]):\n            toml = destination / f"reinvent4_{family}_{run_seed}.toml"\n            log = destination / f"curriculum_{family}_{run_seed}.log"\n            completed = subprocess.run([str(executable), "-l", str(log), str(toml)], cwd=Path(__file__).resolve().parents[1],\n                                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False)\n            stage_files = sorted(destination.glob(f"rl_{family}_{run_seed}_*.csv"))\n            evaluation_rows = sum(max(0, sum(1 for _ in path.open(encoding="utf-8")) - 1) for path in stage_files)\n            run = {"family": family, "seed": run_seed, "returncode": completed.returncode,\n                   "log": str(log.resolve()), "stage_files": [str(path.resolve()) for path in stage_files],\n                   "optimization_reviewer_evaluations": evaluation_rows}\n            runs.append(run)\n            if completed.returncode != 0 or len(stage_files) != len(config["reward"]["stages"]):\n                tail = completed.stdout[-1200:] if completed.stdout else ""\n                raise RuntimeError(f"REINVENT4 curriculum failed for {family}, seed {run_seed}; {tail}")\n    result = {"status": "complete", "backend": "REINVENT4 LibInvent staged_learning", "runs": runs,\n              "optimization_reviewer_evaluations": sum(run["optimization_reviewer_evaluations"] for run in runs)}\n    (destination / "curriculum_result.json").write_text(json.dumps(result, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    return result\n\n\ndef collect_reinvent_curriculum(config: dict[str, Any], generator_dir: str | Path,\n                                 library_path: str | Path, reviewer_path: str | Path,\n                                 output_path: str | Path) -> dict[str, Any]:\n    """Map actual LibInvent outputs back to the allowed commercial library.\n\n    REINVENT is allowed to propose arbitrary decorators, but only exact members\n    of the versioned synthon library are eligible for reviewer scoring.\n    """\n    import csv\n\n    destination = Path(generator_dir)\n    library = read_csv(library_path)\n    lookup = {row["smiles"]: row for row in library}\n    reviewers = ReviewerBundle.load(reviewer_path)\n    context = ScoringContext(config, reviewers)\n    raw_count = valid_count = exact_count = 0\n    unique: dict[tuple[str, str], tuple[dict[str, Any], dict[str, Any]]] = {}\n    run_seed = int(config["project"]["default_seed"])\n    for family in sorted(config["families"]):\n        for stage_index, stage in enumerate(config["reward"]["stages"], start=1):\n            path = destination / f"rl_{family}_{run_seed}_{stage_index}.csv"\n            if not path.exists():\n                continue\n            with path.open(encoding="utf-8", newline="") as handle:\n                for generated in csv.DictReader(handle):\n                    raw_count += 1\n                    if str(generated.get("SMILES_state")) == "0":\n                        continue\n                    valid_count += 1\n                    try:\n                        from .chemistry import standardize_smiles\n                        canonical = standardize_smiles(generated["SMILES"])\n                    except Exception:\n                        continue\n                    candidate = lookup.get(canonical)\n                    if candidate is None or candidate["family"] != family:\n                        continue\n                    exact_count += 1\n                    key = (family, canonical)\n                    metadata = {"reinvent_stage": stage["name"], "reinvent_stage_index": stage_index,\n                                "reinvent_raw_score": generated.get("Score"), "reinvent_step": generated.get("step"),\n                                "reinvent_r_groups": generated.get("R-groups")}\n                    previous = unique.get(key)\n                    if previous is None or float(generated.get("Score") or 0.0) > float(previous[1].get("reinvent_raw_score") or 0.0):\n                        unique[key] = (candidate, metadata)\n    candidates = [item[0] for item in unique.values()]\n    reviewed = context.review_candidates(candidates, "libinvent_rl", int(config["project"]["default_seed"]))\n    metadata_by_key = {key: item[1] for key, item in unique.items()}\n    for row in reviewed:\n        row.update(metadata_by_key[(row["family"], row["smiles"])])\n    _apply_diversity_penalty(reviewed, config)\n    write_csv(output_path, reviewed)\n    result = {"raw_reinvent_evaluations": raw_count, "valid_reinvent_outputs": valid_count,\n              "commercial_library_matches": exact_count, "unique_eligible": len(reviewed),\n              "path": str(Path(output_path).resolve()),\n              "acceptance_1000_unique": len(reviewed) >= 1000}\n    (destination / "curriculum_collection.json").write_text(json.dumps(result, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    return result\n\n\ndef sample_reinvent_candidates(config: dict[str, Any], generator_dir: str | Path,\n                               library_path: str | Path, reviewer_path: str | Path,\n                               output_path: str | Path, seed: int | None = None) -> dict[str, Any]:\n    """Sample real LibInvent agents until the requested unique quota is met.\n\n    The curriculum checkpoint is sampled first.  The less-collapsed TL agent is\n    then used as a diversity backstop.  Structures outside the exact versioned\n    commercial-synthon library are recorded but never returned or scored.\n    """\n    import csv\n\n    destination = Path(generator_dir)\n    executable = Path(config["production"].get("reinvent_executable", "reinvent"))\n    if not executable.is_absolute():\n        executable = Path(__file__).resolve().parents[1] / executable\n    seed = int(seed if seed is not None else config["project"]["default_seed"])\n    total_requested = int(config["execution"]["n_per_run"])\n    quotas = _family_quotas(total_requested, sorted(config["families"]))\n    library = read_csv(library_path)\n    by_family = {family: {row["smiles"]: row for row in library if row["family"] == family}\n                 for family in config["families"]}\n    context = ScoringContext(config, ReviewerBundle.load(reviewer_path))\n    selected: list[dict[str, Any]] = []\n    audit = []\n    for family in sorted(config["families"]):\n        eligible: dict[str, tuple[dict[str, Any], dict[str, Any]]] = {}\n        agents = [\n            ("rl_checkpoint", destination / f"{family}_{seed}_stage4.chkpt"),\n            ("transfer_learning_agent", destination / f"libinvent_{family}.agent"),\n        ]\n        for agent_kind, agent in agents:\n            if len(eligible) >= quotas[family]:\n                break\n            if not agent.exists():\n                continue\n            for round_index in range(int(config["production"].get("sampling_rounds", 3))):\n                if len(eligible) >= quotas[family]:\n                    break\n                sample_count = max(2000, quotas[family] * int(config["production"].get("sampling_oversample", 12)))\n                sample_file = destination / f"sample_{family}_{seed}_{agent_kind}_{round_index}.csv"\n                sample_config = destination / f"sample_{family}_{seed}_{agent_kind}_{round_index}.toml"\n                sample_log = destination / f"sample_{family}_{seed}_{agent_kind}_{round_index}.log"\n                sample_config.write_text("\\n".join([\n                    \'run_type = "sampling"\', \'device = "cpu"\', f"seed = {seed + 1009 * round_index}", "",\n                    "[parameters]", f\'model_file = "{agent.resolve().as_posix()}"\',\n                    f\'smiles_file = "{(destination / ("scaffold_" + family + ".smi")).resolve().as_posix()}"\',\n                    f\'output_file = "{sample_file.resolve().as_posix()}"\', f"num_smiles = {sample_count}",\n                    "unique_molecules = true", "randomize_smiles = true", "",\n                ]), encoding="utf-8")\n                completed = subprocess.run([str(executable), "-l", str(sample_log), str(sample_config)],\n                                           cwd=Path(__file__).resolve().parents[1], text=True,\n                                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False)\n                if completed.returncode != 0 or not sample_file.exists():\n                    raise RuntimeError(f"LibInvent sampling failed for {family}/{agent_kind}; inspect {sample_log}")\n                raw = valid = matches = 0\n                with sample_file.open(encoding="utf-8", newline="") as handle:\n                    for record in csv.DictReader(handle):\n                        raw += 1\n                        if str(record.get("SMILES_state")) == "0":\n                            continue\n                        valid += 1\n                        try:\n                            from .chemistry import standardize_smiles\n                            canonical = standardize_smiles(record["SMILES"])\n                        except Exception:\n                            continue\n                        candidate = by_family[family].get(canonical)\n                        if candidate is None:\n                            continue\n                        matches += 1\n                        eligible.setdefault(canonical, (candidate, {\n                            "reinvent_sampling_agent": agent_kind, "reinvent_sampling_round": round_index,\n                            "reinvent_nll": record.get("NLL"), "reinvent_r_groups": record.get("R-groups"),\n                        }))\n                audit.append({"family": family, "agent": agent_kind, "round": round_index,\n                              "requested": sample_count, "returned_unique": raw, "valid": valid,\n                              "commercial_matches": matches, "eligible_cumulative": len(eligible)})\n        if len(eligible) < quotas[family]:\n            raise RuntimeError(f"LibInvent underfilled {family}: {len(eligible)}/{quotas[family]} unique allowed molecules")\n        # Select without another reviewer call. RL-checkpoint samples are\n        # preferred, followed by lower-NLL samples; the stable hash resolves\n        # ties reproducibly. Exactly the final quota is then evaluated once.\n        eligible_items = list(eligible.items())\n        eligible_items.sort(key=lambda item: (\n            item[1][1].get("reinvent_sampling_agent") != "rl_checkpoint",\n            float(item[1][1].get("reinvent_nll") or float("inf")),\n            stable_hash(f"{seed}:{item[0]}", 16),\n        ))\n        eligible_items = eligible_items[:quotas[family]]\n        family_candidates = [value[0] for _, value in eligible_items]\n        family_metadata = {key: value[1] for key, value in eligible.items()}\n        family_reviewed = context.review_candidates(family_candidates, "libinvent_rl", seed)\n        for row in family_reviewed:\n            row.update(family_metadata[row["smiles"]])\n        selected.extend(family_reviewed)\n    _apply_diversity_penalty(selected, config)\n    write_csv(output_path, selected)\n    result = {"seed": seed, "rows": len(selected), "unique": len({row["smiles"] for row in selected}),\n              "reviewer_calls": len(selected), "quota_met": len(selected) == total_requested,\n              "audit": audit, "path": str(Path(output_path).resolve())}\n    audit_path = destination / f"sampling_audit_{seed}.json"\n    audit_path.write_text(json.dumps(result, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    return result\n\n\ndef run_reinvent_generation(config: dict[str, Any], generator_dir: str | Path,\n                             library_path: str | Path, reviewer_path: str | Path,\n                             output_path: str | Path) -> dict[str, Any]:\n    """Run real curriculum learning and sample one exact-budget set per seed."""\n    curriculum = execute_reinvent_curriculum(config, generator_dir)\n    destination = Path(generator_dir)\n    all_rows: list[dict[str, Any]] = []\n    run_counts: dict[str, int] = {}\n    sampling = []\n    optimization_by_seed: dict[int, int] = defaultdict(int)\n    for run in curriculum["runs"]:\n        optimization_by_seed[int(run["seed"])] += int(run["optimization_reviewer_evaluations"])\n    ledger_entries: dict[str, dict[str, Any]] = {}\n    for run_seed in config["execution"]["seeds"]:\n        run_seed = int(run_seed)\n        seed_path = destination / f"generated_libinvent_{run_seed}.csv"\n        result = sample_reinvent_candidates(\n            config, destination, library_path, reviewer_path, seed_path, seed=run_seed,\n        )\n        rows = read_csv(seed_path)\n        all_rows.extend(rows)\n        key = f"libinvent_rl:{run_seed}"\n        run_counts[key] = len(rows)\n        total_calls = optimization_by_seed[run_seed] + int(result["reviewer_calls"])\n        expected_calls = int(config["execution"]["reviewer_budget_per_run"])\n        if total_calls != expected_calls:\n            raise RuntimeError(\n                f"LibInvent reviewer budget mismatch for seed {run_seed}: {total_calls} != {expected_calls}"\n            )\n        for row in rows:\n            row["candidate_reward_evaluations_in_run"] = total_calls\n        ledger_entries[key] = {\n            "method_id": "libinvent_rl", "seed": run_seed,\n            "candidate_reward_evaluations": total_calls,\n            "optimization_evaluations": optimization_by_seed[run_seed],\n            "final_scoring_evaluations": int(result["reviewer_calls"]),\n            "retained_candidates": len(rows),\n            "source": "REINVENT4_curriculum_plus_final_scoring",\n        }\n        sampling.append(result)\n    write_csv(output_path, all_rows)\n    ledger_path = _update_budget_ledger(Path(output_path).parent, ledger_entries)\n    return {\n        "path": str(Path(output_path).resolve()), "rows": len(all_rows),\n        "run_counts": run_counts, "methods": ["libinvent_rl"],\n        "seeds": list(config["execution"]["seeds"]), "curriculum": curriculum,\n        "sampling": sampling, "backend": "REINVENT4 LibInvent staged_learning",\n        "reviewer_call_counts": {key: value["candidate_reward_evaluations"] for key, value in ledger_entries.items()},\n        "budget_ledger": str(ledger_path.resolve()),\n    }\n'
load_embedded('mostgen.search', _source, PROJECT_ROOT / 'mostgen' / 'search.py')
print('loaded mostgen.search')


loaded mostgen.search


### 3.28 Diversity, bootstrap, ESS и абляции — `metrics.py`


In [14]:
_source = 'from __future__ import annotations\n\nimport json\nimport math\nimport random\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nfrom statistics import fmean\nfrom typing import Any, Callable\n\nfrom rdkit import DataStructs\n\nfrom .chemistry import fingerprint, murcko_scaffold\nfrom .data import read_csv, write_csv\n\n\ndef _truth(value: Any) -> bool:\n    return value is True or str(value).lower() in {"true", "1", "yes"}\n\n\ndef _float(row: dict[str, Any], key: str, default: float = 0.0) -> float:\n    try:\n        return float(row.get(key, default))\n    except (TypeError, ValueError):\n        return default\n\n\ndef _internal_diversity(rows: list[dict[str, Any]], bits: int, seed: int = 701) -> float:\n    unique = sorted({row["smiles"] for row in rows})\n    if len(unique) < 2:\n        return 0.0\n    rng = random.Random(seed)\n    pairs = []\n    max_pairs = min(500, len(unique) * (len(unique) - 1) // 2)\n    seen = set()\n    while len(pairs) < max_pairs:\n        left, right = sorted(rng.sample(range(len(unique)), 2))\n        if (left, right) in seen:\n            continue\n        seen.add((left, right))\n        pairs.append((left, right))\n    fps = [fingerprint(smiles, bits) for smiles in unique]\n    return fmean(1.0 - float(DataStructs.TanimotoSimilarity(fps[left], fps[right])) for left, right in pairs)\n\n\ndef _effective_sample_size(weights: list[float]) -> float:\n    total = sum(weights)\n    squares = sum(value * value for value in weights)\n    return total * total / squares if squares else 0.0\n\n\ndef _run_metrics(rows: list[dict[str, Any]], training_smiles: set[str], bits: int) -> dict[str, Any]:\n    n = len(rows)\n    unique = {row["smiles"] for row in rows}\n    valid = [row for row in rows if _truth(row.get("valid"))]\n    scaffolds = {murcko_scaffold(row["smiles"]) for row in valid}\n    families = Counter(row["family"] for row in valid)\n    rewards = [_float(row, "reward") for row in rows]\n    return {\n        "evaluated": n,\n        "validity": len(valid) / n if n else 0.0,\n        "uniqueness": len(unique) / n if n else 0.0,\n        "novelty": sum(row["smiles"] not in training_smiles for row in valid) / len(valid) if valid else 0.0,\n        "internal_diversity": _internal_diversity(valid, bits),\n        "scaffold_diversity": len(scaffolds) / len(valid) if valid else 0.0,\n        "mean_sa_score": fmean(_float(row, "sa_score", 10.0) for row in valid) if valid else 10.0,\n        "joint_success": sum(_truth(row.get("joint_pass")) for row in rows) / n if n else 0.0,\n        "both_ad_fraction": sum(_truth(row.get("ad_spectral")) and _truth(row.get("ad_most")) for row in rows) / n if n else 0.0,\n        "family_coverage": sum(1 for family in families if families[family] > 0) / 3.0,\n        "nbd_qc_count": families["nbd_qc"],\n        "dewar_pyrimidinone_count": families["dewar_pyrimidinone"],\n        "spiropyran_count": families["spiropyran"],\n        "nonzero_reward_fraction": sum(value > 0.0 for value in rewards) / n if n else 0.0,\n        "effective_sample_size": _effective_sample_size(rewards),\n        "unique_ecfp_clusters": len({row.get("ecfp_cluster") for row in rows}),\n    }\n\n\ndef _bootstrap(values: list[float], samples: int, seed: int) -> tuple[float, float, float]:\n    if not values:\n        return math.nan, math.nan, math.nan\n    if len(values) == 1:\n        return values[0], values[0], values[0]\n    rng = random.Random(seed)\n    means = []\n    for _ in range(samples):\n        means.append(fmean(rng.choice(values) for _ in values))\n    means.sort()\n    low = means[max(0, int(0.025 * len(means)) - 1)]\n    high = means[min(len(means) - 1, int(0.975 * len(means)))]\n    return fmean(values), low, high\n\n\ndef _ablation_rows(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:\n    result = []\n    grouped: dict[str, list[dict[str, Any]]] = defaultdict(list)\n    for row in rows:\n        grouped[row["method_id"]].append(row)\n    for method, members in sorted(grouped.items()):\n        gates: dict[str, Callable[[dict[str, Any]], bool]] = {\n            "full": lambda r: _truth(r.get("joint_pass")),\n            "without_uncertainty_penalty": lambda r: (\n                _float(r, "uvb_auc") >= _float(r, "reference_uvb_median")\n                and _float(r, "uva_auc") >= _float(r, "reference_uva_median")\n                and _float(r, "lambda_c_nm") >= 370.0\n                and _float(r, "energy_kj_mol") >= _float(r, "reference_energy_median")\n                and 4.0 <= _float(r, "half_life_h") <= 24.0\n                and _truth(r.get("safety_pass")) and _truth(r.get("ad_spectral")) and _truth(r.get("ad_most"))\n            ),\n            "without_safety_gate": lambda r: _truth(r.get("joint_uv_pass")) and _truth(r.get("most_pass")) and _truth(r.get("ad_spectral")) and _truth(r.get("ad_most")),\n            "without_ad_gate": lambda r: _truth(r.get("joint_uv_pass")) and _truth(r.get("most_pass")) and _truth(r.get("safety_pass")),\n        }\n        for setting, predicate in gates.items():\n            passed = sum(predicate(row) for row in members)\n            result.append({"method_id": method, "ablation": setting, "evaluated": len(members), "eligible": passed, "success_fraction": passed / len(members) if members else 0.0})\n        top_adjusted = sorted(members, key=lambda row: -_float(row, "reward"))[: min(100, len(members))]\n        top_raw = sorted(members, key=lambda row: -_float(row, "reward_pre_diversity"))[: min(100, len(members))]\n        result.append({\n            "method_id": method, "ablation": "without_diversity_filter",\n            "evaluated": len(members), "eligible": sum(_truth(row.get("joint_pass")) for row in top_raw),\n            "success_fraction": sum(_truth(row.get("joint_pass")) for row in top_raw) / len(top_raw) if top_raw else 0.0,\n            "top100_cluster_diversity_full": len({row.get("ecfp_cluster") for row in top_adjusted}) / len(top_adjusted) if top_adjusted else 0.0,\n            "top100_cluster_diversity_ablated": len({row.get("ecfp_cluster") for row in top_raw}) / len(top_raw) if top_raw else 0.0,\n        })\n    return result\n\n\ndef compute_metrics(\n    generated_path: str | Path,\n    training_path: str | Path,\n    output_dir: str | Path,\n    config: dict[str, Any],\n) -> dict[str, Any]:\n    rows = read_csv(generated_path)\n    training_smiles = {row["smiles"] for row in read_csv(training_path)}\n    grouped: dict[tuple[str, str], list[dict[str, Any]]] = defaultdict(list)\n    for row in rows:\n        grouped[(row["method_id"], row["seed"])].append(row)\n    run_rows = []\n    bits = int(config["reviewers"]["fingerprint_bits"])\n    for (method, seed), members in sorted(grouped.items()):\n        run_rows.append({"method_id": method, "seed": int(seed), **_run_metrics(members, training_smiles, bits)})\n    summary_rows = []\n    metric_names = ("validity", "uniqueness", "novelty", "internal_diversity", "scaffold_diversity", "mean_sa_score", "joint_success", "both_ad_fraction", "family_coverage")\n    for method in sorted({row["method_id"] for row in run_rows}):\n        members = [row for row in run_rows if row["method_id"] == method]\n        summary: dict[str, Any] = {"method_id": method, "seed_runs": len(members)}\n        for metric in metric_names:\n            mean, low, high = _bootstrap([float(row[metric]) for row in members], int(config["execution"]["bootstrap_samples"]), int(config["project"]["default_seed"]) ^ sum(map(ord, method + metric)))\n            summary[f"{metric}_mean"] = mean\n            summary[f"{metric}_ci_low"] = low\n            summary[f"{metric}_ci_high"] = high\n        summary_rows.append(summary)\n    diagnostics = []\n    for (method, seed), members in sorted(grouped.items()):\n        component_values: dict[str, list[float]] = defaultdict(list)\n        for row in members:\n            try:\n                parsed = json.loads(row.get("reward_components_json", "{}"))\n            except json.JSONDecodeError:\n                parsed = {}\n            for key, value in parsed.items():\n                component_values[key].append(float(value))\n        rewards = [_float(row, "reward") for row in members]\n        diagnostics.append({\n            "method_id": method, "seed": seed,\n            "nonzero_fraction": sum(value > 0 for value in rewards) / len(rewards) if rewards else 0.0,\n            "effective_sample_size": _effective_sample_size(rewards),\n            "largest_cluster_fraction": max(Counter(row.get("ecfp_cluster") for row in members).values()) / len(members) if members else 0.0,\n            **{f"component_{key}_mean": fmean(values) for key, values in sorted(component_values.items())},\n        })\n    destination = Path(output_dir)\n    destination.mkdir(parents=True, exist_ok=True)\n    write_csv(destination / "metrics_runs.csv", run_rows)\n    write_csv(destination / "metrics_summary.csv", summary_rows)\n    write_csv(destination / "ablations.csv", _ablation_rows(rows))\n    write_csv(destination / "reward_diagnostics.csv", diagnostics)\n    ledger_path = destination.parent / "search_budget_ledger.json"\n    ledger = json.loads(ledger_path.read_text(encoding="utf-8")) if ledger_path.exists() else {"runs": {}}\n    call_counts = {\n        int(item.get("candidate_reward_evaluations", -1))\n        for item in ledger.get("runs", {}).values()\n    }\n    result = {\n        "run_metrics": run_rows, "summary": summary_rows,\n        "matched_final_set_size": len({len(group) for group in grouped.values()}) == 1,\n        "matched_reviewer_budget": bool(call_counts) and len(call_counts) == 1,\n        "reviewer_call_counts": {\n            key: int(item.get("candidate_reward_evaluations", -1))\n            for key, item in sorted(ledger.get("runs", {}).items())\n        },\n    }\n    (destination / "metrics.json").write_text(json.dumps(result, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    return result\n'
load_embedded('mostgen.metrics', _source, PROJECT_ROOT / 'mostgen' / 'metrics.py')
print('loaded mostgen.metrics')


loaded mostgen.metrics


### 3.30 GFN2-xTB/sTDA-xTB — `oracle.py`


In [15]:
_source = 'from __future__ import annotations\n\nimport importlib.util\nimport json\nimport math\nimport os\nimport shutil\nimport subprocess\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nimport numpy as np\nfrom rdkit import Chem\nfrom rdkit.Chem import AllChem, Descriptors\n\nfrom .data import WAVELENGTHS, write_csv\nfrom .numerics import beer_lambert_transmittance, critical_wavelength, trapezoid_auc\nfrom .provenance import stable_hash\n\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef _resolved_executable(value: str) -> str | None:\n    path = Path(value)\n    if not path.is_absolute():\n        path = ROOT / path\n    if path.is_file() and os.access(path, os.X_OK):\n        return str(path.resolve())\n    return shutil.which(value)\n\n\ndef availability(config: dict[str, Any]) -> dict[str, Any]:\n    production = config["production"]\n    tools = {\n        "xtb_python": importlib.util.find_spec("xtb") is not None,\n        "ase": importlib.util.find_spec("ase") is not None,\n        "stda": _resolved_executable(production["stda_executable"]),\n        "xtb4stda": _resolved_executable(production.get("xtb4stda_executable", "xtb4stda")),\n    }\n    return {"tools": tools, "energy_ready": bool(tools["xtb_python"] and tools["ase"]),\n            "spectrum_ready": bool(tools["stda"] and tools["xtb4stda"]),\n            "ready": all(bool(value) for value in tools.values())}\n\n\ndef _conformers(smiles: str, seed: int, count: int) -> tuple[Chem.Mol, list[tuple[float, int, str]]]:\n    mol = Chem.MolFromSmiles(smiles)\n    if mol is None:\n        raise ValueError("invalid_smiles")\n    mol = Chem.AddHs(mol)\n    conformer_ids: list[int] = []\n    for retry, random_coordinates in enumerate((False, True, True)):\n        params = AllChem.ETKDGv3()\n        params.randomSeed = int((seed + 7919 * retry) & 0x7FFFFFFF)\n        params.useRandomCoords = random_coordinates\n        params.enforceChirality = True\n        params.pruneRmsThresh = 0.35\n        conformer_ids = list(AllChem.EmbedMultipleConfs(mol, numConfs=count, params=params))\n        if conformer_ids:\n            break\n    if not conformer_ids:\n        raise RuntimeError("rdkit_conformer_embedding_failed_after_retries")\n    energies = []\n    for conf_id in conformer_ids:\n        try:\n            if AllChem.MMFFHasAllMoleculeParams(mol):\n                properties = AllChem.MMFFGetMoleculeProperties(mol)\n                forcefield = AllChem.MMFFGetMoleculeForceField(mol, properties, confId=conf_id)\n                method = "MMFF94"\n            else:\n                forcefield = AllChem.UFFGetMoleculeForceField(mol, confId=conf_id)\n                method = "UFF"\n            forcefield.Minimize(maxIts=800)\n            energies.append((float(forcefield.CalcEnergy()), int(conf_id), method))\n        except Exception:\n            continue\n    if not energies:\n        raise RuntimeError("forcefield_conformer_minimization_failed")\n    return mol, sorted(energies)\n\n\ndef _xtb_optimize(smiles: str, workdir: Path, stem: str, seed: int, config: dict[str, Any]) -> dict[str, Any]:\n    from ase import Atoms\n    from ase.io import write as ase_write\n    from ase.optimize import BFGS\n    from xtb.ase.calculator import XTB\n\n    mol, ranked = _conformers(smiles, seed, int(config["oracle"]["conformers"]))\n    candidates = ranked[: min(3, len(ranked))]\n    results = []\n    net_charge = int(Chem.GetFormalCharge(mol))\n    electrons = sum(atom.GetAtomicNum() for atom in mol.GetAtoms()) - net_charge\n    uhf = electrons % 2\n    for ff_energy, conf_id, ff_method in candidates:\n        conformer = mol.GetConformer(conf_id)\n        atoms = Atoms(numbers=[atom.GetAtomicNum() for atom in mol.GetAtoms()],\n                      positions=np.asarray(conformer.GetPositions(), dtype=float))\n        charges = np.zeros(len(atoms)); charges[0] = net_charge\n        moments = np.zeros(len(atoms)); moments[0] = uhf\n        atoms.set_initial_charges(charges)\n        atoms.set_initial_magnetic_moments(moments)\n        atoms.calc = XTB(method="GFN2-xTB", accuracy=1.0, max_iterations=250)\n        optimizer = BFGS(atoms, logfile=None)\n        optimizer.run(fmax=float(config["oracle"]["xtb_fmax_ev_a"]), steps=int(config["oracle"]["xtb_max_steps"]))\n        results.append((float(atoms.get_potential_energy()), atoms, ff_energy, ff_method,\n                        optimizer.converged(), optimizer.get_number_of_steps()))\n    energy_ev, atoms, ff_energy, ff_method, converged, steps = min(results, key=lambda item: item[0])\n    xyz = workdir / f"{stem}.xyz"\n    ase_write(xyz, atoms, format="xyz")\n    return {"energy_ev": energy_ev, "xyz": str(xyz), "conformers_embedded": len(ranked),\n            "conformers_xtb_optimized": len(candidates), "forcefield": ff_method,\n            "selected_forcefield_energy": ff_energy, "xtb_converged": bool(converged),\n            "xtb_steps": int(steps), "formal_charge": net_charge, "uhf": int(uhf)}\n\n\ndef _stda_spectrum(xyz: Path, workdir: Path, stem: str, config: dict[str, Any]) -> dict[str, Any]:\n    production = config["production"]\n    run_dir = workdir / f"stda_{stem}"\n    run_dir.mkdir(parents=True, exist_ok=True)\n    local_xyz = run_dir / "structure.xyz"\n    shutil.copy2(xyz, local_xyz)\n    env = dict(os.environ)\n    home = Path(production["xtb4stda_home"])\n    if not home.is_absolute():\n        home = ROOT / home\n    env.update({"XTB4STDAHOME": str(home.resolve()), "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2"})\n    xtb4stda = _resolved_executable(production["xtb4stda_executable"])\n    stda = _resolved_executable(production["stda_executable"])\n    first = subprocess.run([str(xtb4stda), local_xyz.name], cwd=run_dir, env=env, text=True,\n                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=300, check=False)\n    if first.returncode != 0 or not (run_dir / "wfn.xtb").exists():\n        raise RuntimeError(f"xtb4stda_failed:{first.returncode}")\n    second = subprocess.run([str(stda), "-xtb"], cwd=run_dir, env=env, text=True,\n                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=300, check=False)\n    if second.returncode != 0 or not (run_dir / "tda.dat").exists():\n        raise RuntimeError(f"stda_failed:{second.returncode}")\n    transitions = []\n    in_table = False\n    for line in second.stdout.splitlines():\n        if "state    eV      nm" in line:\n            in_table = True\n            continue\n        if in_table:\n            fields = line.split()\n            if len(fields) < 4 or not fields[0].isdigit():\n                if transitions:\n                    break\n                continue\n            try:\n                transitions.append((float(fields[1]), float(fields[2]), max(0.0, float(fields[3]))))\n            except ValueError:\n                continue\n    width_ev = float(config["oracle"]["spectral_broadening_ev"])\n    curve = []\n    for wavelength in WAVELENGTHS:\n        energy_ev = 1239.841984 / wavelength\n        curve.append(sum(oscillator * math.exp(-0.5 * ((energy_ev - transition_ev) / width_ev) ** 2)\n                         for transition_ev, _, oscillator in transitions))\n    loading = float(config["reviewers"].get("beer_lambert_loading_scale", 1.0))\n    uvb_curve = [value for wavelength, value in zip(WAVELENGTHS, curve) if 290 <= wavelength <= 320]\n    uva_curve = [value for wavelength, value in zip(WAVELENGTHS, curve) if 320 <= wavelength <= 400]\n    return {"transitions": transitions, "curve": curve,\n            "uvb_auc": trapezoid_auc(WAVELENGTHS, curve, 290.0, 320.0),\n            "uva_auc": trapezoid_auc(WAVELENGTHS, curve, 320.0, 400.0),\n            "lambda_c_nm": critical_wavelength(WAVELENGTHS, curve),\n            "uvb_transmittance": beer_lambert_transmittance(uvb_curve, loading),\n            "uva_transmittance": beer_lambert_transmittance(uva_curve, loading),\n            "beer_lambert_loading_scale": loading,\n            "output": str((run_dir / "tda.dat").resolve())}\n\n\ndef evaluate_pair(row: dict[str, Any], destination: Path, config: dict[str, Any], rank: int) -> dict[str, Any]:\n    candidate_id = row.get("candidate_id") or stable_hash(row["smiles"], 20)\n    workdir = destination / candidate_id\n    workdir.mkdir(parents=True, exist_ok=True)\n    seed = int(config["project"]["default_seed"]) + rank * 1009\n    ground = _xtb_optimize(row["smiles"], workdir, "ground", seed, config)\n    charged = _xtb_optimize(row["charged_smiles"], workdir, "charged", seed + 500_003, config)\n    delta_kj_mol = (charged["energy_ev"] - ground["energy_ev"]) * 96.4853321233\n    molecular_weight = float(Descriptors.MolWt(Chem.MolFromSmiles(row["smiles"])))\n    ground_spectrum = _stda_spectrum(Path(ground["xyz"]), workdir, "ground", config)\n    charged_spectrum = _stda_spectrum(Path(charged["xyz"]), workdir, "charged", config)\n    return {"candidate_id": candidate_id, "oracle_rank": rank, "family": row["family"],\n            "gfn2_delta_e_kj_mol": delta_kj_mol,\n            "gfn2_specific_energy_wh_kg": delta_kj_mol * 277.7777778 / molecular_weight,\n            "gfn2_ground_energy_ev": ground["energy_ev"], "gfn2_charged_energy_ev": charged["energy_ev"],\n            "ground_xtb_converged": ground["xtb_converged"], "charged_xtb_converged": charged["xtb_converged"],\n            "stda_uvb_auc": ground_spectrum["uvb_auc"], "stda_uva_auc": ground_spectrum["uva_auc"],\n            "stda_lambda_c_nm": ground_spectrum["lambda_c_nm"],\n            "stda_uvb_transmittance": ground_spectrum["uvb_transmittance"],\n            "stda_uva_transmittance": ground_spectrum["uva_transmittance"],\n            "stda_beer_lambert_loading_scale": ground_spectrum["beer_lambert_loading_scale"],\n            "stda_ground_spectrum_json": json.dumps(list(zip(WAVELENGTHS, ground_spectrum["curve"]))),\n            "stda_charged_spectrum_json": json.dumps(list(zip(WAVELENGTHS, charged_spectrum["curve"]))),\n            "stda_ground_transitions_json": json.dumps(ground_spectrum["transitions"]),\n            "stda_charged_transitions_json": json.dumps(charged_spectrum["transitions"]),\n            "oracle_status": "complete"}\n\n\ndef prepare_oracle_queue(rows: Iterable[dict[str, Any]], output_dir: str | Path,\n                         config: dict[str, Any]) -> dict[str, Any]:\n    destination = Path(output_dir)\n    calculations = destination / "calculations"\n    calculations.mkdir(parents=True, exist_ok=True)\n    status = availability(config)\n    rows = list(rows)\n    queue, results, errors = [], [], []\n    run = bool(config.get("oracle", {}).get("run_automatically")) and status["ready"]\n    limit = min(len(rows), int(config.get("oracle", {}).get("max_candidates", len(rows))))\n    for rank, row in enumerate(rows, start=1):\n        candidate_id = row.get("candidate_id") or stable_hash(row["smiles"], 20)\n        queue.append({"oracle_rank": rank, "candidate_id": candidate_id, "family": row["family"],\n                      "smiles": row["smiles"], "charged_smiles": row["charged_smiles"],\n                      "oracle_status": "scheduled" if run and rank <= limit else "not_run_by_configuration"})\n        if run and rank <= limit:\n            try:\n                results.append(evaluate_pair(row, calculations, config, rank))\n            except Exception as exc:\n                errors.append({"candidate_id": candidate_id, "error": f"{type(exc).__name__}:{exc}"})\n    write_csv(destination / "oracle_queue.csv", queue)\n    write_csv(destination / "oracle_results.csv", results)\n    complete = run and limit > 0 and len(results) == limit and not errors\n    manifest = {"schema_version": "2.0", "availability": status, "queued": len(queue),\n                "requested": limit if run else 0, "completed": len(results), "errors": errors,\n                "calculation_contract": {\n                    "energy": "multi-conformer RDKit -> GFN2-xTB/ASE geometry optimization; charged-ground DeltaE proxy",\n                    "spectrum": "official xtb4stda 1.0 + sTDA 1.6.1 transitions, Gaussian broadening over 290-400 nm",\n                    "inside_rl_loop": False, "automatic_proxy_substitution": False},\n                "status": "complete" if complete else ("partially_complete" if results else "not_run")}\n    (destination / "oracle_manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    return manifest\n'
load_embedded('mostgen.oracle', _source, PROJECT_ROOT / 'mostgen' / 'oracle.py')
print('loaded mostgen.oracle')


loaded mostgen.oracle


### 3.32 Шаблоны и валидация внешних лабораторных данных (без выполнения эксперимента) — `experimental.py`


In [16]:
_source = 'from __future__ import annotations\n\n"""Schemas and fail-closed validation for the external wet-lab campaign.\n\nThis module does not pretend to perform experiments.  It turns a computational\nshortlist into versioned blank result tables and validates files returned by an\nauthorized laboratory before those measurements can be joined to model data.\n"""\n\nimport csv\nimport json\nimport math\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .data import read_csv, write_csv\n\n\nSCHEMAS: dict[str, list[str]] = {\n    "candidate_registry.csv": [\n        "candidate_id", "smiles", "charged_smiles", "family", "batch_id",\n        "identity_confirmed", "nmr_confirmed", "lcms_confirmed", "purity_hplc_fraction",\n    ],\n    "solution_spectra.csv": [\n        "candidate_id", "batch_id", "replicate", "state", "solvent", "temperature_k",\n        "concentration_mol_l", "path_length_cm", "wavelength_nm", "absorbance",\n        "dark_corrected", "instrument_id", "raw_file_sha256",\n    ],\n    "photokinetics.csv": [\n        "candidate_id", "batch_id", "replicate", "irradiation_nm", "photon_flux_mol_s",\n        "initial_state", "pss_charged_fraction", "quantum_yield", "thermal_temperature_k",\n        "half_life_h", "fit_r2", "raw_file_sha256",\n    ],\n    "calorimetry.csv": [\n        "candidate_id", "batch_id", "replicate", "method", "charged_fraction",\n        "delta_h_kj_mol", "specific_energy_wh_kg", "standard_uncertainty_wh_kg",\n        "instrument_id", "raw_file_sha256",\n    ],\n    "cycling.csv": [\n        "candidate_id", "batch_id", "replicate", "cycle_count",\n        "remaining_capacity_fraction", "photoproduct_fraction", "analytical_method",\n        "raw_file_sha256",\n    ],\n    "film_spectra.csv": [\n        "formulation_id", "candidate_id", "batch_id", "replicate", "matrix",\n        "loading_wt_fraction", "thickness_um", "state", "wavelength_nm",\n        "transmittance_fraction", "instrument_id", "raw_file_sha256",\n    ],\n    "oecd_tg432.csv": [\n        "candidate_id", "batch_id", "laboratory", "study_id", "glp_status",\n        "cell_line", "replicate", "uva_dose_j_cm2", "ic50_minus_uv_mg_ml",\n        "ic50_plus_uv_mg_ml", "photo_irritation_factor", "mean_photo_effect",\n        "classification", "report_sha256",\n    ],\n}\n\n\ndef write_experimental_package(shortlist_path: str | Path, output_dir: str | Path) -> dict[str, Any]:\n    destination = Path(output_dir)\n    destination.mkdir(parents=True, exist_ok=True)\n    shortlist = read_csv(shortlist_path) if Path(shortlist_path).exists() else []\n    registry = []\n    seen = set()\n    for row in shortlist:\n        candidate_id = row.get("candidate_id")\n        if not candidate_id or candidate_id in seen:\n            continue\n        seen.add(candidate_id)\n        registry.append({\n            "candidate_id": candidate_id, "smiles": row["smiles"],\n            "charged_smiles": row["charged_smiles"], "family": row["family"],\n            "batch_id": "", "identity_confirmed": "", "nmr_confirmed": "",\n            "lcms_confirmed": "", "purity_hplc_fraction": "",\n        })\n    write_csv(destination / "candidate_registry.csv", registry, SCHEMAS["candidate_registry.csv"])\n    for filename, columns in SCHEMAS.items():\n        if filename != "candidate_registry.csv":\n            write_csv(destination / filename, [], columns)\n    manifest = {\n        "schema_version": "1.0", "candidate_count": len(registry),\n        "status": "ready_for_authorized_laboratory" if registry else "blocked_no_computational_candidates",\n        "tables": SCHEMAS,\n        "minimum_replicates": 3,\n        "claim_boundary": "Blank templates are not experimental evidence. OECD TG 432 must be run by a competent laboratory.",\n        "protocol": "docs/EXPERIMENTAL_VALIDATION.md",\n    }\n    (destination / "experimental_manifest.json").write_text(\n        json.dumps(manifest, indent=2, sort_keys=True) + "\\n", encoding="utf-8",\n    )\n    return manifest\n\n\ndef _truth(value: Any) -> bool:\n    return str(value).strip().lower() in {"true", "1", "yes", "y"}\n\n\ndef _finite(row: dict[str, str], key: str, low: float | None = None,\n            high: float | None = None) -> bool:\n    try:\n        value = float(row[key])\n    except (KeyError, TypeError, ValueError):\n        return False\n    return math.isfinite(value) and (low is None or value >= low) and (high is None or value <= high)\n\n\ndef validate_experimental_results(input_dir: str | Path) -> dict[str, Any]:\n    """Validate identity, ranges, coverage and replication; never infer blanks."""\n    source = Path(input_dir)\n    errors: list[str] = []\n    tables: dict[str, list[dict[str, str]]] = {}\n    for filename, required in SCHEMAS.items():\n        path = source / filename\n        if not path.is_file():\n            errors.append(f"missing_table:{filename}")\n            continue\n        with path.open(encoding="utf-8", newline="") as handle:\n            reader = csv.DictReader(handle)\n            missing = set(required) - set(reader.fieldnames or [])\n            if missing:\n                errors.append(f"missing_columns:{filename}:{\',\'.join(sorted(missing))}")\n            tables[filename] = list(reader)\n    registry = tables.get("candidate_registry.csv", [])\n    candidates = {row.get("candidate_id", "") for row in registry if row.get("candidate_id")}\n    for row in registry:\n        cid = row.get("candidate_id", "")\n        if not (_truth(row.get("identity_confirmed")) and _truth(row.get("nmr_confirmed"))\n                and _truth(row.get("lcms_confirmed"))):\n            errors.append(f"identity_not_confirmed:{cid}")\n        if not _finite(row, "purity_hplc_fraction", 0.95, 1.0):\n            errors.append(f"purity_below_0.95_or_invalid:{cid}")\n    for filename, rows in tables.items():\n        if filename == "candidate_registry.csv":\n            continue\n        for row in rows:\n            cid = row.get("candidate_id", "")\n            if cid not in candidates:\n                errors.append(f"unknown_candidate:{filename}:{cid}")\n    # Each curve must cover the complete requested molecular or film interval.\n    for filename, wavelength_key, group_keys in (\n        ("solution_spectra.csv", "wavelength_nm", ("candidate_id", "batch_id", "replicate", "state", "solvent")),\n        ("film_spectra.csv", "wavelength_nm", ("formulation_id", "replicate", "state")),\n    ):\n        groups: dict[tuple[str, ...], list[float]] = defaultdict(list)\n        for row in tables.get(filename, []):\n            if _finite(row, wavelength_key, 200.0, 900.0):\n                groups[tuple(row.get(key, "") for key in group_keys)].append(float(row[wavelength_key]))\n        for key, wavelengths in groups.items():\n            if min(wavelengths) > 290.0 or max(wavelengths) < 400.0:\n                errors.append(f"incomplete_290_400_coverage:{filename}:{key}")\n    # Scalar assays need three independent replicate identifiers per candidate.\n    for filename in ("photokinetics.csv", "calorimetry.csv", "cycling.csv", "oecd_tg432.csv"):\n        replicates: dict[str, set[str]] = defaultdict(set)\n        for row in tables.get(filename, []):\n            replicates[row.get("candidate_id", "")].add(row.get("replicate", ""))\n        for cid in candidates:\n            if len(replicates.get(cid, set()) - {""}) < 3:\n                errors.append(f"fewer_than_3_replicates:{filename}:{cid}")\n    result = {\n        "schema_version": "1.0", "schema_valid": not errors,\n        "experiment_complete": bool(candidates) and not errors,\n        "status": ("complete" if candidates and not errors else\n                   "blocked_no_candidates" if not candidates else "invalid"),\n        "valid": not errors, "errors": sorted(set(errors)),\n        "candidate_count": len(candidates),\n        "row_counts": {filename: len(rows) for filename, rows in tables.items()},\n    }\n    (source / "experimental_validation.json").write_text(\n        json.dumps(result, indent=2, sort_keys=True) + "\\n", encoding="utf-8",\n    )\n    return result\n'
load_embedded('mostgen.experimental', _source, PROJECT_ROOT / 'mostgen' / 'experimental.py')
print('loaded mostgen.experimental')


loaded mostgen.experimental


### 3.34 Независимая переоценка и карточки — `review.py`


In [17]:
_source = 'from __future__ import annotations\n\nimport json\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .data import read_csv, write_csv\nfrom .experimental import write_experimental_package\nfrom .oracle import prepare_oracle_queue\nfrom .reviewers import ReviewerBundle\nfrom .scoring import ScoringContext\n\n\ndef _truth(value: Any) -> bool:\n    return value is True or str(value).lower() in {"true", "1", "yes"}\n\n\ndef _balanced_top(rows: list[dict[str, str]], count: int) -> list[dict[str, str]]:\n    deduplicated: dict[str, dict[str, str]] = {}\n    for row in sorted(rows, key=lambda item: -float(item.get("reward", 0.0))):\n        deduplicated.setdefault(row["smiles"], row)\n    by_family: dict[str, list[dict[str, str]]] = defaultdict(list)\n    for row in deduplicated.values():\n        if not _truth(row.get("safety_veto")):\n            by_family[row["family"]].append(row)\n    families = sorted(by_family)\n    base, remainder = divmod(count, max(1, len(families)))\n    chosen = []\n    for index, family in enumerate(families):\n        chosen.extend(by_family[family][: base + (1 if index < remainder else 0)])\n    if len(chosen) < count:\n        used = {row["smiles"] for row in chosen}\n        extras = [row for row in sorted(deduplicated.values(), key=lambda item: -float(item.get("reward", 0.0))) if row["smiles"] not in used]\n        chosen.extend(extras[: count - len(chosen)])\n    return chosen[:count]\n\n\ndef _card(row: dict[str, Any], oracle_status: str) -> dict[str, Any]:\n    try:\n        spectrum = json.loads(row.get("spectrum_json", "[]"))\n    except json.JSONDecodeError:\n        spectrum = []\n    return {\n        "candidate": {\n            "candidate_id": row.get("candidate_id"), "smiles": row["smiles"],\n            "charged_smiles": row["charged_smiles"], "family": row["family"],\n            "method_id": row["method_id"], "seed": row["seed"],\n        },\n        "spectrum": {\n            "curve": spectrum, "uvb_auc": row.get("uvb_auc"), "uva_auc": row.get("uva_auc"),\n            "lambda_c_proxy_nm": row.get("lambda_c_nm"), "uncertainty": {\n                "uvb_auc": row.get("uvb_auc_uncertainty"), "uva_auc": row.get("uva_auc_uncertainty"),\n                "lambda_c": row.get("lambda_c_uncertainty"),\n            },\n            "conditional_film_proxy": {\n                "loading_scale": row.get("beer_lambert_loading_scale"),\n                "uvb_transmittance": row.get("uvb_transmittance"),\n                "uvb_transmittance_ucb": row.get("uvb_transmittance_ucb"),\n                "uva_transmittance": row.get("uva_transmittance"),\n                "uva_transmittance_ucb": row.get("uva_transmittance_ucb"),\n                "pass": row.get("film_proxy_pass"),\n            },\n        },\n        "most": {\n            "delta_h_kj_mol": row.get("energy_kj_mol"), "specific_energy_wh_kg": row.get("specific_energy_wh_kg"),\n            "specific_energy_lcb_wh_kg": row.get("specific_energy_lcb"),\n            "half_life_h_at_305k": row.get("half_life_h"), "uncertainty": {\n                "delta_h": row.get("energy_uncertainty"), "specific_energy": row.get("specific_energy_uncertainty"),\n                "log_half_life": row.get("half_life_uncertainty_log"),\n            },\n        },\n        "safety_triage": {\n            "kp_log_cm_s": row.get("kp_log_cm_s"), "phototoxicity_probability": row.get("phototoxicity_probability"),\n            "phototoxicity_uncertain": row.get("phototoxicity_uncertain"), "psoralen_alert": row.get("psoralen_alert"),\n            "known_phototoxic_match": row.get("known_phototoxic_match"), "reactive_alerts": row.get("reactive_alerts"),\n            "sa_score_proxy": row.get("sa_score"),\n            "sensitization_probability": row.get("sensitization_probability"),\n            "irritation_probability": row.get("irritation_probability"),\n        },\n        "applicability_domain": {\n            "spectral_similarity_D_A": row.get("similarity_D_A"), "most_similarity_D_B": row.get("similarity_D_B"),\n            "spectral_inside": row.get("ad_spectral"), "most_inside": row.get("ad_most"),\n        },\n        "decision": {\n            "uv_pass": row.get("joint_uv_pass"), "most_pass": row.get("most_pass"),\n            "safety_pass": row.get("safety_pass"), "joint_pass": row.get("joint_pass"),\n            "failure_reasons": row.get("failure_reasons"), "physical_oracle": oracle_status,\n            "selected": row.get("selected", False),\n        },\n        "evidence": {\n            "reviewer": "independent evaluator, separate from reward models",\n            "data_tier": row.get("reviewer_data_mode", "synthetic_smoke_only"),\n            "endpoint_evidence": row.get("evidence_json", "{}"),\n            "physical_oracle": oracle_status,\n            "claim_level": "research screening candidate only",\n            "not_iso_certified": True,\n            "experimental_phototoxicity_required": "OECD TG 432 or suitable successor method",\n        },\n    }\n\n\ndef review_generated(\n    config: dict[str, Any],\n    generated_path: str | Path,\n    evaluator_path: str | Path,\n    output_dir: str | Path,\n) -> dict[str, Any]:\n    generated = read_csv(generated_path)\n    top = _balanced_top(generated, int(config["execution"]["shortlist_size"]))\n    evaluators = ReviewerBundle.load(evaluator_path)\n    context = ScoringContext(config, evaluators)\n    reviewed = []\n    for row in top:\n        evaluated = context.review_candidate(row, row["method_id"], int(row["seed"]))\n        evaluated["reward_model_reward"] = float(row.get("reward", 0.0))\n        evaluated["preselection_rank"] = len(reviewed) + 1\n        reviewed.append(evaluated)\n    destination = Path(output_dir)\n    destination.mkdir(parents=True, exist_ok=True)\n    oracle_manifest = prepare_oracle_queue(reviewed, destination / "physical_oracle", config)\n    oracle_complete = oracle_manifest["status"] == "complete"\n    oracle_results_path = destination / "physical_oracle" / "oracle_results.csv"\n    oracle_by_id = {row["candidate_id"]: row for row in read_csv(oracle_results_path)} if oracle_results_path.exists() else {}\n    for row in reviewed:\n        physical = oracle_by_id.get(row.get("candidate_id"), {})\n        row.update(physical)\n        row["physical_oracle_pass"] = bool(\n            physical\n            and float(physical.get("gfn2_delta_e_kj_mol", 0.0)) > 0.0\n            and float(physical.get("gfn2_specific_energy_wh_kg", 0.0)) >= float(config["reviewers"].get("specific_energy_min_wh_kg", 0.0))\n            and float(physical.get("stda_lambda_c_nm", 0.0)) >= float(config["reviewers"]["lambda_c_min_nm"])\n            and float(physical.get("stda_uvb_transmittance", 1.0)) <= float(config["reviewers"].get("uvb_transmittance_max", 1.0))\n            and float(physical.get("stda_uva_transmittance", 1.0)) <= float(config["reviewers"].get("uva_transmittance_max", 1.0))\n        )\n    provisional = [\n        row for row in reviewed\n        if _truth(row.get("joint_pass"))\n        and not _truth(row.get("phototoxicity_uncertain"))\n        and not _truth(row.get("psoralen_alert"))\n        and not _truth(row.get("known_phototoxic_match"))\n    ]\n    selected = [row for row in provisional if _truth(row.get("physical_oracle_pass"))] if oracle_complete else []\n    selected_keys = {(row["smiles"], row["method_id"], str(row["seed"])) for row in selected}\n    for row in reviewed:\n        row["physical_oracle_status"] = oracle_manifest["status"]\n        row["selected"] = (row["smiles"], row["method_id"], str(row["seed"])) in selected_keys\n        row["selection_status"] = "selected_research_candidate" if row["selected"] else (\n            "provisional_pending_physical_oracle" if row in provisional else "failed_independent_review"\n        )\n    for row in generated:\n        row["selected"] = (row["smiles"], row["method_id"], str(row["seed"])) in selected_keys\n    write_csv(generated_path, generated)\n    write_csv(destination / "reviewed_top.csv", reviewed)\n    write_csv(destination / "provisional_shortlist.csv", provisional, fieldnames=list(reviewed[0]) if reviewed else [])\n    write_csv(destination / "shortlist.csv", selected, fieldnames=list(reviewed[0]) if reviewed else [])\n    experimental_package = write_experimental_package(\n        destination / "shortlist.csv", destination / "experimental_package",\n    )\n    cards_dir = destination / "cards"\n    cards_dir.mkdir(parents=True, exist_ok=True)\n    for row in reviewed:\n        card = _card(row, oracle_manifest["status"])\n        (cards_dir / f"{row[\'candidate_id\']}.json").write_text(json.dumps(card, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    summary = {\n        "reviewed": len(reviewed), "independent_joint_pass": len(provisional),\n        "selected_after_physical_oracle": len(selected), "physical_oracle": oracle_manifest,\n        "experimental_package": experimental_package,\n        "selection_policy": "Fail closed: no final selection until independent physical oracle results are complete.",\n    }\n    (destination / "review_summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    return summary\n'
load_embedded('mostgen.review', _source, PROJECT_ROOT / 'mostgen' / 'review.py')
print('loaded mostgen.review')


loaded mostgen.review


### 3.36 Отчёт и презентация — `reporting.py`


In [18]:
_source = 'from __future__ import annotations\n\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .data import read_csv\n\n\ndef _pct(value: Any) -> str:\n    try:\n        return f"{100.0 * float(value):.2f}%"\n    except (TypeError, ValueError):\n        return "n/a"\n\n\ndef build_reports(experiment_dir: str | Path, config: dict[str, Any]) -> dict[str, str]:\n    root = Path(experiment_dir)\n    output = root / "report"\n    output.mkdir(parents=True, exist_ok=True)\n    metrics_path = root / "metrics" / "metrics_summary.csv"\n    metrics = read_csv(metrics_path) if metrics_path.exists() else []\n    generated = read_csv(root / "generated.csv")\n    review = json.loads((root / "review" / "review_summary.json").read_text(encoding="utf-8"))\n    table = ["| Метод | Валидность | Уникальность | Diversity | Joint success | Обе AD |", "|---|---:|---:|---:|---:|---:|"]\n    for row in metrics:\n        table.append(\n            f"| {row[\'method_id\']} | {_pct(row[\'validity_mean\'])} | {_pct(row[\'uniqueness_mean\'])} | "\n            f"{float(row[\'internal_diversity_mean\']):.3f} | {_pct(row[\'joint_success_mean\'])} | {_pct(row[\'both_ad_fraction_mean\'])} |"\n        )\n    family_counts = {family: sum(row["family"] == family for row in generated) for family in config["families"]}\n    no_joint = sum(str(row.get("joint_pass", "")).lower() == "true" for row in generated) == 0\n    real_mode = bool(generated) and generated[0].get("reviewer_data_mode") == "real_endpoint_specific"\n    data_manifest_path = root / "data" / "data_manifest.json"\n    data_manifest = json.loads(data_manifest_path.read_text(encoding="utf-8")) if data_manifest_path.exists() else {}\n    generator_description = (\n        "настоящий REINVENT4/LibInvent staged-learning с family-specific TL agents и post-sampling"\n        if real_mode else "детерминированный reaction-library smoke-surrogate"\n    )\n    evidence_description = (\n        "Обучение выполнено на реальных endpoint-записях M13 (включая M11 Deep4Chem и M12 CDEx), M01, U07, U09, U12, U13 и U16; M05 используется как малая full-spectrum проверка. "\n        "Начальный Excel используется только как каталог источников/provenance. Таблица молекулярных ΔH недоступна, поэтому energy-модель не обучена и MOST/joint gate закрыт до GFN2-xTB/эксперимента."\n        if real_mode else\n        "Это smoke-проверка на метках `synthetic_smoke_only`; она тестирует программную механику, но не является научным обучением."\n    )\n    report = f"""# Воспроизводимый скрининг UV-поглощающих MOST-фотопереключателей\n\n## 1. Цель, область утверждений и вычислительный бюджет\n\nЦель прототипа — найти вычислительное пересечение широкополосного поглощения 290–400 нм, молекулярного накопления энергии, времени хранения 4–24 ч при 305 K и консервативного safety-triage. Проектируется одна малая фотопереключаемая молекула, не смесь и не готовая солнцезащитная формуляция. Результаты имеют статус **screening proxy**. Они не подтверждают безопасность, эффективность, фотостабильность или пригодность вещества как косметического ингредиента.\n\nISO 24444:2019 описывает in vivo определение SPF готового продукта, а ISO 24443:2021 — in vitro оценку UVA-защиты продукта. Поэтому ни одна молекула здесь не называется соответствующей ISO. Следующий уровень доказательности требует изготовления плёнки/формуляции и испытания продукта. Фототоксичность требует отдельной экспериментальной проверки, например OECD TG 432.\n\nЗапуск: режим `{config[\'execution\'][\'mode\']}`, backend `{config[\'execution\'][\'backend\']}`, seeds `{config[\'execution\'][\'seeds\']}`. Лимиты: не более {config[\'project\'][\'max_training_structures\']} обучающих структур и {config[\'project\'][\'max_gpu_hours\']:.1f} GPU·ч; фактическая конфигурация заявляет {config[\'execution\'][\'gpu_hours\']:.1f} GPU·ч. Для matched comparison каждый метод получает ровно {config[\'execution\'][\'reviewer_budget_per_run\']} reward-reviewer evaluations на seed и сохраняет одинаковые {config[\'execution\'][\'n_per_run\']} структур. Для LibInvent бюджет состоит из curriculum optimization и финального scoring; фактические вызовы всех методов записаны в `search_budget_ledger.json`.\n\n## 2. Генеративное пространство и сравниваемые методы\n\nТри семейства запускались как family-aware пространства с фиксированными реакционными каркасами и двумя разрешёнными позициями замещения: NBD/QC как основной физически определённый класс, Dewar-pyrimidinone как малодатовый экспериментальный класс и spiropyran/merocyanine как проверка переносимости. Заряженный изомер строился детерминированно; RDKit проверял валентность, санитаризацию и равенство молекулярной формулы пары. Размеры фактически оценённых подмножеств: `{family_counts}`.\n\nСравнены (i) случайный reaction-constrained prior, (ii) adaptive weighted-retraining по той же библиотеке синтонов и (iii) {generator_description}. В реальном режиме neural output принимается только при точном совпадении с версионированной библиотекой допустимых продуктов; enumerative filler не используется.\n\nCurriculum последовательно открывает валидность/семейство/пару, спектр, MOST, затем безопасность/SA/AD. Награда — взвешенное геометрическое среднее непрерывных sigmoid/interval-компонентов с floor `{config[\'reward\'][\'floor\']}`. ECFP-centroid filter штрафует повторное заселение кластеров. Таблица `reward_diagnostics.csv` позволяет проверить долю ненулевых наград, effective sample size, баланс компонентов и scaffold collapse.\n\n## 3. Reviewer-модели, неопределённость и применимость\n\nPipeline сохраняет exact-Murcko scaffold split до обучения и проверяет отсутствие пересечения scaffold между train/validation/test. Reward-модели и независимые evaluator-модели сериализованы раздельно: первые используют ensemble Random Forest и направляют поиск, вторые — отдельный ensemble Extra Trees и переоценивают только top-кандидатов. В real mode λmax превращается в явно маркированный Gaussian-band proxy; он не выдаётся за измеренный полный спектр. Из proxy интегрируются UVB AUC, UVA AUC, critical-wavelength proxy и условная Beer–Lambert transmittance.\n\nJoint UV-pass требует, чтобы нижняя 90%-я доверительная граница обоих AUC была не хуже медианы, нижняя граница λc-proxy была не ниже {config[\'reviewers\'][\'lambda_c_min_nm\']:.0f} нм, а worst-side conditional Beer–Lambert transmittance не превышала заданных UVB/UVA порогов. MOST-pass требует положительную нижнюю границу ΔH, результат не хуже семейной медианы, specific-energy LCB ≥{config[\'reviewers\'][\'specific_energy_min_wh_kg\']:.0f} Wh·kg⁻¹ и средний t½ в окне {config[\'reviewers\'][\'half_life_window_hours\'][0]:.0f}–{config[\'reviewers\'][\'half_life_window_hours\'][1]:.0f} ч. Высокий прогноз вне fingerprint-domain не засчитывается.\n\n{evidence_description}\n\nЧисло endpoint-записей: `{data_manifest.get(\'endpoint_rows\', \'см. data_manifest.json\')}`. M03 не был использован: API Dryad возвращает HTTP 401 без bearer token; нулевой partial-файл исключён. M04 сохранён как физический reference, но не превращён в labels без надёжной идентификации структур.\n\n## 4. Safety gate и правила shortlist\n\nДо вычисления награды применяются SMARTS-поиск линейных и угловых фурокумариновых/псораленовых ядер, точное совпадение с локальным списком известных фототоксичных структур, запрет неподдерживаемых элементов, reactive/unstable alerts, проверка валентности и построения изомерной пары. Псоралены и иные фурокумарины не могут пройти в итоговый CSV. Это консервативно согласуется с заключением SCCP: безопасность фурокумаринов не была подтверждена, а фототоксичность нельзя считать исключённой.\n\nВероятность фототоксичности и Kp — лишь triage. Интервал неопределённости фототоксичности блокирует продвижение независимо от среднего значения. Отсутствие алерта не является доказательством безопасности. В карточке каждого top-кандидата раздельно показаны spectrum/MOST/safety, uncertainty, обе AD, SA-proxy, причины отказа и уровень доказательности.\n\n## 5. Результаты и matched-budget comparison\n\n{chr(10).join(table)}\n\nСгенерировано {len(generated)} строк; независимый evaluator пересмотрел {review[\'reviewed\']}, provisional joint-pass получили {review[\'independent_joint_pass\']}. После физического oracle выбрано {review[\'selected_after_physical_oracle\']}. Статус oracle: `{review[\'physical_oracle\'][\'status\']}`. При отсутствии xTB/sTDA pipeline не подставляет суррогатный «квантовый» результат и оставляет `selected=false` — это намеренное fail-closed поведение.\n\nВ production-конфигурации bootstrap-интервалы агрегируют три seed-запуска; короткий audit с одним seed не оценивает межзапусковую устойчивость. `ablations.csv` показывает эффекты снятия uncertainty, safety, AD и diversity gates; сравнение curriculum с prior следует читать как matched-set методическое сравнение, а не как доказательство превосходства на реальных данных.\n\n## 6. Вывод, ограничения и следующий эксперимент\n\n{\'В этом запуске не найдено кандидатов, прошедших исходный joint gate. Это допустимый научный результат: таблицы причин отказа показывают конфликт требований и неопределённость; он не доказывает физическую невозможность.\' if no_joint else \'Некоторые структуры прошли внутренний reward gate, но они остаются вычислительными гипотезами и требуют независимого физического и экспериментального подтверждения.\'}\n\nСледующий этап: получить надёжно идентифицированные молекулярные ΔH/ΔG‡/t½ записи M03/M04 и дополнительные U07–U17 endpoint-данные; затем выполнить GFN2-xTB conformer review и sTDA-xTB broadened spectrum для 50–100 top-кандидатов. После синтеза нужны проверка идентичности обоих изомеров, циклируемость, quantum yield, растворитель/плёнка, OECD TG 432 и тестирование готовой формуляции по применимым стандартам.\n\n## Приложение A. Протокол воспроизводимости и аудит данных\n\nЕдиницей эксперимента считается тройка `method_id × seed × resolved configuration`. Для неё фиксируются версия Python, путь интерпретатора, версии RDKit/NumPy/Pandas/scikit-learn, платформа, команда запуска, абсолютные пути исходных файлов, SHA-256 каждого локального входа и полный registry внешних источников. URL без локально зафиксированного артефакта получают явный checksum-статус `NOT_FETCHED_NO_LOCAL_ARTIFACT`, а не фиктивный хеш. После завершения создаётся `artifact_manifest.json` с размером и SHA-256 каждого результата. Это позволяет отличить научно значимое изменение конфигурации от случайной подмены входного файла.\n\nИсходные XLSX/DOCX/PPTX не копируются поверх и не редактируются. В real mode производная таблица `reviewer_endpoints.csv` содержит source_id, endpoint, исходные условия, состояние и label-level; `full_spectra.csv` хранится отдельно. Начальный `database_matrix_MOST_UV_skin.xlsx` распарсен в `source_catalog_from_initial_excel.csv`, но не является таблицей SMILES/labels и не входит в fit. `reaction_library.csv` — отдельное дискретное пространство генерации.\n\nScaffold split рассчитывается до fit. Один и тот же generic Murcko scaffold не может появиться в разных split, а loader повторно проверяет пересечение и останавливает обучение при leakage. Validation служит выбору/калибровке, test — финальной диагностике. ITI обозначен только как validation-only class shift и не включён в три генеративных семейства. Малые M03/M04 должны использоваться прежде всего для физической проверки и оценки переноса; литературные M08–M10 не трактуются как доступные виртуальные библиотеки. Регуляторные U01/U03/U06 остаются справочниками, а не автоматически размеченными обучающими строками.\n\n## Приложение B. Как читать метрики и отрицательный результат\n\nValidity отвечает только за машинно проверяемую химическую корректность и не равна устойчивости соединения. Uniqueness считается по каноническому SMILES внутри запуска. Novelty сравнивает результат с фактическим training set, но не с мировой химической литературой. Internal diversity — среднее расстояние Morgan fingerprints на детерминированной выборке пар; scaffold diversity — доля generic Murcko scaffold. SA-score в этом прототипе — прозрачный complexity proxy, не оценка маршрута синтеза. Поэтому высокая novelty или низкий SA не означают коммерческую доступность продукта реакции.\n\nJoint success — наиболее строгая метрика: она требует одновременно UV-pass, MOST-pass, safety-pass и обе AD. Доверительные границы здесь важнее средних значений. Например, молекула с высоким средним UVA AUC отвергается, если ensemble расходится настолько, что нижняя граница не достигает семейного референса. Аналогично положительное среднее ΔH не помогает за пределами family-specific MOST domain. В phototoxicity применяется ещё более консервативное правило: неопределённая область сама по себе блокирует shortlist. Такой порядок уменьшает число эффектных, но неподтверждаемых виртуальных лидов.\n\nBootstrap выполняется по трём независимым seed-результатам, а не по 3000 молекулам как будто они независимые эксперименты. Поэтому интервал отражает вариабельность запуска метода, хотя при трёх seeds он остаётся ориентировочным. `metrics_runs.csv` сохраняет исходные значения, а `metrics_summary.csv` — среднее и percentile interval. Для сравнения методов нужно одновременно смотреть joint success и diversity: метод с немного большим success, но с collapse к одному кластеру, не обязательно предпочтительнее. `reward_diagnostics.csv` показывает effective sample size и максимальную долю одного ECFP-кластера; крайне малый ESS сигнализирует, что несколько молекул доминируют в обучающем сигнале.\n\nАбляция uncertainty показывает, сколько кандидатов появилось бы при использовании только средних прогнозов. Абляция safety или AD не является альтернативным допустимым shortlist — это диагностическая карта конфликта требований. Абляция diversity сравнивает кластерное покрытие top-100 при ранжировании исходной и штрафованной наградой. Curriculum оценивается относительно двух matched-budget baseline, но причинный вывод о пользе curriculum требует повторения на реальных моделях и данных. Если joint-кандидатов нет, следует изучить распределения отдельных gate и failure reasons, а не ослаблять пороги постфактум.\n\n## Приложение C. Переход от прототипа к физическому исследованию\n\n`train-generator` создаёт отдельный REINVENT4 v{config[\'production\'][\'reinvent4_version\']} TOML для каждого `family × seed`, scaffold-файл с двумя attachment points и ExternalProcess bridge к reward-reviewers. Конфигурация использует family TL, staged learning, DAP, четыре curriculum stages и PenalizeSameSmiles. Production preflight требует установленный `reinvent`, существующий LibInvent prior с совпадающим SHA-256 и полный xTB/sTDA toolchain. Отсутствие обязательного артефакта завершает запуск ошибкой до выдачи научного результата.\n\nДля physical oracle RDKit создаёт восемь конформеров обоих состояний, выполняет MMFF/UFF pre-relaxation и сохраняет лучший XYZ. Затем GFN2-xTB должен независимо оптимизировать обе структуры; разность электронных энергий не подменяет свободную энергию, но служит внешней проверкой знака и масштаба storage proxy. sTDA-xTB даёт переходы, которые необходимо уширить с явно записанной шириной линии и повторно интегрировать по UVB/UVA. Расчёты не возвращаются во внутренний RL loop, поэтому сохраняется независимость внешней проверки.\n\nДаже успешный physical oracle не делает молекулу безопасным ингредиентом. До формуляции нужны синтез и аналитическое подтверждение структуры, выделение обоих состояний, измерение спектра и quantum yield, проверка обратимости/усталости, t½ при нескольких температурах и оценка побочных фотопродуктов. Safety-пакет должен включать растворимость, проникновение через кожу, раздражение, сенсибилизацию и экспериментальную фототоксичность. Только после этого имеет смысл изготовить воспроизводимую плёнку с контролируемой загрузкой и сравнивать готовый продукт применимыми методами ISO. Молекулярный расчёт остаётся способом приоритизации, а не заменой этой цепочки доказательств.\n\nИсточники: [REINVENT4](https://github.com/MolecularAI/REINVENT4), [SCCP opinion](https://ec.europa.eu/health/ph_risk/committees/04_sccp/docs/sccp_o_036.pdf), [ISO 24444](https://www.iso.org/standard/72250.html), [ISO 24443](https://www.iso.org/standard/75059.html), [OECD TG 432](https://www.oecd.org/en/publications/2019/06/test-no-432-in-vitro-3t3-nru-phototoxicity-test_g1gh4b69.html).\n"""\n    report_path = output / "report.md"\n    report_path.write_text(report, encoding="utf-8")\n    slides = f"""# 7-минутная презентация\n\n## Слайд 1 — Задача и честная граница утверждений (0:00–0:45)\n\n- Одна UV-поглощающая MOST-молекула, не готовая формуляция.\n- Пересечение UVB/UVA, энергии, 4–24 ч и safety triage.\n- Только screening proxy; не ISO-сертификация и не доказательство безопасности.\n\n## Слайд 2 — Три family-aware пространства (0:45–1:35)\n\n- NBD/QC: основной класс; Dewar-pyrimidinone: low-data; spiropyran: transferability.\n- Reaction-constrained синтоны и фиксированные позиции.\n- RDKit: валентность, формула изомерной пары, canonical identity.\n\n## Слайд 3 — Matched-budget дизайн (1:35–2:25)\n\n- Prior random vs weighted retraining vs {generator_description}.\n- {config[\'execution\'][\'reviewer_budget_per_run\']} reward-reviewer evaluations и {config[\'execution\'][\'n_per_run\']} сохранённых кандидатов на method × seed.\n- REINVENT4 manifests отделены от CPU smoke backend.\n\n## Слайд 4 — Reward и reviewer independence (2:25–3:25)\n\n- Spectrum 290–400 нм → UVB/UVA AUC, λc, Beer–Lambert proxy.\n- λmax/kinetics/Kp/phototoxicity/sensitization models; ΔH intentionally missing.\n- Random Forest reward ≠ Extra Trees evaluator; scaffold split + AD + uncertainty.\n\n## Слайд 5 — Жёсткая безопасность (3:25–4:20)\n\n- Veto псораленов/фурокумаринов до reward.\n- Known phototoxic/reactive/unsupported/invalid pair — немедленный отказ.\n- Uncertain phototoxicity никогда не проходит в shortlist.\n\n## Слайд 6 — Результаты (4:20–5:40)\n\n{chr(10).join(table)}\n\n- Generated: {len(generated)}; independent reviewed: {review[\'reviewed\']}.\n- Provisional: {review[\'independent_joint_pass\']}; final after oracle: {review[\'selected_after_physical_oracle\']}.\n- Oracle status: `{review[\'physical_oracle\'][\'status\']}`.\n\n## Слайд 7 — Решение и следующий шаг (5:40–7:00)\n\n- Отсутствие joint-pass — информативный результат, не доказательство невозможности.\n- Добавить идентифицированные ΔH/ΔG‡ labels и расширить малые safety/kinetics endpoints.\n- GFN2/sTDA-xTB top-50–100 → синтез → OECD TG 432 → испытание плёнки/формуляции.\n"""\n    slides_path = output / "presentation_7min.md"\n    slides_path.write_text(slides, encoding="utf-8")\n    return {"report": str(report_path.resolve()), "presentation": str(slides_path.resolve())}\n'
load_embedded('mostgen.reporting', _source, PROJECT_ROOT / 'mostgen' / 'reporting.py')
print('loaded mostgen.reporting')


loaded mostgen.reporting


### 3.38 Машинные критерии — `validation.py`


In [19]:
_source = 'from __future__ import annotations\n\nimport json\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .data import read_csv\n\n\ndef _truth(value: Any) -> bool:\n    return value is True or str(value).lower() in {"true", "1", "yes"}\n\n\ndef verify_experiment(root: str | Path, config: dict[str, Any]) -> dict[str, Any]:\n    experiment = Path(root)\n    rows = read_csv(experiment / "generated.csv")\n    grouped: dict[tuple[str, str], list[dict[str, str]]] = defaultdict(list)\n    for row in rows:\n        grouped[(row["method_id"], row["seed"])].append(row)\n    expected_runs = {\n        (method, str(seed))\n        for method in config["execution"]["methods"]\n        for seed in config["execution"]["seeds"]\n    }\n    minimum = int(config["execution"]["n_per_run"])\n    ledger_path = experiment / "search_budget_ledger.json"\n    ledger = json.loads(ledger_path.read_text(encoding="utf-8")) if ledger_path.exists() else {"runs": {}}\n    ledger_runs = ledger.get("runs", {})\n    expected_ledger_keys = {f"{method}:{seed}" for method, seed in expected_runs}\n    required_columns = {\n        "smiles", "charged_smiles", "family", "method_id", "seed", "uvb_auc", "uva_auc",\n        "lambda_c_nm", "energy_kj_mol", "specific_energy_wh_kg", "half_life_h",\n        "specific_energy_lcb", "uvb_transmittance_ucb", "uva_transmittance_ucb", "film_proxy_pass",\n        "phototoxicity_probability", "phototoxicity_uncertainty", "similarity_D_A", "similarity_D_B",\n        "ad_spectral", "ad_most", "sa_score", "psoralen_alert", "joint_pass", "selected",\n        "not_iso_certified",\n    }\n    checks = {\n        "all_expected_runs_present": set(grouped) == expected_runs,\n        "minimum_unique_per_run": all(len({row["smiles"] for row in members}) >= minimum for members in grouped.values()),\n        "exact_output_count_per_run": all(len(members) == minimum for members in grouped.values()),\n        "exact_matched_reviewer_budget": set(ledger_runs) == expected_ledger_keys and all(\n            int(item.get("candidate_reward_evaluations", -1)) == int(config["execution"]["reviewer_budget_per_run"])\n            for item in ledger_runs.values()\n        ),\n        "all_three_families": set(row["family"] for row in rows) == set(config["families"]),\n        "zero_psoralen_cores": not any(_truth(row.get("psoralen_alert")) for row in rows),\n        "zero_known_phototoxic_matches": not any(_truth(row.get("known_phototoxic_match")) for row in rows),\n        "no_uncertain_phototoxicity_selected": not any(_truth(row.get("selected")) and _truth(row.get("phototoxicity_uncertain")) for row in rows),\n        "no_iso_claims": all(_truth(row.get("not_iso_certified")) for row in rows),\n        "required_generated_schema": bool(rows) and required_columns <= set(rows[0]),\n        "reward_evaluator_separation": (experiment / "models" / "reward" / "reviewers.pkl").resolve() != (experiment / "models" / "evaluator" / "reviewers.pkl").resolve(),\n        "metrics_present": all((experiment / "metrics" / name).is_file() for name in ("metrics_runs.csv", "metrics_summary.csv", "ablations.csv", "reward_diagnostics.csv")),\n        "report_present": (experiment / "report" / "report.md").is_file(),\n        "presentation_present": (experiment / "report" / "presentation_7min.md").is_file(),\n        "gpu_budget_respected": float(config["execution"]["gpu_hours"]) <= float(config["project"]["max_gpu_hours"]),\n    }\n    verification = {\n        "schema_version": "1.0",\n        "passed": all(checks.values()),\n        "checks": checks,\n        "generated_rows": len(rows),\n        "run_counts": {f"{method}:{seed}": len(members) for (method, seed), members in sorted(grouped.items())},\n        "run_unique_counts": {f"{method}:{seed}": len({row["smiles"] for row in members}) for (method, seed), members in sorted(grouped.items())},\n        "family_counts": dict(sorted(Counter(row["family"] for row in rows).items())),\n        "scientific_exceptions": {\n            "no_joint_candidates_is_allowed": True,\n            "physical_oracle_unavailable_blocks_final_selection": True,\n        },\n    }\n    (experiment / "verification.json").write_text(json.dumps(verification, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    if not verification["passed"]:\n        failed = [name for name, passed in checks.items() if not passed]\n        raise RuntimeError(f"Experiment acceptance verification failed: {failed}")\n    return verification\n'
load_embedded('mostgen.validation', _source, PROJECT_ROOT / 'mostgen' / 'validation.py')
print('loaded mostgen.validation')


loaded mostgen.validation


### 3.40 Единый CLI — `cli.py`


In [20]:
_source = 'from __future__ import annotations\n\nimport argparse\nimport json\nimport shutil\nimport sys\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .config import apply_override, dump_resolved_config, load_config\nfrom .data import prepare_data, read_csv, write_csv\nfrom .metrics import compute_metrics\nfrom .oracle import availability\nfrom .provenance import append_event, sha256_file, write_artifact_manifest, write_manifest\nfrom .reporting import build_reports\nfrom .review import review_generated\nfrom .reviewers import train_reviewers\nfrom .search import execute_generator_training, run_methods, run_reinvent_generation, write_generator_manifests\nfrom .validation import verify_experiment\n\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef _config(args: argparse.Namespace) -> dict[str, Any]:\n    return apply_override(load_config(args.config, args.mode), args.override)\n\n\ndef _layout(output: str | Path) -> dict[str, Path]:\n    root = Path(output).resolve()\n    return {\n        "root": root, "data": root / "data", "models": root / "models",\n        "generator": root / "generator", "generated": root / "generated.csv",\n        "metrics": root / "metrics", "review": root / "review",\n    }\n\n\ndef _initialize(config: dict[str, Any], paths: dict[str, Path]) -> None:\n    paths["root"].mkdir(parents=True, exist_ok=True)\n    dump_resolved_config(config, paths["root"] / "config.resolved.json")\n    inputs = [config["_config_path"], ROOT / "pyproject.toml", ROOT / "requirements.txt"]\n    inputs.extend(sorted((ROOT / "mostgen").glob("*.py")))\n    inputs.append(ROOT / "scripts" / "reinvent_external_score.py")\n    inputs.extend(ROOT / name for name in ("database_matrix_MOST_UV_skin.xlsx", "Задание.docx", "Солнцезащитная плёнка с молекулярным накоплением солнечной энергии.pptx"))\n    write_manifest(paths["root"] / "experiment_manifest.json", config, inputs)\n\n\ndef _require(path: Path, hint: str) -> None:\n    if not path.exists():\n        raise FileNotFoundError(f"Missing {path}; run `{hint}` first")\n\n\ndef _production_checks(config: dict[str, Any]) -> None:\n    production = config["production"]\n    errors = []\n    if production["reinvent4_revision"].startswith("PIN_REQUIRED"):\n        errors.append("REINVENT4 revision is not pinned")\n    prior = Path(production["reaction_prior_path"]) if production["reaction_prior_path"] else None\n    if prior is not None and not prior.is_absolute():\n        prior = ROOT / prior\n    if prior is None or not prior.is_file():\n        errors.append("reaction prior path is not a file")\n    elif not production.get("reaction_prior_sha256"):\n        errors.append("reaction prior SHA-256 is not configured")\n    elif sha256_file(prior) != production["reaction_prior_sha256"]:\n        errors.append("reaction prior SHA-256 mismatch")\n    reinvent_value = production.get("reinvent_executable", "reinvent")\n    reinvent_path = Path(reinvent_value)\n    if not reinvent_path.is_absolute():\n        reinvent_path = ROOT / reinvent_path\n    if not (reinvent_path.is_file() or shutil.which(reinvent_value)):\n        errors.append("REINVENT4 `reinvent` executable is unavailable")\n    if production.get("require_physical_oracle") and not availability(config)["ready"]:\n        errors.append("xTB/sTDA toolchain is unavailable")\n    if errors:\n        raise RuntimeError("Production preflight failed: " + "; ".join(errors))\n\n\ndef command_prepare(args: argparse.Namespace) -> dict[str, Any]:\n    config, paths = _config(args), _layout(args.output)\n    _initialize(config, paths)\n    result = prepare_data(config, paths["data"], ROOT)\n    append_event(paths["root"] / "events.jsonl", "prepare-data", result)\n    return result\n\n\ndef command_train_reviewers(args: argparse.Namespace) -> dict[str, Any]:\n    config, paths = _config(args), _layout(args.output)\n    _initialize(config, paths)\n    training = paths["data"] / ("reviewer_training.csv" if config["execution"]["mode"] == "smoke" else "reviewer_endpoints.csv")\n    _require(training, "prepare-data")\n    result = train_reviewers(config, training, paths["models"])\n    append_event(paths["root"] / "events.jsonl", "train-reviewers", {"rows": result["rows"]})\n    return result\n\n\ndef command_train_generator(args: argparse.Namespace) -> dict[str, Any]:\n    config, paths = _config(args), _layout(args.output)\n    _initialize(config, paths)\n    if config["execution"]["mode"] == "production":\n        _production_checks(config)\n    result = write_generator_manifests(config, paths["generator"])\n    if config["execution"]["mode"] != "smoke":\n        result["training"] = execute_generator_training(config, paths["generator"])\n    append_event(paths["root"] / "events.jsonl", "train-generator", result)\n    return result\n\n\ndef command_search(args: argparse.Namespace, methods: list[str]) -> dict[str, Any]:\n    config, paths = _config(args), _layout(args.output)\n    _initialize(config, paths)\n    _require(paths["data"] / "reaction_library.csv", "prepare-data")\n    _require(paths["models"] / "reward" / "reviewers.pkl", "train-reviewers")\n    output_name = "generated.csv" if args.command in {"generate", "run-all"} else "baselines.csv"\n    if methods == ["libinvent_rl"] and config["execution"]["mode"] != "smoke":\n        _production_checks(config)\n        for family in config["families"]:\n            _require(paths["generator"] / f"libinvent_{family}.agent", "train-generator")\n        result = run_reinvent_generation(\n            config, paths["generator"], paths["data"] / "reaction_library.csv",\n            paths["models"] / "reward" / "reviewers.pkl", paths["root"] / output_name,\n        )\n    else:\n        result = run_methods(\n            config, paths["data"] / "reaction_library.csv", paths["models"] / "reward" / "reviewers.pkl",\n            paths["root"] / output_name, methods,\n        )\n    append_event(paths["root"] / "events.jsonl", args.command, result)\n    return result\n\n\ndef command_review(args: argparse.Namespace) -> dict[str, Any]:\n    config, paths = _config(args), _layout(args.output)\n    _initialize(config, paths)\n    _require(paths["generated"], "generate")\n    _require(paths["models"] / "evaluator" / "reviewers.pkl", "train-reviewers")\n    result = review_generated(config, paths["generated"], paths["models"] / "evaluator" / "reviewers.pkl", paths["review"])\n    append_event(paths["root"] / "events.jsonl", "review", {"reviewed": result["reviewed"], "selected": result["selected_after_physical_oracle"]})\n    return result\n\n\ndef command_run_all(args: argparse.Namespace) -> dict[str, Any]:\n    config, paths = _config(args), _layout(args.output)\n    _initialize(config, paths)\n    data_result = prepare_data(config, paths["data"], ROOT)\n    append_event(paths["root"] / "events.jsonl", "prepare-data", data_result)\n    training_path = paths["data"] / ("reviewer_training.csv" if config["execution"]["mode"] == "smoke" else "reviewer_endpoints.csv")\n    model_result = train_reviewers(config, training_path, paths["models"])\n    append_event(paths["root"] / "events.jsonl", "train-reviewers", {"rows": model_result["rows"]})\n    if config["execution"]["mode"] != "smoke":\n        _production_checks(config)\n    generator_result = write_generator_manifests(config, paths["generator"])\n    if config["execution"]["mode"] != "smoke":\n        generator_result["training"] = execute_generator_training(config, paths["generator"])\n    if config["execution"]["mode"] == "smoke":\n        search_result = run_methods(\n            config, paths["data"] / "reaction_library.csv", paths["models"] / "reward" / "reviewers.pkl",\n            paths["generated"], list(config["execution"]["methods"]),\n        )\n    else:\n        baseline_path = paths["root"] / "baselines.csv"\n        baseline_result = run_methods(\n            config, paths["data"] / "reaction_library.csv", paths["models"] / "reward" / "reviewers.pkl",\n            baseline_path, ["prior_random", "weighted_retraining"],\n        )\n        rl_path = paths["root"] / "libinvent_generated.csv"\n        rl_result = run_reinvent_generation(\n            config, paths["generator"], paths["data"] / "reaction_library.csv",\n            paths["models"] / "reward" / "reviewers.pkl", rl_path,\n        )\n        combined = read_csv(baseline_path) + read_csv(rl_path)\n        write_csv(paths["generated"], combined)\n        search_result = {\n            "path": str(paths["generated"]), "rows": len(combined),\n            "run_counts": {**baseline_result["run_counts"], **rl_result["run_counts"]},\n            "methods": list(config["execution"]["methods"]),\n            "seeds": list(config["execution"]["seeds"]),\n            "libinvent_backend": rl_result["backend"],\n        }\n    metrics_result = compute_metrics(paths["generated"], training_path, paths["metrics"], config)\n    review_result = review_generated(config, paths["generated"], paths["models"] / "evaluator" / "reviewers.pkl", paths["review"])\n    reports = build_reports(paths["root"], config)\n    verification = verify_experiment(paths["root"], config)\n    result = {\n        "experiment": str(paths["root"]), "data_rows": data_result["training_rows"],\n        "generated_rows": search_result["rows"], "run_counts": search_result["run_counts"],\n        "matched_budget": metrics_result["matched_reviewer_budget"],\n        "independent_reviewed": review_result["reviewed"],\n        "selected": review_result["selected_after_physical_oracle"], "reports": reports,\n        "generator_status": generator_result["status"],\n        "acceptance_verified": verification["passed"],\n    }\n    append_event(paths["root"] / "events.jsonl", "run-all-complete", result)\n    (paths["root"] / "run_summary.json").write_text(json.dumps(result, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    write_artifact_manifest(paths["root"])\n    return result\n\n\ndef build_parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser(prog="mostgen", description="Reproducible UV/MOST screening pipeline")\n    subparsers = parser.add_subparsers(dest="command", required=True)\n    help_text = {\n        "prepare-data": "prepare immutable-source derivatives and scaffold splits",\n        "train-reviewers": "train separate reward and evaluator ensembles",\n        "sample-baselines": "run matched-budget prior and weighted baselines",\n        "train-generator": "write family-specific REINVENT4/LibInvent manifests",\n        "generate": "run real REINVENT4 LibInvent curriculum generation",\n        "review": "independently review top candidates and queue physical oracle",\n        "run-all": "execute the entire reproducible workflow",\n    }\n    for name, description in help_text.items():\n        child = subparsers.add_parser(name, help=description)\n        child.add_argument("--config", default=str(ROOT / "config" / "default.json"))\n        child.add_argument("--override", help="JSON configuration patch")\n        child.add_argument("--mode", choices=("smoke", "full", "production"), default="full")\n        child.add_argument("--output", default="runs/latest")\n    return parser\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = build_parser()\n    args = parser.parse_args(argv)\n    try:\n        if args.command == "prepare-data":\n            result = command_prepare(args)\n        elif args.command == "train-reviewers":\n            result = command_train_reviewers(args)\n        elif args.command == "sample-baselines":\n            result = command_search(args, ["prior_random", "weighted_retraining"])\n        elif args.command == "train-generator":\n            result = command_train_generator(args)\n        elif args.command == "generate":\n            result = command_search(args, ["libinvent_rl"])\n        elif args.command == "review":\n            result = command_review(args)\n        else:\n            result = command_run_all(args)\n        print(json.dumps(result, indent=2, sort_keys=True, default=str))\n        return 0\n    except Exception as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n'
load_embedded('mostgen.cli', _source, PROJECT_ROOT / 'mostgen' / 'cli.py')
print('loaded mostgen.cli')


loaded mostgen.cli


### 3.42 CLI entrypoint — `__main__.py`


In [21]:
_source = 'from .cli import main\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n\n'
load_embedded('mostgen.__main__', _source, PROJECT_ROOT / 'mostgen' / '__main__.py')
print('loaded mostgen.__main__')


loaded mostgen.__main__


### 3.x ExternalProcess scoring bridge — `scripts/reinvent_external_score.py`


In [22]:
REINVENT_EXTERNAL_SCORER_SOURCE = '#!/usr/bin/env python3\n"""REINVENT4 ExternalProcess bridge for MOSTGen reward reviewers.\n\nREINVENT passes one full SMILES per stdin line.  The bridge deliberately scores\nonly canonical structures present in the versioned reaction library, thereby\nkeeping neural LibInvent generation inside the same commercial-synthon space\nas both baselines. Unknown, invalid, vetoed, or wrong-family structures receive\nzero. Output follows REINVENT4 ExternalProcess payload version 1.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(PROJECT_ROOT))\n\nfrom mostgen.chemistry import ChemistryError, standardize_smiles  # noqa: E402\nfrom mostgen.config import load_config  # noqa: E402\nfrom mostgen.data import read_csv  # noqa: E402\nfrom mostgen.reviewers import ReviewerBundle  # noqa: E402\nfrom mostgen.scoring import ScoringContext  # noqa: E402\n\n\ndef build_parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config", required=True)\n    parser.add_argument("--model", required=True)\n    parser.add_argument("--library", required=True)\n    parser.add_argument("--family", required=True)\n    parser.add_argument("--stage", choices=("chemistry", "spectrum", "most", "safety"), required=True)\n    return parser\n\n\ndef main(argv: list[str] | None = None) -> int:\n    args = build_parser().parse_args(argv)\n    config = load_config(args.config, mode="full")\n    library = {}\n    for row in read_csv(args.library):\n        if row["family"] == args.family:\n            library[row["smiles"]] = row\n    requested = [line.strip() for line in sys.stdin if line.strip()]\n    candidates = []\n    positions = []\n    scores = [0.0] * len(requested)\n    for index, smiles in enumerate(requested):\n        try:\n            canonical = standardize_smiles(smiles)\n        except ChemistryError:\n            continue\n        candidate = library.get(canonical)\n        if candidate is not None:\n            candidates.append(candidate)\n            positions.append(index)\n    if candidates:\n        context = ScoringContext(config, ReviewerBundle.load(args.model))\n        reviewed = context.review_candidates(candidates, "reinvent4_external", int(config["project"]["default_seed"]))\n        for position, row in zip(positions, reviewed):\n            if row.get("safety_veto") or not row.get("valid"):\n                value = 0.0\n            else:\n                stages = json.loads(row.get("stage_rewards_json", "{}"))\n                value = float(stages.get(args.stage, 0.0))\n                if args.stage == "safety" and row.get("phototoxicity_uncertain"):\n                    value = 0.0\n            scores[position] = max(0.0, min(1.0, value))\n    print(json.dumps({"version": 1, "payload": {"stage_score": scores}}))\n    return 0\n\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
from mostgen.provenance import sha256_file
display({'embedded_lines': len(REINVENT_EXTERNAL_SCORER_SOURCE.splitlines()), 'disk_sha256': sha256_file(PROJECT_ROOT / 'scripts/reinvent_external_score.py')})


{'embedded_lines': 75,
 'disk_sha256': '63c996d6a4344a977f2a14f9bf4fb7148b608447ebb42bace68ced9174e340e6'}

## 4. Конфигурация audit и production-критерий

Исполняемый audit намеренно использует один seed и 60 молекул (20 на семейство),
но **реальные данные и настоящий neural LibInvent**, а не synthetic backend.
Это проверка работоспособности, не заявление о выполнении production-критерия.
Полная конфигурация в этой же ячейке сохраняет 3 seeds и ≥1000 уникальных
структур на запуск; для неё достаточно убрать audit-overrides.


In [23]:
CONFIG_JSON = '{\n  "schema_version": "1.0",\n  "project": {\n    "name": "mostgen_uv_most",\n    "version": "0.2.0",\n    "default_seed": 1701,\n    "max_training_structures": 30000,\n    "max_gpu_hours": 8.0,\n    "disclaimer": "Research screening proxy; not ISO-certified and not a cosmetic safety determination."\n  },\n  "execution": {\n    "backend": "hybrid_reinvent4",\n    "methods": ["prior_random", "weighted_retraining", "libinvent_rl"],\n    "seeds": [1701, 2903, 4211],\n    "n_per_run": 1000,\n    "reviewer_budget_per_run": 1576,\n    "shortlist_size": 75,\n    "bootstrap_samples": 300,\n    "gpu_hours": 0.0\n  },\n  "smoke": {\n    "seeds": [17],\n    "n_per_run": 40,\n    "reviewer_budget_per_run": 80,\n    "shortlist_size": 12,\n    "bootstrap_samples": 40,\n    "training_rows_per_family": 72\n  },\n  "production": {\n    "reinvent4_repository": "https://github.com/MolecularAI/REINVENT4",\n    "reinvent4_version": "4.8",\n    "reinvent4_revision": "80a8d21",\n    "reinvent_executable": "external/reinvent-venv/bin/reinvent",\n    "reaction_prior_path": "external/REINVENT4/priors/libinvent.prior",\n    "reaction_prior_sha256": "03e6cbe8a53e59a4ac3aa6728d041f1957bdd07b5eefdf2cfc5c8591036075af",\n    "xtb_python_package": "xtb==20.2 (libxtb 6.5.1 bindings)",\n    "stda_executable": "external/xtb4stda/bin/stda",\n    "xtb4stda_executable": "external/xtb4stda/bin/xtb4stda",\n    "xtb4stda_home": "external/xtb4stda",\n    "require_rdkit": true,\n    "require_physical_oracle": true,\n    "transfer_learning_epochs": 1,\n    "rl_steps_per_stage": 2,\n    "rl_batch_size": 24,\n    "sampling_rounds": 3,\n    "sampling_oversample": 18,\n    "libinvent_supported_elements": ["C", "N", "O", "F", "S", "Cl", "Br"]\n  },\n  "data": {\n    "m13_max_rows": 25000,\n    "initial_excel_role": "source catalog and provenance; never a molecular-label table"\n  },\n  "oracle": {\n    "run_automatically": false,\n    "max_candidates": 75,\n    "conformers": 8,\n    "xtb_fmax_ev_a": 0.12,\n    "xtb_max_steps": 120,\n    "spectral_broadening_ev": 0.20\n  },\n  "elements": ["B", "C", "N", "O", "F", "Si", "P", "S", "Cl", "Br"],\n  "validation_only_families": ["iti"],\n  "families": {\n    "nbd_qc": {\n      "role": "primary",\n      "scaffold_id": "NBD-1,4-disubstituted",\n      "libinvent_scaffold": "C1([*:0])=CC2C=CC1([*:1])C2",\n      "ground_template": "C1({r1})=CC2C=CC1({r2})C2",\n      "charged_template": "C1({r1})C2C3C1C4C2C34{r2}",\n      "reaction_smarts": "[*:1].[C:2]1=[C:3][C:4]2[C:5]=[C:6][C:7]1[C:8]2>>[*:1]-[C:2]1=[C:3][C:4]2[C:5]=[C:6][C:7]1[C:8]2",\n      "allowed_positions": ["C1", "C4"]\n    },\n    "dewar_pyrimidinone": {\n      "role": "low_data_experimental",\n      "scaffold_id": "pyrimidin-4-one-2,6-disubstituted",\n      "libinvent_scaffold": "O=c1[nH]c([*:0])nc([*:1])c1",\n      "ground_template": "O=c1[nH]c({r1})nc({r2})c1",\n      "charged_template": "O=C1N2C({r1})=NC({r2})C12",\n      "reaction_smarts": "[*:1].[O:2]=[c:3]1[nH:4][c:5][n:6][c:7][c:8]1>>[O:2]=[c:3]1[nH:4][c:5]([*:1])[n:6][c:7][c:8]1",\n      "allowed_positions": ["C2", "C6"]\n    },\n    "spiropyran": {\n      "role": "transferability_check",\n      "scaffold_id": "indolino-benzopyran-6,8-disubstituted",\n      "libinvent_scaffold": "c1c([*:0])cc2c(c1[*:1])OC1(CCN(C)C1)C2",\n      "ground_template": "c1c({r1})cc2c(c1{r2})OC1(CCN(C)C1)C2",\n      "charged_template": "C[NH+]1CCC(C1)=Cc1c([O-])cc({r1})cc1{r2}",\n      "reaction_smarts": "[*:1].[c:2]1[c:3][c:4][c:5]2[c:6]([c:7]1)[O:8][C:9]1(CC[N:10](C)C1)[C:11]2>>[*:1]-[c:2]1[c:3][c:4][c:5]2[c:6]([c:7]1)[O:8][C:9]1(CC[N:10](C)C1)[C:11]2",\n      "allowed_positions": ["C6", "C8"]\n    }\n  },\n  "synthons": [\n    {"id": "S01", "smiles": "F", "commercial": true},\n    {"id": "S02", "smiles": "Cl", "commercial": true},\n    {"id": "S03", "smiles": "Br", "commercial": true},\n    {"id": "S04", "smiles": "C", "commercial": true},\n    {"id": "S05", "smiles": "CC", "commercial": true},\n    {"id": "S06", "smiles": "CCC", "commercial": true},\n    {"id": "S07", "smiles": "C(F)(F)F", "commercial": true},\n    {"id": "S08", "smiles": "C#N", "commercial": true},\n    {"id": "S09", "smiles": "N", "commercial": true},\n    {"id": "S10", "smiles": "NC", "commercial": true},\n    {"id": "S11", "smiles": "N(C)C", "commercial": true},\n    {"id": "S12", "smiles": "O", "commercial": true},\n    {"id": "S13", "smiles": "OC", "commercial": true},\n    {"id": "S14", "smiles": "OCC", "commercial": true},\n    {"id": "S15", "smiles": "C(=O)O", "commercial": true},\n    {"id": "S16", "smiles": "C(=O)OC", "commercial": true},\n    {"id": "S17", "smiles": "C(=O)N", "commercial": true},\n    {"id": "S18", "smiles": "S(=O)(=O)C", "commercial": true},\n    {"id": "S19", "smiles": "c5ccccc5", "commercial": true},\n    {"id": "S20", "smiles": "c5ccc(F)cc5", "commercial": true},\n    {"id": "S21", "smiles": "c5ccc(Cl)cc5", "commercial": true},\n    {"id": "S22", "smiles": "c5ccc(C#N)cc5", "commercial": true},\n    {"id": "S23", "smiles": "c5ccc(OC)cc5", "commercial": true},\n    {"id": "S24", "smiles": "c5ccc(N(C)C)cc5", "commercial": true},\n    {"id": "S25", "smiles": "c5ccncc5", "commercial": true},\n    {"id": "S26", "smiles": "c5ncccc5", "commercial": true},\n    {"id": "S27", "smiles": "c5ccsc5", "commercial": true},\n    {"id": "S28", "smiles": "C=C", "commercial": true},\n    {"id": "S29", "smiles": "C=C(C)C", "commercial": true},\n    {"id": "S30", "smiles": "C5CC5", "commercial": true},\n    {"id": "S31", "smiles": "C5CCC5", "commercial": true},\n    {"id": "S32", "smiles": "C5CCOC5", "commercial": true},\n    {"id": "S33", "smiles": "C(F)F", "commercial": true},\n    {"id": "S34", "smiles": "P(=O)(O)O", "commercial": true}\n  ],\n  "reviewers": {\n    "ensemble_size": 5,\n    "fingerprint_bits": 256,\n    "ad_similarity_threshold": 0.18,\n    "confidence_z": 1.645,\n    "half_life_window_hours": [4.0, 24.0],\n    "lambda_c_min_nm": 370.0,\n    "specific_energy_min_wh_kg": 50.0,\n    "beer_lambert_loading_scale": 1.0,\n    "uvb_transmittance_max": 0.50,\n    "uva_transmittance_max": 0.50,\n    "phototoxicity_max_probability": 0.35,\n    "sensitization_max_probability": 0.50,\n    "irritation_max_probability": 0.50,\n    "kp_max_log_cm_s": -4.0,\n    "uncertain_probability_band": [0.25, 0.45],\n    "trees_per_member": 28\n  },\n  "reward": {\n    "floor": 0.001,\n    "stages": [\n      {"name": "chemistry", "components": ["validity", "family", "pair"]},\n      {"name": "spectrum", "components": ["uvb", "uva", "lambda_c", "film_uvb", "film_uva"]},\n      {"name": "most", "components": ["energy", "specific_energy", "half_life"]},\n      {"name": "safety", "components": ["phototoxicity", "permeation", "sensitization", "irritation", "sa", "ad"]}\n    ],\n    "weights": {"uvb": 1.2, "uva": 1.5, "lambda_c": 1.0, "film_uvb": 0.8, "film_uva": 1.0, "energy": 1.2, "specific_energy": 1.0, "half_life": 1.0, "phototoxicity": 1.5, "permeation": 0.8, "sensitization": 1.0, "irritation": 1.0, "sa": 0.6, "ad": 1.0},\n    "diversity_penalty": 0.12\n  },\n  "safety": {\n    "psoralen_similarity_warning_threshold": 0.45,\n    "psoralen_core_smarts": [\n      "C1=CC(=O)OC2=CC3=C(C=CO3)C=C21",\n      "C1=CC2=C(C=CO2)C3=C1C=CC(=O)O3"\n    ],\n    "known_phototoxic_smiles": [\n      "C1=CC(=O)OC2=CC3=C(C=CO3)C=C21",\n      "C1=CC2=C(C=CO2)C3=C1C=CC(=O)O3",\n      "COC1=C2C=CC(=O)OC2=CC3=C1C=CO3"\n    ],\n    "u16_qsardb_directory": "data/raw/U16_qsardb/files",\n    "reactive_fragments": ["N=N=N", "OO", "C(Cl)(Cl)Cl", "[N-]=[N+]=N", "P(Cl)Cl", "S(Cl)(=O)=O"]\n  },\n  "sources": [\n    {"id": "M01", "role": "photoswitch calibration", "url": "LOCAL_OR_LICENSED_DATA_REQUIRED", "license": "verify_before_use"},\n    {"id": "M03", "role": "MOST energy and spectrum validation", "url": "LOCAL_OR_LICENSED_DATA_REQUIRED", "license": "verify_before_use"},\n    {"id": "M04", "role": "MOST state validation", "url": "LOCAL_OR_LICENSED_DATA_REQUIRED", "license": "verify_before_use"},\n    {"id": "M08-M10", "role": "literature orientation only; no bulk training data claimed", "url": "BIBLIOGRAPHY_RESOLUTION_REQUIRED", "license": "metadata_only"},\n    {"id": "M11", "role": "spectral pretraining", "url": "LOCAL_DATA_REQUIRED", "license": "verify_before_use"},\n    {"id": "M12", "role": "spectral pretraining", "url": "LOCAL_DATA_REQUIRED", "license": "verify_before_use"},\n    {"id": "M13", "role": "unified spectral transfer", "url": "LOCAL_DATA_REQUIRED", "license": "verify_before_use"},\n    {"id": "U07-U10", "role": "irritation and sensitisation", "url": "LOCAL_DATA_REQUIRED", "license": "verify_before_use"},\n    {"id": "U11-U14", "role": "skin permeation Kp/Jmax", "url": "LOCAL_DATA_REQUIRED", "license": "verify_before_use"},\n    {"id": "U16-U17", "role": "phototoxicity", "url": "LOCAL_DATA_REQUIRED", "license": "verify_before_use"},\n    {"id": "U01/U03/U06", "role": "versioned regulatory and expert references only", "url": "REGISTRY_RESOLUTION_REQUIRED", "license": "metadata_only"},\n    {"id": "REINVENT4", "role": "LibInvent generator", "url": "https://github.com/MolecularAI/REINVENT4", "version": "4.8@80a8d21", "license": "Apache-2.0"},\n    {"id": "SCCP", "role": "furocoumarin safety opinion", "url": "https://ec.europa.eu/health/ph_risk/committees/04_sccp/docs/sccp_o_036.pdf", "license": "public_document"},\n    {"id": "ISO24444", "role": "finished-product in vivo SPF method", "url": "https://www.iso.org/standard/72250.html", "license": "metadata_only"},\n    {"id": "ISO24443", "role": "finished-product in vitro UVA method", "url": "https://www.iso.org/standard/75059.html", "license": "metadata_only"},\n    {"id": "OECD432", "role": "experimental phototoxicity", "url": "https://www.oecd.org/en/publications/2019/06/test-no-432-in-vitro-3t3-nru-phototoxicity-test_g1gh4b69.html", "license": "public_metadata"}\n  ]\n}\n'

from mostgen.config import validate_config

config = json.loads(CONFIG_JSON)
config["_config_path"] = str(PROJECT_ROOT / "config/default.json")
config["execution"].update({
    "mode": "full", "backend": "reinvent4", "seeds": [1701],
    "n_per_run": 60, "reviewer_budget_per_run": 636,
    "shortlist_size": 12, "bootstrap_samples": 40,
})
config["data"]["m13_max_rows"] = 12000
config["reviewers"]["ensemble_size"] = 3
config["reviewers"]["trees_per_member"] = 10
config["production"].update({"sampling_rounds": 1, "sampling_oversample": 12})
config["oracle"].update({"run_automatically": False, "max_candidates": 1,
                          "conformers": 3, "xtb_max_steps": 60, "xtb_fmax_ev_a": 0.18})
config["training_rows_per_family"] = 180
validate_config(config)

production_contract = json.loads(CONFIG_JSON)["execution"]
display({"audit": config["execution"], "production_contract": production_contract,
         "warning": "audit_pass != production_acceptance"})


{'audit': {'backend': 'reinvent4',
  'methods': ['prior_random', 'weighted_retraining', 'libinvent_rl'],
  'seeds': [1701],
  'n_per_run': 60,
  'reviewer_budget_per_run': 636,
  'shortlist_size': 12,
  'bootstrap_samples': 40,
  'gpu_hours': 0.0,
  'mode': 'full'},
 'production_contract': {'backend': 'hybrid_reinvent4',
  'methods': ['prior_random', 'weighted_retraining', 'libinvent_rl'],
  'seeds': [1701, 2903, 4211],
  'n_per_run': 1000,
  'reviewer_budget_per_run': 1576,
  'shortlist_size': 75,
  'bootstrap_samples': 300,
  'gpu_hours': 0.0},
 'warning': 'audit_pass != production_acceptance'}

## 5. Реальные данные и роль исходного Excel

Используемые для fit источники:

- M13 UV/VisML: λmax transfer data, включая M11 Deep4Chem и M12 CDEx;
- M01 photoswitch database: λmax и thermal Z→E kinetics;
- U07 NICE: irritation/corrosion calls, только exact DTXSID join;
- U09 HPPT: human skin sensitization;
- U12 SkinPiX и U13 human epidermis: logKp с явным преобразованием единиц;
- U16 QsarDB: 3T3 NRU phototoxicity;
- M05: 22 condition-rich full-spectrum records двух spiropyrans, только как
  отдельная calibration/validation evidence.

`database_matrix_MOST_UV_skin.xlsx`, данный в начале, **используется**, но лишь
как каталог литературы/источников и provenance. В нём нет пригодной таблицы
`SMILES → endpoint`, поэтому он не попадает в обучение. M03 недоступен через
file API без bearer token (HTTP 401); нулевой partial исключён. M04 не превращён
в labels без надёжной автоматической идентификации структур. Синтетические
energy labels не подставляются.


In [24]:
from mostgen.data import prepare_data, read_csv, write_csv
from mostgen.reviewers import train_reviewers
from mostgen.provenance import write_artifact_manifest

RUN_ROOT = PROJECT_ROOT / "runs/notebook_real_audit"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
data_manifest = prepare_data(config, RUN_ROOT / "data", PROJECT_ROOT)
endpoints = read_csv(RUN_ROOT / "data/reviewer_endpoints.csv")
catalog = read_csv(RUN_ROOT / "data/source_catalog_from_initial_excel.csv")
display({"training_mode": data_manifest["training_mode"],
         "endpoint_rows": data_manifest["endpoint_rows"],
         "unique_structures_total": data_manifest["unique_structures_total"],
         "full_spectrum_records": data_manifest["full_spectrum_records"],
         "reaction_library_rows": data_manifest["reaction_library_rows"],
         "initial_excel_catalog_rows": len(catalog),
         "initial_excel_usage": data_manifest["initial_excel_usage"],
         "unavailable_labels": data_manifest["unavailable_labels"]})


/home/sigmatau17/TEST/venv/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


/home/sigmatau17/TEST/venv/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


{'training_mode': 'real_endpoint_specific',
 'endpoint_rows': {'kp_log_cm_s': 355,
  'lambda_max_nm': 12749,
  'log_half_life_h': 75,
  'phototoxicity': 53,
  'skin_irritation': 763,
  'skin_sensitization': 1811},
 'unique_structures_total': 9390,
 'full_spectrum_records': 22,
 'reaction_library_rows': 3267,
 'initial_excel_catalog_rows': 181,
 'initial_excel_usage': 'Used as a literature/source catalog and provenance input; it contains no molecular training labels.',
 'unavailable_labels': {'M03': 'Metadata available, but Dryad file endpoint returned HTTP 401 without bearer token; empty partial file excluded.',
  'energy_kj_mol': 'No sufficiently identified open molecular training table is available; no synthetic replacement is made.'}}

## 6. Reviewer models и формирование оценки

Для каждой структуры ансамбль выдаёт mean и calibrated uncertainty. Предсказанный
λmax превращается в явно объявленный Gaussian-band proxy (σ=34 nm), после чего:

\[
AUC_{UVB}=\int_{290}^{320}A(\lambda)d\lambda,\quad
AUC_{UVA}=\int_{320}^{400}A(\lambda)d\lambda.
\]

`λc` — точка 90% cumulative area. UV gate использует нижние 90% границы AUC и
λc≥370 nm. MOST gate требует положительную нижнюю границу ΔH, семейный reference
и t½=4–24 h при 305 K. Safety gate проверяет upper bounds phototoxicity, Kp и
sensitization, SA proxy, AD и hard structural veto.

Dense reward — weighted geometric mean непрерывных sigmoid/interval scores с
floor 0.001. Отсутствующей ΔH даётся только низкий shaping-component 0.20;
hard MOST-pass при этом всегда false. Это сохраняет обучаемость curriculum, но
не позволяет превратить отсутствие данных в «успех».


In [25]:
model_cards = train_reviewers(config, RUN_ROOT / "data/reviewer_endpoints.csv", RUN_ROOT / "models")
compact_metrics = {
    purpose: {endpoint: values for endpoint, values in model_cards[purpose]["metrics"].items()}
    for purpose in ("reward", "evaluator")
}
display({"rows": model_cards["rows"], "unique": model_cards["unique_structures"],
         "energy_model_status": model_cards["energy_model_status"],
         "independent_instances": model_cards["independent_instances"],
         "metrics": compact_metrics})


{'rows': 15806,
 'unique': 9390,
 'energy_model_status': 'not_trained; physical GFN2-xTB oracle required',
 'independent_instances': True,
 'metrics': {'reward': {'lambda_max_nm': {'n_total': 12749,
    'n_train': 8869,
    'n_calibration': 2435,
    'n_test': 1445,
    'unique_structures': 8225,
    'conformal_q90': 90.06523809523816,
    'mae': 42.03749658891965,
    'rmse': 56.804980220013555,
    'interval_90_coverage': 0.8941176470588236},
   'log_half_life_h': {'n_total': 75,
    'n_train': 54,
    'n_calibration': 9,
    'n_test': 12,
    'unique_structures': 75,
    'conformal_q90': 2.628849016656819,
    'mae': 1.1728123511360502,
    'rmse': 1.1964348741069797,
    'interval_90_coverage': 1.0},
   'kp_log_cm_s': {'n_total': 355,
    'n_train': 237,
    'n_calibration': 75,
    'n_test': 43,
    'unique_structures': 125,
    'conformal_q90': 1.5311317792357366,
    'mae': 0.8432824766316591,
    'rmse': 1.0522270404989509,
    'interval_90_coverage': 0.9069767441860465},
   'p

## 7. Настоящий REINVENT4 LibInvent

Сначала создаются и обучаются три TL agents. Затем для текущего seed запускаются
12 stages (4×3 family) с DAP и `PenalizeSameSmiles`. Post-sampling сначала
использует RL checkpoint, затем TL agent как neural diversity backstop. Ни
случайное перечисление библиотеки, ни enumerative filler не добавляются.
Нейросетевые предложения вне exact allowed library учитываются в audit, но
отбрасываются до итогового scoring.


In [26]:
from mostgen.cli import _production_checks
from mostgen.search import (
    execute_generator_training, run_methods, run_reinvent_generation,
    write_generator_manifests,
)

_production_checks(config)
generator_manifest = write_generator_manifests(config, RUN_ROOT / "generator")
transfer_learning = execute_generator_training(config, RUN_ROOT / "generator")
libinvent_result = run_reinvent_generation(
    config, RUN_ROOT / "generator", RUN_ROOT / "data/reaction_library.csv",
    RUN_ROOT / "models/reward/reviewers.pkl", RUN_ROOT / "libinvent_generated.csv",
)
display({"transfer_learning": transfer_learning,
         "curriculum_backend": libinvent_result["backend"],
         "curriculum_optimization_calls": libinvent_result["curriculum"]["optimization_reviewer_evaluations"],
         "sampling": libinvent_result["sampling"]})


{'transfer_learning': {'status': 'trained',
  'generator': 'REINVENT4 LibInvent',
  'runs': [{'family': 'dewar_pyrimidinone',
    'returncode': 0,
    'agent': '/home/sigmatau17/TEST/runs/notebook_real_audit/generator/libinvent_dewar_pyrimidinone.agent',
    'agent_exists': True,
    'log': '/home/sigmatau17/TEST/runs/notebook_real_audit/generator/transfer_learning_dewar_pyrimidinone.log'},
   {'family': 'nbd_qc',
    'returncode': 0,
    'agent': '/home/sigmatau17/TEST/runs/notebook_real_audit/generator/libinvent_nbd_qc.agent',
    'agent_exists': True,
    'log': '/home/sigmatau17/TEST/runs/notebook_real_audit/generator/transfer_learning_nbd_qc.log'},
   {'family': 'spiropyran',
    'returncode': 0,
    'agent': '/home/sigmatau17/TEST/runs/notebook_real_audit/generator/libinvent_spiropyran.agent',
    'agent_exists': True,
    'log': '/home/sigmatau17/TEST/runs/notebook_real_audit/generator/transfer_learning_spiropyran.log'}]},
 'curriculum_backend': 'REINVENT4 LibInvent staged_learn

## 8. Baselines, matched reviewer budget, metrics и independent review

Baselines работают в том же reaction-library space. Каждый метод получает
ровно 636 reward-reviewer evaluations: LibInvent = 576 curriculum + 60 final,
baseline = 636 рассмотренных library candidates. После поиска каждый метод
сохраняет одинаковые 60 уникальных структур (20 на семейство). Фактические
вызовы проверяются по `search_budget_ledger.json`, а не выводятся из числа строк.


In [27]:
from mostgen.metrics import compute_metrics
from mostgen.review import review_generated
from mostgen.reporting import build_reports
from mostgen.validation import verify_experiment

baseline_result = run_methods(
    config, RUN_ROOT / "data/reaction_library.csv", RUN_ROOT / "models/reward/reviewers.pkl",
    RUN_ROOT / "baselines.csv", ["prior_random", "weighted_retraining"],
)
combined = read_csv(RUN_ROOT / "baselines.csv") + read_csv(RUN_ROOT / "libinvent_generated.csv")
write_csv(RUN_ROOT / "generated.csv", combined)
metrics_result = compute_metrics(
    RUN_ROOT / "generated.csv", RUN_ROOT / "data/reviewer_endpoints.csv", RUN_ROOT / "metrics", config,
)
review_result = review_generated(
    config, RUN_ROOT / "generated.csv", RUN_ROOT / "models/evaluator/reviewers.pkl", RUN_ROOT / "review",
)
reports = build_reports(RUN_ROOT, config)
verification = verify_experiment(RUN_ROOT, config)
write_artifact_manifest(RUN_ROOT)

rows = read_csv(RUN_ROOT / "generated.csv")
library = {(row["family"], row["smiles"]) for row in read_csv(RUN_ROOT / "data/reaction_library.csv")}
display({"run_counts": dict(Counter((r["method_id"], r["seed"]) for r in rows)),
         "family_counts": dict(Counter((r["method_id"], r["family"]) for r in rows)),
         "all_exact_library": all((r["family"], r["smiles"]) in library for r in rows),
         "psoralen_cores": sum(str(r["psoralen_alert"]).lower() == "true" for r in rows),
         "known_phototoxic_matches": sum(str(r["known_phototoxic_match"]).lower() == "true" for r in rows),
         "joint_pass": sum(str(r["joint_pass"]).lower() == "true" for r in rows),
         "selected": review_result["selected_after_physical_oracle"],
         "laboratory_experiments_executed": False,
         "experimental_package": review_result["experimental_package"]["status"],
         "audit_verification": verification["passed"]})


{'run_counts': {('prior_random', '1701'): 60,
  ('weighted_retraining', '1701'): 60,
  ('libinvent_rl', '1701'): 60},
 'family_counts': {('prior_random', 'dewar_pyrimidinone'): 20,
  ('prior_random', 'nbd_qc'): 20,
  ('prior_random', 'spiropyran'): 20,
  ('weighted_retraining', 'dewar_pyrimidinone'): 20,
  ('weighted_retraining', 'nbd_qc'): 20,
  ('weighted_retraining', 'spiropyran'): 20,
  ('libinvent_rl', 'dewar_pyrimidinone'): 20,
  ('libinvent_rl', 'nbd_qc'): 20,
  ('libinvent_rl', 'spiropyran'): 20},
 'all_exact_library': True,
 'psoralen_cores': 0,
 'known_phototoxic_matches': 0,
 'joint_pass': 0,
 'selected': 0,
 'laboratory_experiments_executed': False,
 'experimental_package': 'blocked_no_computational_candidates',
 'audit_verification': True}

## 9. Независимый вычислительный physical unit-oracle

Чтобы проверить сам toolchain даже при нуле provisional joint-pass, здесь
рассчитывается простой NBD/QC library member F/F. Это unit-oracle, а не выбранный
кандидат: ΔE — electronic-energy proxy, а sTDA curve — broadened proxy. Для
production top-50–100 нужны более строгая конформерная сходимость и последующая
экспериментальная проверка. Эта ячейка не выполняет лабораторный пункт 9 из
перечня пользователя: синтез, измерения и OECD здесь не запускаются.


In [28]:
from mostgen.oracle import availability, evaluate_pair

library_rows = read_csv(RUN_ROOT / "data/reaction_library.csv")
oracle_candidate = next(row for row in library_rows
                        if row["family"] == "nbd_qc" and row["synthon_a"] == "S01" and row["synthon_b"] == "S01")
physical = evaluate_pair(oracle_candidate, RUN_ROOT / "review/physical_unit_oracle", config, 1)
write_artifact_manifest(RUN_ROOT)
display({"availability": availability(config),
         "candidate": {key: oracle_candidate[key] for key in ("smiles", "charged_smiles", "family")},
         "result": {key: physical[key] for key in ("gfn2_delta_e_kj_mol", "gfn2_specific_energy_wh_kg",
                                                     "stda_uvb_auc", "stda_uva_auc", "stda_lambda_c_nm",
                                                     "ground_xtb_converged", "charged_xtb_converged")}})


{'availability': {'tools': {'xtb_python': True,
   'ase': True,
   'stda': '/home/sigmatau17/TEST/external/xtb4stda/bin/stda',
   'xtb4stda': '/home/sigmatau17/TEST/external/xtb4stda/bin/xtb4stda'},
  'energy_ready': True,
  'spectrum_ready': True,
  'ready': True},
 'candidate': {'smiles': 'FC1=CC2C=CC1(F)C2',
  'charged_smiles': 'FC1C2C3C1C1C2C31F',
  'family': 'nbd_qc'},
 'result': {'gfn2_delta_e_kj_mol': 68.55865264402378,
  'gfn2_specific_energy_wh_kg': 148.64128581902284,
  'stda_uvb_auc': 1.6284872350761315e-08,
  'stda_uva_auc': 5.3091147729262924e-14,
  'stda_lambda_c_nm': 296.5334800209333,
  'ground_xtb_converged': True,
  'charged_xtb_converged': True}}

## 10. Итоговый ответ и необходимые доработки


In [29]:
failure_counts = Counter()
for row in rows:
    failure_counts.update(reason for reason in row.get("failure_reasons", "").split(";") if reason)

display(Markdown(f"""
### Работает ли pipeline?

**Да, программно и end-to-end:** реальные источники распарсены, independent
reviewers обучены, family TL и 12 curriculum stages REINVENT4 завершились,
post-sampling дал требуемые audit-квоты, safety/AD/review/report/oracle исполнились.

### Генерируются ли нужные по ТЗ молекулы?

**По химическому пространству — да:** три заданных семейства, валидные
ground/charged pairs, только разрешённые синтоны, 0 псораленовых/фурокумариновых
ядер. **По совокупности целевых свойств — пока не доказано:** joint-pass =
{sum(str(r['joint_pass']).lower() == 'true' for r in rows)}. Главная причина —
нет открытой надёжной molecular ΔH training table; energy/AD gate правильно
закрыт. Это не доказательство физической невозможности.

### Что требует улучшения?

1. Получить/вручную курировать M03/M04 ΔH, ΔG‡, t½, state/condition labels.
2. Расширить очень малые U16 phototoxicity и M01 class-shift kinetics выборки.
3. Заменить λmax Gaussian proxy большим набором реальных полных спектров и
   solvent/state-aware моделью.
4. Выполнить 3 независимых production seeds и ≥1000 neural-unique molecules на
   запуск; текущая выполненная ячейка — честный audit 60, не production acceptance.
5. Провести GFN2/sTDA top-50–100, затем синтез, cycling/quantum yield/t½,
   OECD TG 432 и испытание готовой плёнки/формуляции.

Частые причины отказа в audit: `{dict(failure_counts.most_common(8))}`.

Начальный Excel: **использован как source catalog/provenance ({len(catalog)}
строк), не использован для fit**, поскольку не содержит молекулярной таблицы
SMILES/endpoint.
"""))



### Работает ли pipeline?

**Да, программно и end-to-end:** реальные источники распарсены, independent
reviewers обучены, family TL и 12 curriculum stages REINVENT4 завершились,
post-sampling дал требуемые audit-квоты, safety/AD/review/report/oracle исполнились.

### Генерируются ли нужные по ТЗ молекулы?

**По химическому пространству — да:** три заданных семейства, валидные
ground/charged pairs, только разрешённые синтоны, 0 псораленовых/фурокумариновых
ядер. **По совокупности целевых свойств — пока не доказано:** joint-pass =
0. Главная причина —
нет открытой надёжной molecular ΔH training table; energy/AD gate правильно
закрыт. Это не доказательство физической невозможности.

### Что требует улучшения?

1. Получить/вручную курировать M03/M04 ΔH, ΔG‡, t½, state/condition labels.
2. Расширить очень малые U16 phototoxicity и M01 class-shift kinetics выборки.
3. Заменить λmax Gaussian proxy большим набором реальных полных спектров и
   solvent/state-aware моделью.
4. Выполнить 3 независимых production seeds и ≥1000 neural-unique molecules на
   запуск; текущая выполненная ячейка — честный audit 60, не production acceptance.
5. Провести GFN2/sTDA top-50–100, затем синтез, cycling/quantum yield/t½,
   OECD TG 432 и испытание готовой плёнки/формуляции.

Частые причины отказа в audit: `{'most_energy_or_half_life': 180, 'energy_model_unavailable_requires_physical_oracle': 180, 'most_out_of_domain': 180, 'phototoxicity_uncertain': 180, 'sensitization_risk': 180, 'irritation_risk': 180, 'uv_joint_lcb_or_lambda': 161, 'beer_lambert_film_proxy': 156}`.

Начальный Excel: **использован как source catalog/provenance (181
строк), не использован для fit**, поскольку не содержит молекулярной таблицы
SMILES/endpoint.


## 11. Команды CLI

Полный production-прогон (3 метода × 3 seed × ≥1000 структур) реализован той же
кодовой базой. Он намеренно не маскируется результатом короткого audit:

```bash
./venv/bin/python -m mostgen run-all --mode production --output runs/production
```

Промежуточные стадии доступны как `prepare-data`, `train-reviewers`,
`sample-baselines`, `train-generator`, `generate`, `review`. Все конфиги,
checksums, model cards, CSV, oracle files и отчёты сохраняются внутри run-dir.
